# Full End-to-End Pipeline

This master notebook combines the project notebooks into one sequential workflow: Petri dish detector training, 736x736 crop preparation, YOLO26x-seg colony segmentation training, and the final two-model prediction/anomaly pipeline.

Heavy training and batch inference blocks are disabled by default. Change the flags in the next cell before running the corresponding section.


# ColonyNet full_pipline

`full_pipline` - автономная папка для полного pipeline анализа изображений чашек Петри: от локализации чашки до сегментации бактериальных колоний и интерпретируемого поиска аномальных колоний.

Папка содержит локальные копии весов моделей, чтобы финальный inference можно было запускать без поиска файлов в `runs/...`.

## Краткая схема

```text
raw photo
  -> YOLO26s Petri detector
  -> crop Petri dish + resize to 736x736
  -> YOLO26x-seg colony instance segmentation
  -> mask cleaning and non-overlap resolution
  -> feature extraction per colony
  -> technical filtering
  -> unsupervised anomaly scoring
  -> selected_anomalies / review_candidates / technical_warnings
```

## Содержимое папки

| файл | назначение | найден | размер, MB | путь |
| --- | --- | --- | --- | --- |
| petri_detector_yolo26s_best.pt | детектор чашки Петри, первый этап | 1 | 19.3900 | C:/ColonyNet/full_pipline/models/petri_detector_yolo26s_best.pt |
| colony_yolo26x_seg_best.pt | YOLO26x-seg instance segmentation колоний, второй этап | 1 | 135.1700 | C:/ColonyNet/full_pipline/models/colony_yolo26x_seg_best.pt |

| файл | назначение |
| --- | --- |
| `full_pipeline.py` | скриптовая версия полного inference pipeline |
| `full_pipeline.ipynb` | короткий notebook для запуска готовых моделей |
| `full_end_to_end_pipeline.ipynb` | master-notebook от обучения моделей до всех предсказаний подряд |
| `README.md` | этот технический отчёт |
| `__init__.py` | делает папку импортируемым Python-пакетом |

## Исходные notebooks

| источник | роль |
| --- | --- |
| train_yolo26n_s_petri_curcle.ipynb | split, offline augmentation, обучение YOLO26n/YOLO26s детектора чашки, test metrics, crop 736 |
| yolo_colonies_mlflow_736_ maskratio_1.ipynb | crop/remap YOLO-seg labels, offline augmentation 736, VRAM-safe training variants |
| yolo_colonies_mlflow.ipynb | MLflow обучение и сравнение YOLO26-seg моделей, предсказания и отчёты |
| notebooks/colony_anomaly_detection_yolo_x_improved.ipynb | разработка interpretable anomaly detection после YOLO masks |
| full_pipline/full_pipeline.py | скриптовая версия полного inference pipeline |
| full_pipline/full_pipeline.ipynb | короткий notebook только для применения готовых моделей |
| full_pipline/full_end_to_end_pipeline.ipynb | master-notebook от обучения моделей до всех предсказаний подряд |

## Запуск

Из корня проекта:

```powershell
.\.venv\Scripts\python.exe full_pipline\full_pipeline.py
```

Выходная папка по умолчанию:

```text
outputs/colony_anomaly_detection/petri_full_pipeline
```

В master-notebook тяжёлые блоки отключены по умолчанию:

| флаг | по умолчанию | что запускает |
| --- | --- | --- |
| RUN_PETRI_DETECTOR_TRAINING | 0 | обучение YOLO detector чашки |
| RUN_CROP_736_PREPARATION | 0 | подготовку crop 736 и segmentation dataset |
| RUN_COLONY_SEGMENTATION_TRAINING | 0 | обучение YOLO26-seg моделей |
| RUN_TRAINING_SET_PREDICTIONS | 0 | preview masks на train crop |
| RUN_FINAL_FULL_PIPELINE | 0 | финальный прогон двух моделей и anomaly detection |

## Финальные изображения для полного прогона

| N | файл | exists | путь |
| --- | --- | --- | --- |
| 1 | IMG_4615.jpg | 1 | C:/ColonyNet/Петри/IMG_4615.jpg |
| 2 | IMG_4655.jpg | 1 | C:/ColonyNet/Петри/IMG_4655.jpg |
| 3 | IMG_4377.jpg | 1 | C:/ColonyNet/Петри/IMG_4377.jpg |
| 4 | IMG_7438.jpg | 1 | C:/ColonyNet/Петри/IMG_7438.jpg |
| 5 | IMG_6137.jpg | 1 | C:/ColonyNet/Петри/IMG_6137.jpg |

## Данные для детектора чашки Петри

| путь | количество |
| --- | --- |
| `Petri_curcle/images/train` | 127 изображений |
| `Petri_curcle/labels/train` | 127 label-файлов |
| объектов bbox на исходное изображение | 1 чашка Петри |

После split и offline augmentation:

| split | images | labels | оригиналы | аугм. | bbox objects | obj/img avg | min | max |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| train | 440 | 440 | 88 | 352 | 440 | 1.0000 | 1 | 1 |
| val | 25 | 25 | 25 | 0 | 25 | 1.0000 | 1 | 1 |
| test | 14 | 14 | 14 | 0 | 14 | 1.0000 | 1 | 1 |

Особенности:

- В исходной разметке для детектора на каждом изображении ожидается один bbox чашки Петри.
- Split задавался как `train=0.7`, `val=0.2`, `test=0.1`.
- Offline augmentation применялась только к train, поэтому val/test остаются ближе к независимой оценке качества.
- Train содержит 88 оригиналов и 352 augmented копии, всего 440 изображений.

## Аугментации детектора чашки Петри

| тип | параметр | значение | смысл |
| --- | --- | --- | --- |
| offline | AUG_TARGET_SPLITS | train | аугментация применяется только к train |
| offline | AUG_MULTIPLIER | 5 | 1 оригинал + 4 soft-варианта |
| offline | геометрия | без сильных поворотов | чашка не искажается агрессивно |
| offline | фотометрия | HSV/Lab/brightness/contrast/noise/blur | мягкая имитация освещения и камеры |
| online YOLO | hsv_h | 0.0100 | малый сдвиг оттенка |
| online YOLO | hsv_s | 0.2200 | изменение насыщенности |
| online YOLO | hsv_v | 0.1400 | изменение яркости |
| online YOLO | degrees | 3.0000 | небольшой поворот |
| online YOLO | translate | 0.0400 | небольшой сдвиг |
| online YOLO | scale | 0.0800 | небольшой масштаб |
| online YOLO | fliplr | 0.5000 | горизонтальный flip |
| online YOLO | flipud | 0.0000 | вертикальный flip отключён |
| online YOLO | mosaic/mixup/copy_paste | 0.0000 | отключены, чтобы не создавать неестественные чашки |

## Обучение детектора чашки Петри

| параметр | значение |
| --- | --- |
| модели | yolo26n.pt, yolo26s.pt |
| imgsz | 736 |
| batch | 8 |
| EPOCHS в config | 100 |
| epochs в текущем train-вызове | 50 |
| split | 70/20/10 |
| project | runs/detect/runs/petri_curcle |
| production model | yolo26s_petri_curcle/weights/best.pt |

## Метрики детектора чашки Петри

Файл:

```text
Petri_curcle/split_dataset/multi_model_test_metrics.csv
```

| model | precision_B | recall_B | mAP50_B | mAP50_95_B | fitness | test_dir |
| --- | --- | --- | --- | --- | --- | --- |
| n | 1.0000 | 0.9987 | 0.9950 | 0.9950 | 0.9950 | C:\ColonyNet\runs\detect\runs\petri_curcle\yolo26n_petri_curcle_test |
| s | 0.9965 | 1.0000 | 0.9950 | 0.9950 | 0.9950 | C:\ColonyNet\runs\detect\runs\petri_curcle\yolo26s_petri_curcle_test |

Интерпретация:

- `precision_B` - доля предсказанных bbox, которые соответствуют реальной чашке.
- `recall_B` - доля размеченных чашек, найденных моделью.
- `mAP50_B` - mAP при IoU 0.50.
- `mAP50_95_B` - mAP по диапазону IoU 0.50-0.95.
- Для полного pipeline особенно важен `recall_B`: если чашка не найдена, второй этап не получает корректный crop.

В production pipeline используется `YOLO26s`, потому что он даёт recall 1.0 на test split и остаётся лёгким относительно YOLO26x-seg.

## Crop чашки до 736x736

Файл:

```text
Petri_curcle/cropped_736/crop_report.csv
```

| метрика | значение |
| --- | --- |
| строк в crop_report | 127 |
| успешно saved | 127 |
| conf min | 0.9422 |
| conf mean | 0.9843 |
| conf median | 0.9854 |
| conf max | 0.9878 |
| orig_w mean | 4459.5748 |
| orig_h mean | 3527.4331 |
| bbox_w mean | 2183.9528 |
| bbox_h mean | 2158.6378 |
| out_w | 736 |
| out_h | 736 |

Почему crop делается с небольшим запасом фона:

- лучше оставить небольшой внешний контекст, чем срезать край чашки;
- периферийные колонии могут находиться близко к границе;
- локальный фон используется в признаках `local_contrast`, `local_color_delta_lab`, `poor_local_background`;
- слишком агрессивный crop ухудшает анализ объектов на краю.

## Данные для сегментации колоний

Исходные crop 736x736:

| путь | значение |
| --- | --- |
| `cropped_736/images/train` | 10 изображений |
| `cropped_736/labels/train` | 10 label-файлов |
| всего размеченных colony objects | 2951 |
| среднее объектов на crop | 295.10 |
| минимум объектов на crop | 73 |
| максимум объектов на crop | 1165 |

Подготовленный augmented dataset:

| split | images | labels | colony objects | obj/img avg | min | max |
| --- | --- | --- | --- | --- | --- | --- |
| train | 512 | 512 | 145340 | 283.8672 | 73 | 1165 |
| val | 64 | 64 | 22327 | 348.8594 | 73 | 1165 |
| test | 64 | 64 | 21197 | 331.2031 | 73 | 1165 |

Важно: папка называется `cropped_736_aug_leaky`, потому что augmented варианты одного исходного crop могут попадать в разные split. Это удобно для разработки pipeline, но для строгой научной оценки лучше использовать group split по исходному изображению.

## Аугментации сегментационного датасета

| группа | значения | количество | комментарий |
| --- | --- | --- | --- |
| геометрия | none, hflip, vflip, hvflip | 4 | повороты не используются, чтобы не деформировать контекст чашки |
| фотометрия | none, clahe, gamma, hsv, blur, noise | 6 | имитация освещения, контраста, фокуса и шума |
| PHOTO_REPEATS | для clahe/gamma/hsv/blur/noise | 3 | три случайных варианта каждого фотометрического режима |
| варианты на один crop | 4 * (1 + 5 * 3) | 64 | 1 none + 5 фотометрий по 3 повтора на каждую геометрию |
| исходных crop | cropped_736/images/train | 10 | ручная/подготовленная segmentation-разметка |
| итого augmented pool | cropped_736_aug_leaky/_pool | 640 | полный pool перед split |

Фотометрические режимы:

| режим | что меняет | зачем нужен |
| --- | --- | --- |
| `none` | без изменения изображения | сохраняет исходный вид |
| `clahe` | локальный контраст | имитирует разные условия видимости слабых колоний |
| `gamma` | нелинейная яркость | имитирует экспозицию/освещение |
| `hsv` | оттенок/насыщенность/яркость | повышает устойчивость к цветовым сдвигам камеры |
| `blur` | лёгкое размытие | имитирует фокус/оптическое размытие |
| `noise` | шум | имитирует шум сенсора и JPEG/освещение |

## Обучение YOLO26-seg для колоний

| параметр | значение |
| --- | --- |
| dataset | cropped_736_aug_leaky/dataset/data.yaml |
| imgsz | 736 |
| epochs | 50 |
| patience | 80 |
| batch в основном multi-model notebook | 8 |
| batch в VRAM-safe notebook | 1 |
| mask_ratio | 1 в основном сравнении / 2 в VRAM-safe варианте |
| overlap_mask | False в VRAM-safe варианте |
| MLflow experiment | colony_yolo_seg_736_4 / colony_yolo_seg_736 |
| папка эксперимента | runs/segment/runs/segment/runs/colony_seg_mlflow_736 |
| production model | yolo26x-seg_cropped736_offline_aug/weights/best.pt |

## Метрики YOLO26-seg моделей

Файл:

```text
runs/segment/runs/colony_seg_compare_736/metrics_20260423_094753/model_metrics_summary_736.csv
```

| model | box_P | box_R | box_mAP50 | box_mAP50_95 | seg_P | seg_R | seg_mAP50 | seg_mAP50_95 | fitness | infer_ms |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| yolo26l | 0.9953 | 0.4280 | 0.7123 | 0.6594 | 0.8675 | 0.3731 | 0.6328 | 0.3836 | 1.0430 | 38.1976 |
| yolo26m | 0.9932 | 0.4284 | 0.7121 | 0.6563 | 0.8687 | 0.3746 | 0.6337 | 0.3901 | 1.0464 | 30.6782 |
| yolo26n | 0.9623 | 0.4165 | 0.6949 | 0.5307 | 0.7878 | 0.3410 | 0.5708 | 0.2953 | 0.8260 | 17.9300 |
| yolo26s | 0.9849 | 0.4259 | 0.7075 | 0.5944 | 0.8405 | 0.3634 | 0.6138 | 0.3479 | 0.9424 | 19.2877 |
| yolo26x | 0.9988 | 0.4289 | 0.7138 | 0.6961 | 0.8701 | 0.3736 | 0.6341 | 0.3958 | 1.0919 | 70.3737 |

Интерпретация:

- `box_P`, `box_R`, `box_mAP50`, `box_mAP50_95` относятся к bbox части YOLO.
- `seg_P`, `seg_R`, `seg_mAP50`, `seg_mAP50_95` относятся к mask segmentation.
- `seg_mAP50_95` строже, чем `seg_mAP50`, потому что усредняет качество по IoU 0.50-0.95.
- `infer_ms` - время инференса на изображение в текущем окружении.
- `YOLO26x` выбран для production, потому что показывает лучший `seg_mAP50_95` среди сравниваемых моделей.

## Loss и обучение сегментаторов

| model | epochs | train_box | train_seg | train_cls | val_box | val_seg | val_cls | val_dfl |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| yolo26l | 51 | 0.4513 | 0.0085 | 0.1237 | 0.4884 | 0.7622 | 0.1550 | 0.0005 |
| yolo26m | 51 | 0.4580 | 0.0096 | 0.1246 | 0.5054 | 0.6796 | 0.1598 | 0.0006 |
| yolo26n | 51 | 0.8210 | 0.0222 | 0.2937 | 1.0345 | 3.3198 | 0.4170 | 0.0013 |
| yolo26s | 51 | 0.5603 | 0.0112 | 0.1641 | 0.7690 | 1.5736 | 0.2623 | 0.0009 |
| yolo26x | 51 | 0.4227 | 0.0076 | 0.1142 | 0.3468 | 0.5456 | 0.1013 | 0.0003 |

## Preview количества найденных масок

Файл:

```text
runs/segment/runs/colony_seg_predicts_736/predict_summary_cropped736.csv
```

| model | masks_4363 | masks_4615 | masks_4672 | masks_6137 |
| --- | --- | --- | --- | --- |
| yolo26l-seg_cropped736_offline_aug | 100 | 73 | 110 | 134 |
| yolo26m-seg_cropped736_offline_aug | 99 | 73 | 110 | 133 |
| yolo26n-seg_cropped736_offline_aug | 99 | 72 | 112 | 133 |
| yolo26s-seg_cropped736_offline_aug | 100 | 73 | 110 | 133 |
| yolo26x-seg_cropped736_offline_aug | 99 | 73 | 110 | 133 |

Это не основная метрика качества, но быстрый sanity-check: модели должны возвращать сопоставимое количество instance masks на одних и тех же контрольных изображениях.

## Финальный двухмодельный inference pipeline

| шаг | модель/метод | вход | выход | зачем нужен |
| --- | --- | --- | --- | --- |
| 1 | YOLO26s detect | raw photo | bbox чашки | локализует чашку на исходной фотографии |
| 2 | crop + resize | bbox чашки | 736x736 RGB | приводит вход к формату сегментатора |
| 3 | YOLO26x-seg | 736x736 crop | instance masks колоний | выделяет каждую колонию как отдельный объект |
| 4 | mask cleaning | сырые masks | clean non-overlapping masks | убирает фрагменты, дырки и пересечения |
| 5 | feature extraction | image + masks | таблица признаков | форма, цвет, яркость, текстура, фон, соседи |
| 6 | technical filtering | features | valid objects + warnings | исключает мусор из biological anomaly detection |
| 7 | unsupervised scoring | valid features | anomaly scores | оценивает нетипичность без меток |
| 8 | recommendation logic | scores + reliability | selected/review/technical | отделяет уверенные кандидаты от review и технических ошибок |
| 9 | reporting | tables + masks | CSV/XLSX/PNG/NPY | сохраняет проверяемый результат |

Конфигурация inference:

| группа | параметр | значение | комментарий |
| --- | --- | --- | --- |
| preprocess | preprocess_mode | detect_petri | для raw фото сначала ищется чашка |
| preprocess | petri_bbox_margin | 0.0100 | малый запас фона вокруг чашки |
| petri detector | conf / iou / max_det | 0.25 / 0.45 / 10 | поиск одной основной чашки |
| segmentation | imgsz | 736 | размер входа YOLO26x-seg |
| segmentation | conf | 0.1000 | понижен для recall; мусор отсекается фильтрами |
| segmentation | iou | 0.6500 | баланс между overlap и дубликатами |
| segmentation | max_det | 1000 | важно для плотных чашек с >300 колониями |
| mask filtering | min_area | 25 | мелкие фрагменты считаются техническими |
| mask filtering | max_area_fraction | 0.1000 | слишком крупные masks проверяются как сомнительные |
| texture | glcm_levels | 32 | квантование GLCM для устойчивости |
| neighbors | neighbor_k | 5 | локальное сравнение с ближайшими колониями |

## Что считается аномальной колонией

Аномальная колония в этом pipeline - это не заранее размеченный класс. Это объект, который статистически нетипичен относительно других валидных колоний на той же чашке Петри.

Score ближе к 1 означает более высокую нетипичность, но объект попадает в `selected_anomalies` только если кроме высокого score есть достаточная независимая доказательность и надёжность segmentation.

## Группы признаков anomaly detection

| группа | ключевые признаки | что означает |
| --- | --- | --- |
| size_shape_score | log_area, area_to_median_ratio, circularity, eccentricity, solidity, radial_contour_cv | размер, округлость, вытянутость, дефекты контура |
| color_background_score | Lab, S/V, mean_H_sin/cos, local_delta_L/a/b, local_color_delta_lab | цвет колонии и отличие от локального фона |
| intensity_score | intensity_iqr, entropy, center/rim delta, radial slope | яркость, неоднородность, профиль центр-периферия |
| texture_score | laplacian_var, GLCM, LBP entropy | зернистость, контраст, однородность и локальные паттерны |
| histogram_score | hist entropy, dark/bright fractions, JS/Wasserstein distances | распределение яркости/цвета внутри колонии |
| spatial_context_score | nearest distances, edge distances, local_density_r | изолированность и плотность окружения; малый вес |
| neighbor_difference_score | relative_*_vs_neighbors, feature_knn_distance | отличие от ближайших spatial и feature-space соседей |
| morphotype_score | within_morphotype_z, rare_morphotype_flag | отличие внутри вычисленного морфотипа |

## Что исключается из biological anomaly score

| категория | исключённые признаки | почему |
| --- | --- | --- |
| YOLO-служебные | yolo_conf, bbox_area_yolo, x1, y1, x2, y2 | это свойства модели/box, а не биологии колонии |
| технические флаги | technical_warning, too_small, too_large, low_yolo_conf, mask_fragment_after_overlap | не должны превращать мусор segmentation в биологическую аномалию |
| абсолютные координаты | centroid_x, centroid_y, centroid_x_norm, centroid_y_norm | колония не должна быть аномальной только потому, что она слева или сверху |
| нестабильные признаки | orientation, mean_H, std_H, intensity_min, intensity_max | orientation нестабилен; hue цикличен; min/max шумовые |
| край чашки | distance_to_plate_center, distance_to_plate_edge | используется как контекст/warning, но не как основное биологическое доказательство |

## Как считается итоговый anomaly score

Для валидных колоний считаются несколько независимых источников evidence: robust z-score, LOF, Isolation Forest, групповые scores, cluster outlier score и morphotype context.

Веса rank/evidence блоков:

| rank/evidence | вес | роль |
| --- | --- | --- |
| lof_score_rank | 0.1800 | локальная плотность в feature-space |
| isolation_forest_score_rank | 0.1200 | многомерный outlier score |
| robust_z_score_rank | 0.1200 | робастное отклонение по признакам |
| size_shape_score_rank | 0.1500 | форма и размер |
| texture_score_rank | 0.1500 | текстурная нетипичность |
| intensity_score_rank | 0.1000 | яркостный профиль |
| color_background_score_rank | 0.0800 | цвет и локальный фон |
| histogram_score_rank | 0.0800 | распределения яркости/цвета |
| neighbor_difference_score_rank | 0.0500 | отличие от соседей; ограниченный вес |
| spatial_context_score_rank | 0.0300 | пространство; очень малый вес |
| morphotype_score_rank | 0.0400 | отличие внутри морфотипа |
| cluster_outlier_score_rank | 0.0300 | диагностический cluster outlier, если есть сигнал |

Итоговые колонки:

| колонка | смысл |
| --- | --- |
| `final_anomaly_score_raw` | взвешенная сумма evidence-блоков; используется для объективного отбора |
| `final_anomaly_score_percentile` | percentile внутри текущей чашки; показывает относительное место объекта |
| `final_anomaly_score` | совместимая колонка для сортировки и визуализации; сейчас равна raw |
| `independent_evidence_count` | сколько независимых групп признаков поддержали аномальность |
| `consensus_score` | доля методов/групп, согласных с тем, что объект нетипичен |
| `overall_reliability_score` | надёжность segmentation/texture/background/shape признаков |

## Recommendation status

| status | попадает в selected_anomalies | смысл |
| --- | --- | --- |
| select_candidate | да | валидная колония с высоким score, достаточной доказательностью и надёжностью |
| review_only | нет | есть сигнал, но недостаточно независимых доказательств или надёжности |
| review_segmentation | нет | возможна слипшаяся или сомнительная маска |
| rare_morphotype | нет | редкий морфотип, но не обязательно аномалия |
| technical_exclude | нет | техническая ошибка/фрагмент/край/low confidence |
| weak_outlier | нет | попал бы в top по percentile, но raw/evidence слабые |
| no_valid_evidence | нет | нет достаточных оснований для отбора |

Правило отбора в `selected_anomalies`:

```text
valid_for_anomaly == True
technical_warning == False
recommendation_status == "select_candidate"
```

## Выходные файлы

| файл | уровень | содержание |
| --- | --- | --- |
| petri_crop_736.png | per image | crop чашки после первой модели |
| detections_yolo.csv | per image | YOLO confidence и bbox по masks |
| colony_features.csv | per image | все интерпретируемые признаки колоний |
| colony_anomaly_scores.csv | per image | scores, ranks, reliability, explanations, recommendation_status |
| selected_anomalies.csv | per image | только select_candidate |
| review_candidates.csv | per image | review_only/review_segmentation/rare_morphotype/weak_outlier |
| technical_warnings.csv | per image | технически исключённые объекты |
| morphotypes.csv | per image | морфотипы и размеры кластеров |
| masks_labeled.npy | per image | label mask: 0 фон, 1..N colony_id |
| anomalies_visualization.png | per image | визуализация выбранных и review объектов |
| petri_5_images_summary.csv | batch | сводка по 5 изображениям |
| petri_5_images_report.xlsx | batch | Excel-отчёт по summary/selected/review/technical |

## Как читать результат лабораторному специалисту

1. Открыть `anomalies_visualization.png` и посмотреть выделенные контуры.
2. Открыть `selected_anomalies.csv`: это основные кандидаты для дальнейшего анализа.
3. Если selected пустой, открыть `review_candidates.csv`: там могут быть редкие морфотипы или сомнительные объекты.
4. Проверить `technical_warnings.csv`: он показывает, какие объекты исключены как мусор segmentation или технические случаи.
5. Для выбранной колонии смотреть `explanations_text`: там указано, за счёт каких признаков объект нетипичен.
6. Если `recommendation_status=review_segmentation`, объект не стоит автоматически считать биологической аномалией: возможно, это слипшаяся или ошибочная маска.

## Ограничения текущей версии

- Сегментационный source dataset мал: 10 исходных crop 736x736.
- Текущий augmented split называется `leaky`; для строгой оценки нужен group split по исходному crop.
- `seg_recall` в сравнительной таблице ниже precision, поэтому часть колоний может пропускаться моделью.
- Anomaly detection является безучительственным: он ранжирует нетипичность, но не доказывает биологическую природу аномалии.
- Spatial context имеет малый вес намеренно: изолированность не должна сама по себе делать колонию аномальной.
- Технические флаги не удаляются из отчёта, но исключаются из biological selection.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "full_pipline":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Safety flags. Set only the section you need to True.
RUN_PETRI_DETECTOR_TRAINING = False
RUN_CROP_736_PREPARATION = False
RUN_COLONY_SEGMENTATION_TRAINING = False
RUN_TRAINING_SET_PREDICTIONS = False
RUN_FINAL_FULL_PIPELINE = False

print("Project root:", PROJECT_ROOT)
print("Training blocks are disabled by default.")


## Execution Order

1. Train or reuse the Petri dish detector.
2. Prepare/crop images to 736x736 if raw photos are used for segmentation training.
3. Train or reuse YOLO26x-seg for colony instance segmentation.
4. Run final full prediction: Petri detector -> crop/resize -> colony segmentation -> anomaly detection.

The local production weights are expected in `full_pipline/models`: `petri_detector_yolo26s_best.pt` and `colony_yolo26x_seg_best.pt`.


## 1. Petri Detector Training and 736 Crop Preparation

Source notebook: `C:/ColonyNet/train_yolo26n_s_petri_curcle.ipynb`.

Code cells in this section are guarded by `RUN_PETRI_DETECTOR_TRAINING`.


# Petri_curcle: split, train (`yolo26n.pt` + `yolo26s.pt`), test, crop 736x736


In [ ]:
# Source: C:\ColonyNet\train_yolo26n_s_petri_curcle.ipynb, cell 1
# This block runs only when RUN_PETRI_DETECTOR_TRAINING is True.
if RUN_PETRI_DETECTOR_TRAINING:
    # Uncomment if required
    # %pip install -q ultralytics pyyaml pandas matplotlib pillow opencv-python

    from pathlib import Path
    import random
    import shutil
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from PIL import Image
    import cv2
    import yaml
    from ultralytics import YOLO
    from IPython.display import display
    import torch

    plt.style.use("seaborn-v0_8-whitegrid")
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)


In [ ]:
# Source: C:\ColonyNet\train_yolo26n_s_petri_curcle.ipynb, cell 2
# This block runs only when RUN_PETRI_DETECTOR_TRAINING is True.
if RUN_PETRI_DETECTOR_TRAINING:
    # Configuration
    PROJECT_ROOT = Path.cwd()
    DATASET_ROOT = PROJECT_ROOT / "Petri_curcle"

    SOURCE_IMAGES = DATASET_ROOT / "images" / "train"
    SOURCE_LABELS = DATASET_ROOT / "labels" / "train"

    SPLIT_ROOT = DATASET_ROOT / "split_dataset"
    SPLITS = {"train": 0.7, "val": 0.2, "test": 0.1}
    assert abs(sum(SPLITS.values()) - 1.0) < 1e-9

    MODEL_CONFIGS = {
        "n": {"weights": "yolo26n.pt", "train_name": "yolo26n_petri_curcle"},
        "s": {"weights": "yolo26s.pt", "train_name": "yolo26s_petri_curcle"},
    }

    EPOCHS = 100
    IMGSZ = 736
    BATCH = 8
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

    TRAIN_PROJECT = "runs/petri_curcle"
    PREDICT_MODEL_KEY = "s"  # which trained model to use for preview + crop

    # Offline soft augmentation (files are physically added to split_dataset)
    USE_OFFLINE_SOFT_AUG = True
    AUG_TARGET_SPLITS = ["train"]
    AUG_MULTIPLIER = 5  # total factor including originals
    AUG_CLEAR_PREVIOUS = True

    # Online soft augmentation (Ultralytics train args)
    YOLO_TRAIN_AUG_ARGS = {
        "hsv_h": 0.01,
        "hsv_s": 0.22,
        "hsv_v": 0.14,
        "degrees": 3.0,
        "translate": 0.04,
        "scale": 0.08,
        "shear": 1.0,
        "perspective": 0.0,
        "fliplr": 0.5,
        "flipud": 0.0,
        "mosaic": 0.0,
        "mixup": 0.0,
        "copy_paste": 0.0,
        "erasing": 0.1,
    }

    # Crop settings for predict workflow
    TARGET_W = 736
    TARGET_H = 736
    CROP_CONF = 0.25
    CROP_IOU = 0.5
    OVERWRITE_CROPS = True

    print(f"Dataset root: {DATASET_ROOT.resolve()}")
    print(f"Source images: {SOURCE_IMAGES}")
    print(f"Source labels: {SOURCE_LABELS}")
    print(f"Model configs: {MODEL_CONFIGS}")
    print(f"Device: {DEVICE}")
    print(f"Offline aug enabled: {USE_OFFLINE_SOFT_AUG}, multiplier={AUG_MULTIPLIER}, splits={AUG_TARGET_SPLITS}")


In [ ]:
# Source: C:\ColonyNet\train_yolo26n_s_petri_curcle.ipynb, cell 3
# This block runs only when RUN_PETRI_DETECTOR_TRAINING is True.
if RUN_PETRI_DETECTOR_TRAINING:
    # Split source train set into train/val/test and create data_split.yaml
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    image_files = sorted([p for p in SOURCE_IMAGES.glob("*") if p.suffix.lower() in image_exts])

    pairs = []
    missing_labels = []
    for img in image_files:
        lbl = SOURCE_LABELS / f"{img.stem}.txt"
        if lbl.exists():
            pairs.append((img, lbl))
        else:
            missing_labels.append(img.name)

    if not pairs:
        raise RuntimeError("No image-label pairs found in source directories.")

    idx = list(range(len(pairs)))
    rng = random.Random(SEED)
    rng.shuffle(idx)

    n_total = len(idx)
    n_train = int(n_total * SPLITS["train"])
    n_val = int(n_total * SPLITS["val"])
    n_test = n_total - n_train - n_val

    split_indices = {
        "train": idx[:n_train],
        "val": idx[n_train:n_train + n_val],
        "test": idx[n_train + n_val:],
    }

    if SPLIT_ROOT.exists():
        shutil.rmtree(SPLIT_ROOT)

    for split in split_indices:
        (SPLIT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
        (SPLIT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

    for split, indices in split_indices.items():
        for i in indices:
            img, lbl = pairs[i]
            shutil.copy2(img, SPLIT_ROOT / "images" / split / img.name)
            shutil.copy2(lbl, SPLIT_ROOT / "labels" / split / lbl.name)

    data_yaml = {
        "path": str(SPLIT_ROOT.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "Petri_curcle"},
    }

    data_yaml_path = SPLIT_ROOT / "data_split.yaml"
    with data_yaml_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

    summary_df = pd.DataFrame(
        {
            "split": ["train", "val", "test"],
            "images": [len(split_indices["train"]), len(split_indices["val"]), len(split_indices["test"])],
        }
    )
    summary_df["labels"] = summary_df["images"]

    if missing_labels:
        print(f"Missing labels: {len(missing_labels)} files (ignored)")

    print(f"Total valid pairs: {len(pairs)}")
    print(f"Split root: {SPLIT_ROOT.resolve()}")
    print(f"data yaml: {data_yaml_path}")
    display(summary_df)


In [ ]:
# Source: C:\ColonyNet\train_yolo26n_s_petri_curcle.ipynb, cell 4
# This block runs only when RUN_PETRI_DETECTOR_TRAINING is True.
if RUN_PETRI_DETECTOR_TRAINING:
    # Soft offline augmentation for Petri_curcle/split_dataset
    # Creates extra files like IMG_1234__soft01.jpg + IMG_1234__soft01.txt
    import re

    soft_suffix_re = re.compile(r"__soft\d{2}$")
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


    def gamma_correct(image, gamma):
        lut = np.array([((x / 255.0) ** gamma) * 255.0 for x in range(256)], dtype=np.float32)
        lut = np.clip(lut, 0, 255).astype(np.uint8)
        return cv2.LUT(image, lut)


    def mild_color_jitter(image, rng):
        out = image.astype(np.float32)
        alpha = float(rng.uniform(0.95, 1.07))
        beta = float(rng.uniform(-8.0, 8.0))
        out = np.clip(out * alpha + beta, 0, 255).astype(np.uint8)
        out = gamma_correct(out, gamma=float(rng.uniform(0.93, 1.07)))

        hsv = cv2.cvtColor(out, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 1] *= float(rng.uniform(0.93, 1.08))
        hsv[..., 2] *= float(rng.uniform(0.95, 1.05))
        return cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR)


    def mild_noise_blur(image, rng):
        sigma_noise = float(rng.uniform(2.0, 5.0))
        noisy = image.astype(np.float32) + rng.normal(0.0, sigma_noise, size=image.shape).astype(np.float32)
        noisy = np.clip(noisy, 0, 255).astype(np.uint8)
        k = 3 if float(rng.random()) < 0.75 else 5
        sigma_blur = float(rng.uniform(0.2, 0.9))
        return cv2.GaussianBlur(noisy, (k, k), sigmaX=sigma_blur)


    def mild_clahe(image, rng):
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=float(rng.uniform(1.2, 1.8)), tileGridSize=(8, 8))
        l2 = clahe.apply(l)
        out = cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)
        return mild_color_jitter(out, rng)


    def mild_sharpen(image, rng):
        blur = cv2.GaussianBlur(image, (0, 0), sigmaX=float(rng.uniform(0.4, 0.8)))
        return cv2.addWeighted(image, 1.12, blur, -0.12, 0.0)


    def soft_augment(image, variant_idx, rng):
        mode = (variant_idx - 1) % 4
        if mode == 0:
            return mild_color_jitter(image, rng)
        if mode == 1:
            return mild_noise_blur(image, rng)
        if mode == 2:
            return mild_clahe(image, rng)
        return mild_sharpen(mild_color_jitter(image, rng), rng)


    def collect_pairs(images_dir, labels_dir):
        pairs = []
        for img_path in sorted(images_dir.glob("*")):
            if not img_path.is_file() or img_path.suffix.lower() not in image_exts:
                continue
            if soft_suffix_re.search(img_path.stem):
                continue
            lbl_path = labels_dir / f"{img_path.stem}.txt"
            if lbl_path.exists():
                pairs.append((img_path, lbl_path))
        return pairs


    def clear_previous_soft(images_dir, labels_dir):
        removed = 0
        for p in images_dir.glob("*"):
            if p.is_file() and soft_suffix_re.search(p.stem):
                p.unlink()
                removed += 1
        for p in labels_dir.glob("*.txt"):
            if p.is_file() and soft_suffix_re.search(p.stem):
                p.unlink()
                removed += 1
        return removed


    aug_report = {
        "dataset_root": str(SPLIT_ROOT.resolve()),
        "splits": {},
        "settings": {
            "use_offline_soft_aug": bool(USE_OFFLINE_SOFT_AUG),
            "aug_target_splits": list(AUG_TARGET_SPLITS),
            "aug_multiplier": int(AUG_MULTIPLIER),
            "aug_clear_previous": bool(AUG_CLEAR_PREVIOUS),
        },
    }

    if not USE_OFFLINE_SOFT_AUG:
        print("Offline augmentation disabled, skipping.")
    else:
        if AUG_MULTIPLIER < 2:
            raise ValueError("AUG_MULTIPLIER must be >= 2")

        rng = np.random.default_rng(SEED)
        for split in AUG_TARGET_SPLITS:
            images_dir = SPLIT_ROOT / "images" / split
            labels_dir = SPLIT_ROOT / "labels" / split
            if not images_dir.exists() or not labels_dir.exists():
                raise FileNotFoundError(f"Split not found: {split}")

            removed = clear_previous_soft(images_dir, labels_dir) if AUG_CLEAR_PREVIOUS else 0
            base_pairs = collect_pairs(images_dir, labels_dir)

            created = 0
            failed = 0
            for img_path, lbl_path in base_pairs:
                image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
                if image is None:
                    failed += (AUG_MULTIPLIER - 1)
                    continue

                label_text = lbl_path.read_text(encoding="utf-8")
                for aug_idx in range(1, AUG_MULTIPLIER):
                    out_stem = f"{img_path.stem}__soft{aug_idx:02d}"
                    out_img = images_dir / f"{out_stem}{img_path.suffix.lower()}"
                    out_lbl = labels_dir / f"{out_stem}.txt"

                    aug_img = soft_augment(image, aug_idx, rng)
                    ok = cv2.imwrite(str(out_img), aug_img)
                    if not ok:
                        failed += 1
                        continue
                    out_lbl.write_text(label_text, encoding="utf-8")
                    created += 1

            all_img_stems = {
                p.stem for p in images_dir.glob("*")
                if p.is_file() and p.suffix.lower() in image_exts
            }
            all_lbl_stems = {p.stem for p in labels_dir.glob("*.txt") if p.is_file()}
            matched = len(all_img_stems & all_lbl_stems)

            aug_report["splits"][split] = {
                "base_pairs": len(base_pairs),
                "created_pairs": created,
                "failed_pairs": failed,
                "removed_previous_soft_files": removed,
                "matched_pairs_after": matched,
                "expected_after": len(base_pairs) * AUG_MULTIPLIER,
            }

            print(
                f"[{split}] base={len(base_pairs)} created={created} failed={failed} "
                f"matched_after={matched} expected={len(base_pairs) * AUG_MULTIPLIER}"
            )

        aug_report["generated_at_utc"] = pd.Timestamp.utcnow().isoformat()
        aug_report_path = SPLIT_ROOT / "soft_aug_report_notebook.json"
        with aug_report_path.open("w", encoding="utf-8") as f:
            json.dump(aug_report, f, ensure_ascii=False, indent=2)
        print(f"Saved augmentation report: {aug_report_path}")


In [ ]:
# Source: C:\ColonyNet\train_yolo26n_s_petri_curcle.ipynb, cell 5
# This block runs only when RUN_PETRI_DETECTOR_TRAINING is True.
if RUN_PETRI_DETECTOR_TRAINING:
    # Train YOLO for all configured models (n + s)
    train_runs = {}

    for model_key, cfg in MODEL_CONFIGS.items():
        print(f"\n=== Training model: {model_key} ({cfg['weights']}) ===")
        model = YOLO(cfg["weights"])

        _ = model.train(
            data='C:/ColonyNet/Petri_curcle/split_dataset/data_split.yaml',
            epochs=50,
            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            project=TRAIN_PROJECT,
            name=cfg["train_name"],
            exist_ok=True,
            plots=True,
            **YOLO_TRAIN_AUG_ARGS,
        )

        run_dir = Path(model.trainer.save_dir)
        best_weights = run_dir / "weights" / "best.pt"
        if not best_weights.exists():
            raise FileNotFoundError(f"Best checkpoint not found: {best_weights}")

        train_runs[model_key] = {
            "weights": cfg["weights"],
            "run_dir": run_dir,
            "best_weights": best_weights,
        }

    train_runs_df = pd.DataFrame(
        [
            {
                "model": k,
                "base_weights": v["weights"],
                "run_dir": str(v["run_dir"]),
                "best_weights": str(v["best_weights"]),
            }
            for k, v in train_runs.items()
        ]
    )
    display(train_runs_df)


In [ ]:
# Source: C:\ColonyNet\train_yolo26n_s_petri_curcle.ipynb, cell 6
# This block runs only when RUN_PETRI_DETECTOR_TRAINING is True.
if RUN_PETRI_DETECTOR_TRAINING:
    # Evaluate both best checkpoints on test split

    def to_float(v):
        try:
            return float(v)
        except Exception:
            return float("nan")


    eval_rows = []
    for model_key, info in train_runs.items():
        best_model = YOLO(str(info["best_weights"]))
        test_metrics = best_model.val(
            data=str(data_yaml_path),
            split="test",
            project=TRAIN_PROJECT,
            name=f"{MODEL_CONFIGS[model_key]['train_name']}_test",
            exist_ok=True,
            plots=True,
            save_json=True,
        )

        eval_rows.append(
            {
                "model": model_key,
                "precision_B": to_float(test_metrics.box.mp),
                "recall_B": to_float(test_metrics.box.mr),
                "mAP50_B": to_float(test_metrics.box.map50),
                "mAP50_95_B": to_float(test_metrics.box.map),
                "fitness": to_float(getattr(test_metrics, "fitness", float("nan"))),
                "test_dir": str(Path(test_metrics.save_dir)),
            }
        )

    metrics_df = pd.DataFrame(eval_rows).sort_values("mAP50_95_B", ascending=False).reset_index(drop=True)
    display(metrics_df)

    metrics_csv_path = SPLIT_ROOT / "multi_model_test_metrics.csv"
    metrics_df.to_csv(metrics_csv_path, index=False)
    print(f"Saved metrics table: {metrics_csv_path}")

    predict_model_key = PREDICT_MODEL_KEY if PREDICT_MODEL_KEY in train_runs else metrics_df.loc[0, "model"]
    predict_weights = train_runs[predict_model_key]["best_weights"]
    print(f"Predict/crop model key: {predict_model_key}")
    print(f"Predict/crop weights: {predict_weights}")


In [ ]:
# Source: C:\ColonyNet\train_yolo26n_s_petri_curcle.ipynb, cell 7
# This block runs only when RUN_PETRI_DETECTOR_TRAINING is True.
if RUN_PETRI_DETECTOR_TRAINING:
    # Show sample predictions on test images for selected model
    sample_images = sorted((SPLIT_ROOT / "images" / "test").glob("*"))
    if not sample_images:
        print("No test images found for prediction preview.")
    else:
        sample_images = sample_images[:6]
        pred_model = YOLO(str(predict_weights))
        preds = pred_model.predict(
            source=[str(p) for p in sample_images],
            conf=0.25,
            iou=0.5,
            imgsz=max(TARGET_W, TARGET_H),
            verbose=False,
        )

        n = len(sample_images)
        ncols = 3
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
        axes = np.array(axes).reshape(-1)

        for ax in axes:
            ax.axis("off")

        for ax, pred, src in zip(axes, preds, sample_images):
            annotated = pred.plot()  # BGR
            ax.imshow(annotated[..., ::-1])
            ax.set_title(src.name)
            ax.axis("off")

        plt.tight_layout()
        plt.show()


## 2. Detection-Based Crop to 736x736

Source notebook: `C:/ColonyNet/train_yolo26n_s_petri_curcle.ipynb`.

Code cells in this section are guarded by `RUN_CROP_736_PREPARATION`.


## Crop Detected Petri Dishes and Resize to 736x736


In [ ]:
# Source: C:\ColonyNet\train_yolo26n_s_petri_curcle.ipynb, cell 9
# This block runs only when RUN_CROP_736_PREPARATION is True.
if RUN_CROP_736_PREPARATION:
    # Crop inference configuration
    CROP_SOURCE_DIR = DATASET_ROOT / "images" / "train"
    CROP_OUTPUT_DIR = DATASET_ROOT / f"cropped_{TARGET_W}"
    CROP_REPORT_CSV = CROP_OUTPUT_DIR / "crop_report.csv"
    CROP_MODEL_PATH = predict_weights

    print(f"Crop model: {Path(CROP_MODEL_PATH)}")
    print(f"Source dir: {CROP_SOURCE_DIR}")
    print(f"Output dir: {CROP_OUTPUT_DIR}")
    print(f"Target size: {TARGET_W}x{TARGET_H}")


In [ ]:
# Source: C:\ColonyNet\train_yolo26n_s_petri_curcle.ipynb, cell 10
# This block runs only when RUN_CROP_736_PREPARATION is True.
if RUN_CROP_736_PREPARATION:
    # Run detection-based cropping and resize to 736x736
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

    if not CROP_SOURCE_DIR.exists():
        raise FileNotFoundError(f"Source directory not found: {CROP_SOURCE_DIR}")
    if not Path(CROP_MODEL_PATH).exists():
        raise FileNotFoundError(f"Model weights not found: {CROP_MODEL_PATH}")

    if OVERWRITE_CROPS and CROP_OUTPUT_DIR.exists():
        shutil.rmtree(CROP_OUTPUT_DIR)
    CROP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    crop_model = YOLO(str(CROP_MODEL_PATH))
    source_images = sorted([p for p in CROP_SOURCE_DIR.glob("*") if p.suffix.lower() in image_exts])

    if not source_images:
        raise RuntimeError(f"No images found in: {CROP_SOURCE_DIR}")

    rows = []
    for i, img_path in enumerate(source_images, start=1):
        img = cv2.imread(str(img_path))
        if img is None:
            rows.append({"file": img_path.name, "status": "read_error"})
            continue

        h, w = img.shape[:2]
        pred = crop_model.predict(
            source=str(img_path),
            conf=CROP_CONF,
            iou=CROP_IOU,
            imgsz=max(TARGET_W, TARGET_H),
            verbose=False,
        )[0]

        if pred.boxes is None or len(pred.boxes) == 0:
            rows.append({"file": img_path.name, "status": "no_detection", "orig_w": w, "orig_h": h})
            continue

        confs = pred.boxes.conf.detach().cpu().numpy()
        boxes = pred.boxes.xyxy.detach().cpu().numpy()
        best_idx = int(np.argmax(confs))

        x1, y1, x2, y2 = boxes[best_idx]
        x1 = max(0, int(np.floor(x1)))
        y1 = max(0, int(np.floor(y1)))
        x2 = min(w, int(np.ceil(x2)))
        y2 = min(h, int(np.ceil(y2)))

        if x2 <= x1 or y2 <= y1:
            rows.append({"file": img_path.name, "status": "invalid_bbox", "orig_w": w, "orig_h": h})
            continue

        crop = img[y1:y2, x1:x2]
        interp = cv2.INTER_AREA if crop.shape[1] >= TARGET_W and crop.shape[0] >= TARGET_H else cv2.INTER_LINEAR
        resized = cv2.resize(crop, (TARGET_W, TARGET_H), interpolation=interp)

        out_path = CROP_OUTPUT_DIR / img_path.name
        cv2.imwrite(str(out_path), resized)

        rows.append(
            {
                "file": img_path.name,
                "status": "saved",
                "conf": float(confs[best_idx]),
                "orig_w": w,
                "orig_h": h,
                "bbox_w": x2 - x1,
                "bbox_h": y2 - y1,
                "out_w": TARGET_W,
                "out_h": TARGET_H,
                "out_path": str(out_path),
            }
        )

        if i % 25 == 0 or i == len(source_images):
            print(f"Processed {i}/{len(source_images)}")

    report_df = pd.DataFrame(rows)
    status_counts = report_df["status"].value_counts().rename_axis("status").reset_index(name="count")
    display(status_counts)

    report_df.to_csv(CROP_REPORT_CSV, index=False)
    print(f"Crop output: {CROP_OUTPUT_DIR.resolve()}")
    print(f"Report CSV: {CROP_REPORT_CSV.resolve()}")

    saved_df = report_df[report_df["status"] == "saved"].copy()
    if saved_df.empty:
        print("No saved crops to display.")
    else:
        preview_paths = [Path(p) for p in saved_df["out_path"].head(6).tolist()]

        n = len(preview_paths)
        cols = 3
        rows_n = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 4 * rows_n))
        axes = np.array(axes).reshape(-1)

        for ax in axes:
            ax.axis("off")

        for ax, p in zip(axes, preview_paths):
            img = Image.open(p)
            ax.imshow(img)
            ax.set_title(p.name)
            ax.axis("off")

        plt.tight_layout()
        plt.show()


## 3. Segmentation Dataset Preparation: Crop, Remap, Offline Augmentation

Source notebook: `C:/ColonyNet/yolo_colonies_mlflow_736_ maskratio_1.ipynb`.

Code cells in this section are guarded by `RUN_CROP_736_PREPARATION`.


# YOLO Colonies + MLflow (736, mask_ratio=2)

Pipeline in this notebook:
1. Detection-based crop of Petri dish + resize to 1024x1024 with label remap
2. Same offline augmentations as before (leaky split strategy by request)
3. Train YOLO-seg models with MLflow (`imgsz=736`, `mask_ratio=2`)


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow_736_ maskratio_1.ipynb, cell 1
# This block runs only when RUN_CROP_736_PREPARATION is True.
if RUN_CROP_736_PREPARATION:
    from pathlib import Path
    import random
    import shutil
    import zlib

    import cv2
    import numpy as np
    import pandas as pd
    from ultralytics import YOLO

    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)

    # -------- Source (before Petri crop) --------
    RAW_ROOT = Path("Новая папка")
    RAW_IMG_DIR = RAW_ROOT / "images" / "train"
    RAW_LBL_DIR = RAW_ROOT / "labels" / "train"

    # -------- Petri detector --------
    CROP_WEIGHTS = Path("runs/detect/runs/petri_curcle/yolo26n_petri_curcle/weights/best.pt")
    CROP_CONF = 0.25
    CROP_IOU = 0.5

    # -------- Cropped dataset (target 1024x1024) --------
    DATASET_ROOT = Path("cropped_736")
    SRC_IMG_DIR = DATASET_ROOT / "images" / "train"
    SRC_LBL_DIR = DATASET_ROOT / "labels" / "train"
    TARGET_W = 736
    TARGET_H = 736
    OVERWRITE_CROPPED = True

    # -------- Offline-aug workspace --------
    WORK_ROOT = DATASET_ROOT.parent / "cropped_736_aug_leaky"
    POOL_IMG_DIR = WORK_ROOT / "_pool" / "images"
    POOL_LBL_DIR = WORK_ROOT / "_pool" / "labels"
    SPLIT_ROOT = WORK_ROOT / "dataset"

    for req in [RAW_IMG_DIR, RAW_LBL_DIR]:
        if not req.exists():
            raise FileNotFoundError(f"Required path not found: {req}")
    if not CROP_WEIGHTS.exists():
        raise FileNotFoundError(f"Crop model not found: {CROP_WEIGHTS}")

    print(f"RAW_ROOT: {RAW_ROOT.resolve()}")
    print(f"DATASET_ROOT (cropped): {DATASET_ROOT.resolve()}")
    print(f"WORK_ROOT: {WORK_ROOT.resolve()}")
    print(f"CROP_WEIGHTS: {CROP_WEIGHTS.resolve()}")


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow_736_ maskratio_1.ipynb, cell 2
# This block runs only when RUN_CROP_736_PREPARATION is True.
if RUN_CROP_736_PREPARATION:
    # Detection-based crop + label remap -> cropped_736

    def load_yolo_seg_crop(txt_path: Path):
        anns = []
        if not txt_path.exists():
            return anns
        text = txt_path.read_text(encoding="utf-8", errors="ignore")
        text = text.replace("\r", "\n").replace("\n", "\n")
        for raw in text.splitlines():
            line = raw.strip()
            if not line:
                continue
            parts = line.replace(",", " ").split()
            if len(parts) < 7:
                continue
            cls = parts[0]
            coords = []
            for tok in parts[1:]:
                try:
                    coords.append(float(tok))
                except ValueError:
                    pass
            if len(coords) < 6:
                continue
            if len(coords) % 2 != 0:
                coords = coords[:-1]
            if len(coords) < 6:
                continue
            pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
            anns.append((cls, pts))
        return anns


    def transform_seg_crop(coords_norm, img_w, img_h, x1, y1, x2, y2):
        pts = coords_norm.copy()
        pts[:, 0] *= img_w
        pts[:, 1] *= img_h

        min_x, max_x = float(pts[:, 0].min()), float(pts[:, 0].max())
        min_y, max_y = float(pts[:, 1].min()), float(pts[:, 1].max())
        if max_x <= x1 or min_x >= x2 or max_y <= y1 or min_y >= y2:
            return None

        pts[:, 0] = np.clip(pts[:, 0], x1, x2 - 1e-6)
        pts[:, 1] = np.clip(pts[:, 1], y1, y2 - 1e-6)

        crop_w = max(1e-6, float(x2 - x1))
        crop_h = max(1e-6, float(y2 - y1))
        pts[:, 0] = (pts[:, 0] - x1) / crop_w
        pts[:, 1] = (pts[:, 1] - y1) / crop_h
        pts = np.clip(pts, 0.0, 1.0)

        if (pts[:, 0].max() - pts[:, 0].min()) < 1e-4:
            return None
        if (pts[:, 1].max() - pts[:, 1].min()) < 1e-4:
            return None
        return pts


    def write_yolo_seg_crop(txt_path: Path, anns):
        txt_path.parent.mkdir(parents=True, exist_ok=True)
        lines = []
        for cls, pts in anns:
            pts = np.clip(pts, 0.0, 1.0)
            flat = pts.reshape(-1)
            lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in flat))
        txt_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


    if OVERWRITE_CROPPED and DATASET_ROOT.exists():
        shutil.rmtree(DATASET_ROOT)

    SRC_IMG_DIR.mkdir(parents=True, exist_ok=True)
    SRC_LBL_DIR.mkdir(parents=True, exist_ok=True)

    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    images = sorted([p for p in RAW_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in image_exts])
    if not images:
        raise RuntimeError(f"No images found in: {RAW_IMG_DIR}")

    crop_model = YOLO(str(CROP_WEIGHTS))

    saved = 0
    fallback_full = 0
    dropped_masks = 0

    for i, img_path in enumerate(images, start=1):
        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img is None:
            continue

        h, w = img.shape[:2]
        pred = crop_model.predict(source=str(img_path), conf=CROP_CONF, iou=CROP_IOU, verbose=False)[0]

        if pred.boxes is None or len(pred.boxes) == 0:
            x1, y1, x2, y2 = 0, 0, w, h
            fallback_full += 1
        else:
            confs = pred.boxes.conf.detach().cpu().numpy()
            boxes = pred.boxes.xyxy.detach().cpu().numpy()
            best_idx = int(np.argmax(confs))
            bx1, by1, bx2, by2 = boxes[best_idx]
            x1 = max(0, int(np.floor(bx1)))
            y1 = max(0, int(np.floor(by1)))
            x2 = min(w, int(np.ceil(bx2)))
            y2 = min(h, int(np.ceil(by2)))
            if x2 <= x1 or y2 <= y1:
                x1, y1, x2, y2 = 0, 0, w, h
                fallback_full += 1

        crop = img[y1:y2, x1:x2]
        interp = cv2.INTER_AREA if crop.shape[1] >= TARGET_W and crop.shape[0] >= TARGET_H else cv2.INTER_LINEAR
        resized = cv2.resize(crop, (TARGET_W, TARGET_H), interpolation=interp)

        out_img = SRC_IMG_DIR / img_path.name
        cv2.imwrite(str(out_img), resized)

        in_lbl = RAW_LBL_DIR / f"{img_path.stem}.txt"
        anns = load_yolo_seg_crop(in_lbl)

        out_anns = []
        for cls, coords in anns:
            t = transform_seg_crop(coords, w, h, x1, y1, x2, y2)
            if t is None:
                dropped_masks += 1
                continue
            out_anns.append((cls, t))

        out_lbl = SRC_LBL_DIR / f"{img_path.stem}.txt"
        write_yolo_seg_crop(out_lbl, out_anns)

        saved += 1
        if i % 25 == 0 or i == len(images):
            print(f"Processed {i}/{len(images)}")

    print(f"Saved cropped images: {saved}")
    print(f"Fallback full-image crops: {fallback_full}")
    print(f"Dropped polygons after crop: {dropped_masks}")
    print(f"Cropped images dir: {SRC_IMG_DIR.resolve()}")


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow_736_ maskratio_1.ipynb, cell 3
# This block runs only when RUN_CROP_736_PREPARATION is True.
if RUN_CROP_736_PREPARATION:
    # Offline augmentation + split (same scheme as previous notebook)
    OVERWRITE_WORK_ROOT = True

    GEOM_MODES = ["none", "hflip", "vflip", "hvflip"]
    PHOTO_MODES = ["none", "clahe", "gamma", "hsv", "blur", "noise"]
    PHOTO_REPEATS = 3

    VARIANTS = []
    for geom in GEOM_MODES:
        for photo in PHOTO_MODES:
            if photo == "none":
                name = "orig" if geom == "none" else f"{geom}_orig"
                VARIANTS.append({"name": name, "geom": geom, "photo": photo})
            else:
                for rep in range(1, PHOTO_REPEATS + 1):
                    VARIANTS.append({"name": f"{geom}_{photo}_r{rep}", "geom": geom, "photo": photo})

    IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    OUTPUT_IMAGE_EXT = ".jpg"


    def read_yolo_seg(label_path: Path):
        anns = []
        if not label_path.exists():
            return anns

        text = label_path.read_text(encoding="utf-8", errors="ignore")
        text = text.replace("\r", "\n").replace("\n", "\n")

        for raw in text.splitlines():
            line = raw.strip()
            if not line:
                continue
            parts = line.replace(",", " ").split()
            if len(parts) < 7:
                continue

            cls = parts[0]
            coords = []
            for token in parts[1:]:
                try:
                    coords.append(float(token))
                except ValueError:
                    pass

            if len(coords) < 6:
                continue
            if len(coords) % 2 != 0:
                coords = coords[:-1]
            if len(coords) < 6:
                continue

            pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
            anns.append((cls, pts))
        return anns


    def write_yolo_seg(label_path: Path, anns):
        label_path.parent.mkdir(parents=True, exist_ok=True)
        lines = []
        for cls, pts in anns:
            pts = np.clip(pts, 0.0, 1.0)
            flat = pts.reshape(-1)
            lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in flat))
        label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


    def transform_points(pts, geom):
        out = pts.copy()
        if geom in {"hflip", "hvflip"}:
            out[:, 0] = 1.0 - out[:, 0]
        if geom in {"vflip", "hvflip"}:
            out[:, 1] = 1.0 - out[:, 1]
        return np.clip(out, 0.0, 1.0)


    def apply_geom(image, anns, geom):
        if geom == "none":
            return image, anns
        if geom == "hflip":
            out_img = cv2.flip(image, 1)
        elif geom == "vflip":
            out_img = cv2.flip(image, 0)
        elif geom == "hvflip":
            out_img = cv2.flip(image, -1)
        else:
            raise ValueError(f"Unknown geom transform: {geom}")

        out_anns = [(cls, transform_points(pts, geom)) for cls, pts in anns]
        return out_img, out_anns


    def apply_photo(image, photo, rng):
        if photo == "none":
            return image

        img = image.copy()

        if photo == "clahe":
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            clip = float(rng.uniform(2.0, 4.0))
            clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
            l2 = clahe.apply(l)
            return cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)

        if photo == "gamma":
            gamma = float(rng.uniform(0.7, 1.5))
            inv = 1.0 / gamma
            table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)], dtype=np.uint8)
            return cv2.LUT(img, table)

        if photo == "hsv":
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
            h_gain = float(rng.uniform(-8, 8))
            s_gain = float(rng.uniform(0.75, 1.35))
            v_gain = float(rng.uniform(0.75, 1.35))
            hsv[..., 0] = (hsv[..., 0] + h_gain) % 180
            hsv[..., 1] = np.clip(hsv[..., 1] * s_gain, 0, 255)
            hsv[..., 2] = np.clip(hsv[..., 2] * v_gain, 0, 255)
            return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

        if photo == "blur":
            k = int(rng.choice([3, 5]))
            sigma = float(rng.uniform(0.3, 1.2))
            return cv2.GaussianBlur(img, (k, k), sigmaX=sigma)

        if photo == "noise":
            sigma = float(rng.uniform(4.0, 14.0))
            noise = rng.normal(0, sigma, size=img.shape).astype(np.float32)
            out = np.clip(img.astype(np.float32) + noise, 0, 255)
            return out.astype(np.uint8)

        raise ValueError(f"Unknown photo transform: {photo}")


    if OVERWRITE_WORK_ROOT and WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)

    POOL_IMG_DIR.mkdir(parents=True, exist_ok=True)
    POOL_LBL_DIR.mkdir(parents=True, exist_ok=True)

    src_images = sorted([p for p in SRC_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    if not src_images:
        raise RuntimeError(f"No source images found in: {SRC_IMG_DIR}")

    written = 0
    for i, img_path in enumerate(src_images, start=1):
        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img is None:
            continue

        in_label = SRC_LBL_DIR / f"{img_path.stem}.txt"
        anns = read_yolo_seg(in_label)

        for var in VARIANTS:
            out_stem = f"{img_path.stem}__{var['name']}"
            seed_local = SEED + zlib.crc32(out_stem.encode("utf-8"))
            rng = np.random.default_rng(seed_local)

            g_img, g_anns = apply_geom(img, anns, var["geom"])
            p_img = apply_photo(g_img, var["photo"], rng)

            out_img = POOL_IMG_DIR / f"{out_stem}{OUTPUT_IMAGE_EXT}"
            out_lbl = POOL_LBL_DIR / f"{out_stem}.txt"

            cv2.imwrite(str(out_img), p_img)
            write_yolo_seg(out_lbl, g_anns)
            written += 1

        if i % 10 == 0 or i == len(src_images):
            print(f"Augmented {i}/{len(src_images)} source images")

    print(f"Source images: {len(src_images)}")
    print(f"Variants per image: {len(VARIANTS)}")
    print(f"Pool samples written: {written}")

    # Split augmented pool into train/val/test AFTER augmentation (leaky by design)
    TRAIN_RATIO = 0.80
    VAL_RATIO = 0.10
    TEST_RATIO = 0.10
    SPLIT_SEED = 42

    if abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1.0")

    pool_images = sorted([p for p in POOL_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    pool_stems = [p.stem for p in pool_images]

    rng = random.Random(SPLIT_SEED)
    rng.shuffle(pool_stems)

    n = len(pool_stems)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    n_test = n - n_train - n_val

    splits = {
        "train": pool_stems[:n_train],
        "val": pool_stems[n_train:n_train + n_val],
        "test": pool_stems[n_train + n_val:],
    }

    for split_name in ["train", "val", "test"]:
        (SPLIT_ROOT / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (SPLIT_ROOT / "labels" / split_name).mkdir(parents=True, exist_ok=True)

    for split_name, stems in splits.items():
        img_dst = SPLIT_ROOT / "images" / split_name
        lbl_dst = SPLIT_ROOT / "labels" / split_name

        for stem in stems:
            src_img = POOL_IMG_DIR / f"{stem}{OUTPUT_IMAGE_EXT}"
            src_lbl = POOL_LBL_DIR / f"{stem}.txt"

            shutil.copy2(src_img, img_dst / src_img.name)
            if src_lbl.exists():
                shutil.copy2(src_lbl, lbl_dst / src_lbl.name)
            else:
                (lbl_dst / f"{stem}.txt").write_text("", encoding="utf-8")

    print(f"Total: {n}")
    print(f"Train: {len(splits['train'])}")
    print(f"Val:   {len(splits['val'])}")
    print(f"Test:  {len(splits['test'])}")

    # Write data config for YOLO
    DATA_YAML = SPLIT_ROOT / "data.yaml"
    train_dir = (SPLIT_ROOT / "images" / "train").resolve().as_posix()
    val_dir = (SPLIT_ROOT / "images" / "val").resolve().as_posix()
    test_dir = (SPLIT_ROOT / "images" / "test").resolve().as_posix()

    yaml_text = "\n".join([
        f'train: "{train_dir}"',
        f'val: "{val_dir}"',
        f'test: "{test_dir}"',
        "nc: 1",
        "names: [colony]",
    ]) + "\n"

    DATA_YAML.write_text(yaml_text, encoding="utf-8")
    print("DATA_YAML:", DATA_YAML.resolve())
    print(yaml_text)



    # Optional: simplify train polygons only (keep val/test unchanged)
    SIMPLIFY_TRAIN_LABELS = True
    SIMPLIFY_EPS_FRAC = 0.0035      # epsilon = frac * polygon_perimeter
    SIMPLIFY_MIN_VERTICES = 12      # keep round colonies reasonably smooth
    SIMPLIFY_MAX_AREA_DELTA = 0.10  # reject simplification if area shifts >10%
    SIMPLIFY_BACKUP_ORIGINAL = True


    def simplify_polygon_safe(pts_norm, img_w, img_h, eps_frac, min_vertices, max_area_delta):
        """Soft Douglas-Peucker simplification with safety checks."""
        pts = np.asarray(pts_norm, dtype=np.float32)
        if pts.ndim != 2 or pts.shape[1] != 2 or pts.shape[0] < 3:
            return pts

        cnt = np.empty_like(pts)
        cnt[:, 0] = np.clip(pts[:, 0] * (img_w - 1), 0, img_w - 1)
        cnt[:, 1] = np.clip(pts[:, 1] * (img_h - 1), 0, img_h - 1)
        cnt_cv = cnt.reshape(-1, 1, 2).astype(np.float32)

        perimeter = float(cv2.arcLength(cnt_cv, True))
        if not np.isfinite(perimeter) or perimeter <= 1e-6:
            return pts

        orig_area = float(abs(cv2.contourArea(cnt_cv)))
        if not np.isfinite(orig_area) or orig_area <= 1e-6:
            return pts

        eps = max(0.5, float(eps_frac) * perimeter)
        approx_cv = cv2.approxPolyDP(cnt_cv, epsilon=eps, closed=True)

        if approx_cv is None:
            return pts

        approx = approx_cv.reshape(-1, 2)
        if approx.shape[0] < max(3, int(min_vertices)):
            return pts

        new_area = float(abs(cv2.contourArea(approx_cv)))
        if not np.isfinite(new_area) or new_area <= 1e-6:
            return pts

        rel_area_delta = abs(new_area - orig_area) / max(orig_area, 1e-6)
        if rel_area_delta > float(max_area_delta):
            return pts

        out = np.empty_like(approx, dtype=np.float32)
        out[:, 0] = np.clip(approx[:, 0] / max(1, (img_w - 1)), 0.0, 1.0)
        out[:, 1] = np.clip(approx[:, 1] / max(1, (img_h - 1)), 0.0, 1.0)
        return out


    if SIMPLIFY_TRAIN_LABELS:
        split_name = "train"
        split_img_dir = SPLIT_ROOT / "images" / split_name
        split_lbl_dir = SPLIT_ROOT / "labels" / split_name

        if SIMPLIFY_BACKUP_ORIGINAL:
            backup_dir = SPLIT_ROOT / "labels" / f"{split_name}_orig_before_simplify"
            if backup_dir.exists():
                shutil.rmtree(backup_dir)
            shutil.copytree(split_lbl_dir, backup_dir)
            print(f"Backup labels: {backup_dir.resolve()}")

        img_map = {
            p.stem: p
            for p in split_img_dir.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        }

        files_processed = 0
        missing_images = 0
        polygons_total = 0
        polygons_simplified = 0

        for lbl_path in sorted(split_lbl_dir.glob("*.txt")):
            anns = read_yolo_seg(lbl_path)
            if not anns:
                continue

            img_path = img_map.get(lbl_path.stem)
            if img_path is None:
                missing_images += 1
                continue

            img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
            if img is None:
                missing_images += 1
                continue

            h, w = img.shape[:2]
            out_anns = []

            for cls, pts in anns:
                polygons_total += 1
                simp = simplify_polygon_safe(
                    pts_norm=pts,
                    img_w=w,
                    img_h=h,
                    eps_frac=SIMPLIFY_EPS_FRAC,
                    min_vertices=SIMPLIFY_MIN_VERTICES,
                    max_area_delta=SIMPLIFY_MAX_AREA_DELTA,
                )

                changed = (simp.shape[0] != pts.shape[0]) or (not np.allclose(simp, pts, atol=1e-6))
                if changed:
                    polygons_simplified += 1

                out_anns.append((cls, simp))

            write_yolo_seg(lbl_path, out_anns)
            files_processed += 1

        ratio = (100.0 * polygons_simplified / max(1, polygons_total))
        print(f"Simplify split: {split_name}")
        print(f"Files processed: {files_processed}")
        print(f"Missing images: {missing_images}")
        print(f"Polygons simplified: {polygons_simplified}/{polygons_total} ({ratio:.2f}%)")
    else:
        print("Train polygon simplification is disabled.")


## 4. YOLO Colony Segmentation Training with MLflow

Source notebook: `C:/ColonyNet/yolo_colonies_mlflow.ipynb`.

Code cells in this section are guarded by `RUN_COLONY_SEGMENTATION_TRAINING`.


# YOLO Segmentation + MLflow (Multi-model)

This notebook is focused only on MLflow-managed training for:
- `yolo26n-seg.pt`
- `yolo26s-seg.pt`
- `yolo26m-seg.pt`
- `yolo26l-seg.pt`
- `yolo26x-seg.pt`

It expects a prepared dataset YAML (`data.yaml`) from your main pipeline.


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow.ipynb, cell 2
# This block runs only when RUN_COLONY_SEGMENTATION_TRAINING is True.
if RUN_COLONY_SEGMENTATION_TRAINING:
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    import mlflow
    from ultralytics import YOLO

    import torch
    torch.cuda.empty_cache()

    # Set your data YAML path explicitly if needed
    DATA_YAML = Path('cropped_736_aug_leaky/dataset/data.yaml')

    # Fallback locations
    if not DATA_YAML.exists():
        candidates = [
            Path('C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml'),
        ]
        for c in candidates:
            if c.exists():
                DATA_YAML = c
                break

    if not DATA_YAML.exists():
        # Last-resort search (first match)
        found = sorted(Path('.').rglob('cropped_736_aug_leaky/dataset/data.yaml'))
        if found:
            DATA_YAML = found[0]

    if not DATA_YAML.exists():
        raise FileNotFoundError('data.yaml not found. Prepare dataset first and set DATA_YAML manually.')

    print('DATA_YAML:', DATA_YAML.resolve())


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow.ipynb, cell 3
# This block runs only when RUN_COLONY_SEGMENTATION_TRAINING is True.
if RUN_COLONY_SEGMENTATION_TRAINING:
    # MLflow: train multiple YOLO-seg models (n/s/m/l/x) and log metrics/artifacts
    from pathlib import Path
    import json
    import os
    import sys
    import threading

    import cv2
    import mlflow
    from mlflow.exceptions import MlflowException
    from mlflow.tracking import MlflowClient
    from mlflow.entities import ViewType
    import numpy as np
    import pandas as pd
    import torch
    import yaml
    from ultralytics import YOLO, settings as yolo_settings

    # Ensure UTF-8 output to avoid MLflow unicode print errors on Windows cp1251 consoles
    os.environ.setdefault("PYTHONIOENCODING", "utf-8")
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

    # If you have MLflow server, set e.g. "http://127.0.0.1:5000".
    # If None, local file-backed MLflow store is used.
    MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"

    MLFLOW_EXPERIMENT = "colony_yolo_seg_736_4"
    LOCAL_MLFLOW_DIR = Path("mlruns_yolo_seg")

    MODELS_TO_TRAIN = [
        # "yolo26n-seg.pt",
        # "yolo26s-seg.pt",
        "yolo26m-seg.pt",
        "yolo26l-seg.pt",
        "yolo26x-seg.pt",
    ]

    MLFLOW_PROJECT = "runs/segment/runs/colony_seg_mlflow_736"
    RUN_SUFFIX = "cropped736_offline_aug"
    RESULTS_CSV_NAME = "results_736_4.csv"

    # Training setup
    TRAIN_EPOCHS = 50
    TRAIN_BATCH = 8
    TRAIN_IMGSZ = 736
    TRAIN_PATIENCE = 80
    TRAIN_DEVICE = None  # e.g. 0 for first GPU, or None for auto
    STOP_ON_ERROR = False

    # Custom metric inference setup (for Dice + counting quality)
    PRED_CONF = 0.25
    PRED_IOU = 0.7
    PRED_MAX_DET = 300
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    LIVE_LOG_POLL_SEC = 2.0


    def to_float(v):
        try:
            return float(v)
        except Exception:
            return float("nan")


    def summarize_section(section, prefix):
        if section is None:
            return {
                f"precision_{prefix}": float("nan"),
                f"recall_{prefix}": float("nan"),
                f"mAP50_{prefix}": float("nan"),
                f"mAP50_95_{prefix}": float("nan"),
            }
        return {
            f"precision_{prefix}": to_float(getattr(section, "mp", float("nan"))),
            f"recall_{prefix}": to_float(getattr(section, "mr", float("nan"))),
            f"mAP50_{prefix}": to_float(getattr(section, "map50", float("nan"))),
            f"mAP50_95_{prefix}": to_float(getattr(section, "map", float("nan"))),
        }


    def safe_metric_name(name: str) -> str:
        s = str(name).strip().replace(" ", "_")
        for bad in ["(", ")", "/", "\\", ":", ",", "|", "-", "."]:
            s = s.replace(bad, "_")
        while "__" in s:
            s = s.replace("__", "_")
        return s.strip("_")


    def maybe_log_artifact(path: Path, artifact_path: str):
        if path.exists():
            mlflow.log_artifact(str(path), artifact_path=artifact_path)


    def resolve_test_split_dirs(data_yaml_path: Path):
        data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8")) or {}
        test_raw = data.get("test")
        if not test_raw:
            raise KeyError("`test` path is missing in data.yaml")

        test_img_dir = Path(str(test_raw).strip().strip('"').strip("'"))
        if not test_img_dir.is_absolute():
            test_img_dir = (data_yaml_path.parent / test_img_dir).resolve()

        split_name = test_img_dir.name
        label_candidates = [data_yaml_path.parent / "labels" / split_name]

        if test_img_dir.parent.name == "images":
            label_candidates.append(test_img_dir.parent.parent / "labels" / split_name)

        replaced = Path(
            str(test_img_dir)
            .replace("\\images\\", "\\labels\\")
            .replace("/images/", "/labels/")
        )
        label_candidates.append(replaced)

        for cand in label_candidates:
            if cand.exists():
                return test_img_dir, cand

        return test_img_dir, label_candidates[0]


    def read_yolo_seg_polygons(label_path: Path):
        if not label_path.exists():
            return []

        txt = label_path.read_text(encoding="utf-8", errors="ignore")
        if not txt.strip():
            return []
        # Fix malformed files containing literal "\n" between coordinates.
        txt = txt.replace("\r", "").replace("\n", chr(10))

        polygons = []
        for raw_line in txt.splitlines():
            line = raw_line.strip()
            if not line:
                continue

            parts = line.split()
            if len(parts) < 7:
                continue

            coords = []
            for token in parts[1:]:
                try:
                    coords.append(float(token))
                except Exception:
                    for sub in token.replace(",", " ").replace(";", " ").split():
                        try:
                            coords.append(float(sub))
                        except Exception:
                            pass

            if len(coords) < 6:
                continue
            if len(coords) % 2 == 1:
                coords = coords[:-1]

            pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
            pts = np.clip(pts, 0.0, 1.0)
            if pts.shape[0] >= 3:
                polygons.append(pts)

        return polygons


    def polygons_norm_to_mask(polygons_norm, h: int, w: int):
        mask = np.zeros((h, w), dtype=np.uint8)
        if h <= 0 or w <= 0:
            return mask

        for pts in polygons_norm:
            arr = np.asarray(pts, dtype=np.float32)
            if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
                continue

            arr_px = np.empty_like(arr)
            arr_px[:, 0] = np.clip(arr[:, 0] * (w - 1), 0, w - 1)
            arr_px[:, 1] = np.clip(arr[:, 1] * (h - 1), 0, h - 1)
            cv2.fillPoly(mask, [np.round(arr_px).astype(np.int32)], 1)

        return mask


    def result_to_pred_mask(result, h: int, w: int):
        mask = np.zeros((h, w), dtype=np.uint8)
        pred_count = 0

        masks_obj = getattr(result, "masks", None)
        if masks_obj is None:
            return mask, pred_count

        polys = getattr(masks_obj, "xy", None)
        if polys is not None and len(polys) > 0:
            pred_count = int(len(polys))
            for poly in polys:
                arr = np.asarray(poly, dtype=np.float32)
                if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
                    continue
                arr[:, 0] = np.clip(arr[:, 0], 0, w - 1)
                arr[:, 1] = np.clip(arr[:, 1], 0, h - 1)
                cv2.fillPoly(mask, [np.round(arr).astype(np.int32)], 1)
            return mask, pred_count

        data = getattr(masks_obj, "data", None)
        if data is None:
            return mask, pred_count

        arr = data.detach().cpu().numpy()
        pred_count = int(arr.shape[0])
        if arr.size == 0:
            return mask, pred_count

        union = (arr > 0.5).any(axis=0).astype(np.uint8)
        if union.shape != (h, w):
            union = cv2.resize(union, (w, h), interpolation=cv2.INTER_NEAREST)
        return union.astype(np.uint8), pred_count


    def dice_score(pred_mask, gt_mask, eps: float = 1e-7):
        a = pred_mask.astype(bool)
        b = gt_mask.astype(bool)

        sa = float(a.sum(dtype=np.float64))
        sb = float(b.sum(dtype=np.float64))
        if sa == 0.0 and sb == 0.0:
            return 1.0

        inter = float(np.logical_and(a, b).sum(dtype=np.float64))
        return float((2.0 * inter + eps) / (sa + sb + eps))


    def find_experiment_any_state(client: MlflowClient, experiment_name: str):
        for exp in client.search_experiments(view_type=ViewType.ALL):
            if exp.name == experiment_name:
                return exp
        return None


    def set_or_restore_experiment(experiment_name: str):
        client = MlflowClient()
        exp = find_experiment_any_state(client, experiment_name)

        if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
            print(f"Experiment '{experiment_name}' is deleted. Restoring...")
            client.restore_experiment(exp.experiment_id)

        try:
            return mlflow.set_experiment(experiment_name)
        except MlflowException as e:
            msg = str(e).lower()
            if "deleted experiment" in msg:
                exp = find_experiment_any_state(client, experiment_name)
                if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
                    client.restore_experiment(exp.experiment_id)
                    return mlflow.set_experiment(experiment_name)
            raise


    def close_active_mlflow_runs(max_iters: int = 10):
        """Best-effort cleanup of dangling active MLflow runs."""
        closed = 0
        for _ in range(max_iters):
            active = mlflow.active_run()
            if active is None:
                break
            try:
                mlflow.end_run()
                closed += 1
            except Exception as e:
                print(f"[WARN] Failed to end active MLflow run: {e}")
                break
        return closed


    def resolve_checkpoint_for_training(ckpt_name: str) -> str:
        ckpt_path = Path(ckpt_name)
        if not ckpt_path.exists():
            # Let Ultralytics download by model name.
            return ckpt_name

        try:
            # Validate checkpoint with standard torch load.
            _ = torch.load(str(ckpt_path), map_location="cpu")
            return str(ckpt_path)
        except Exception:
            broken = ckpt_path.with_name(f"{ckpt_path.name}.broken")
            suffix = 1
            while broken.exists():
                broken = ckpt_path.with_name(f"{ckpt_path.name}.broken{suffix}")
                suffix += 1
            ckpt_path.rename(broken)
            print(f"[WARN] Corrupt checkpoint moved to: {broken}")
            print(f"[WARN] Will try to auto-download fresh weights for: {ckpt_name}")
            return ckpt_name


    def compute_test_custom_metrics(best_model, data_yaml: Path, imgsz: int, device):
        test_img_dir, test_lbl_dir = resolve_test_split_dirs(data_yaml)
        if not test_img_dir.exists():
            raise FileNotFoundError(f"Test image dir not found: {test_img_dir}")

        test_images = [
            p for p in sorted(test_img_dir.iterdir())
            if p.is_file() and p.suffix.lower() in IMG_EXTS
        ]
        if not test_images:
            empty_metrics = {
                "dice_M_mean": float("nan"),
                "dice_M_median": float("nan"),
                "mae_count": float("nan"),
                "rmse_count": float("nan"),
                "mape_count_nonzero": float("nan"),
                "test_images_eval": 0,
                "test_images_missing_labels": 0,
            }
            return empty_metrics, pd.DataFrame()

        pred_iter = best_model.predict(
            source=str(test_img_dir),
            imgsz=imgsz,
            conf=PRED_CONF,
            iou=PRED_IOU,
            max_det=PRED_MAX_DET,
            device=device,
            stream=True,
            verbose=False,
            save=False,
        )

        rows = []
        for res in pred_iter:
            image_path = Path(res.path)
            h, w = map(int, res.orig_shape)

            label_path = test_lbl_dir / f"{image_path.stem}.txt"
            gt_polys = read_yolo_seg_polygons(label_path)
            gt_mask = polygons_norm_to_mask(gt_polys, h, w)

            pred_mask, pred_count = result_to_pred_mask(res, h, w)
            gt_count = int(len(gt_polys))
            count_error = int(pred_count - gt_count)
            abs_error = abs(count_error)
            ape = (abs_error / gt_count) if gt_count > 0 else float("nan")

            rows.append(
                {
                    "image": image_path.name,
                    "label_exists": int(label_path.exists()),
                    "gt_count": gt_count,
                    "pred_count": int(pred_count),
                    "count_error": count_error,
                    "count_abs_error": abs_error,
                    "count_ape": float(ape),
                    "dice": dice_score(pred_mask, gt_mask),
                }
            )

        df = pd.DataFrame(rows)
        if df.empty:
            metrics = {
                "dice_M_mean": float("nan"),
                "dice_M_median": float("nan"),
                "mae_count": float("nan"),
                "rmse_count": float("nan"),
                "mape_count_nonzero": float("nan"),
                "test_images_eval": 0,
                "test_images_missing_labels": 0,
            }
            return metrics, df

        sq = np.square(df["count_error"].to_numpy(dtype=np.float64))
        ape_valid = df["count_ape"].dropna()

        metrics = {
            "dice_M_mean": float(df["dice"].mean()),
            "dice_M_median": float(df["dice"].median()),
            "mae_count": float(df["count_abs_error"].mean()),
            "rmse_count": float(np.sqrt(sq.mean())),
            "mape_count_nonzero": float(ape_valid.mean() * 100.0) if len(ape_valid) else float("nan"),
            "test_images_eval": int(len(df)),
            "test_images_missing_labels": int((df["label_exists"] == 0).sum()),
        }
        return metrics, df


    def find_train_results_csv(train_dir: Path, project_dir: str, run_name: str):
        candidates = []
        names = [RESULTS_CSV_NAME, "results.csv"]

        if train_dir is not None:
            for nm in names:
                candidates.append(train_dir / nm)

        proj = Path(project_dir)
        for nm in names:
            candidates.append(proj / run_name / nm)
            candidates.append(Path("runs") / "segment" / proj / run_name / nm)

        for base in [proj, Path("runs") / "segment" / proj, Path("runs") / "segment"]:
            if base.exists():
                try:
                    for nm in names:
                        candidates.extend(
                            sorted(
                                base.glob(f"**/{run_name}*/{nm}"),
                                key=lambda p: p.stat().st_mtime,
                                reverse=True,
                            )
                        )
                except Exception:
                    pass

        for c in candidates:
            if c is not None and c.exists():
                return c
        return None

    def pick_active_results_csv(results_csv_candidates):
        existing = [p for p in results_csv_candidates if p is not None and p.exists()]
        if not existing:
            return None
        try:
            return max(existing, key=lambda p: p.stat().st_mtime)
        except Exception:
            return existing[0]


    def log_partial_train_metrics_to_run(run_id: str, results_csv: Path):
        if not run_id or results_csv is None or not results_csv.exists():
            return False

        try:
            df = pd.read_csv(results_csv)
        except Exception as e:
            print(f"[WARN] Cannot read partial results.csv: {e}")
            return False

        if df.empty:
            return False

        client = MlflowClient()
        logged_any = False
        for step_idx, row in df.iterrows():
            step_raw = row.get("epoch", step_idx)
            step_float = to_float(step_raw)
            step = int(step_float) if np.isfinite(step_float) else int(step_idx)

            for k, v in row.items():
                vv = to_float(v)
                if np.isfinite(vv):
                    client.log_metric(run_id, f"train_{safe_metric_name(k)}", float(vv), step=step)
                    logged_any = True

        return logged_any


    def stream_train_metrics_live(run_id: str, results_csv_candidates, stop_event, poll_seconds: float = LIVE_LOG_POLL_SEC):
        """Stream new rows from Ultralytics results.csv to MLflow as train_* metrics."""
        client = MlflowClient()
        next_row = 0
        active_csv = None

        while not stop_event.is_set():
            current_csv = pick_active_results_csv(results_csv_candidates)

            if current_csv is not None and current_csv != active_csv:
                active_csv = current_csv
                next_row = 0
                print(f"[LIVE] train metrics source: {active_csv}")

            if active_csv is not None and active_csv.exists():
                try:
                    df = pd.read_csv(active_csv)
                except Exception:
                    if stop_event.wait(poll_seconds):
                        break
                    continue

                if len(df) > next_row:
                    for row_idx in range(next_row, len(df)):
                        row = df.iloc[row_idx]
                        step_raw = row.get("epoch", row_idx)
                        step_float = to_float(step_raw)
                        step = int(step_float) if np.isfinite(step_float) else int(row_idx)

                        for k, v in row.items():
                            vv = to_float(v)
                            if np.isfinite(vv):
                                client.log_metric(run_id, f"train_{safe_metric_name(k)}", float(vv), step=step)
                    next_row = len(df)

            if stop_event.wait(poll_seconds):
                break

        # Final flush in case the last rows were written right before stop.
        if active_csv is not None and active_csv.exists():
            try:
                df = pd.read_csv(active_csv)
                if len(df) > next_row:
                    for row_idx in range(next_row, len(df)):
                        row = df.iloc[row_idx]
                        step_raw = row.get("epoch", row_idx)
                        step_float = to_float(step_raw)
                        step = int(step_float) if np.isfinite(step_float) else int(row_idx)

                        for k, v in row.items():
                            vv = to_float(v)
                            if np.isfinite(vv):
                                client.log_metric(run_id, f"train_{safe_metric_name(k)}", float(vv), step=step)
            except Exception:
                pass


    # Make this cell self-contained: resolve DATA_YAML if previous setup cell was not executed.
    if "DATA_YAML" not in globals() or DATA_YAML is None:
        DATA_YAML = Path("cropped_736_aug_leaky/dataset/data.yaml")
    else:
        DATA_YAML = Path(DATA_YAML)

    if not DATA_YAML.exists():
        fallback_candidates = [
            Path("C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml"),
            Path("cropped_736/dataset/data.yaml"),
            Path("C:/ColonyNet/cropped_736/dataset/data.yaml"),
        ]
        for candidate in fallback_candidates:
            if candidate.exists():
                DATA_YAML = candidate
                break

    if not DATA_YAML.exists():
        found = sorted(Path(".").rglob("cropped_736_aug_leaky/dataset/data.yaml"))
        if not found:
            found = sorted(Path(".").rglob("cropped_736/dataset/data.yaml"))
        if found:
            DATA_YAML = found[0]

    if not DATA_YAML.exists():
        raise FileNotFoundError("data.yaml not found. Run dataset preparation first or set DATA_YAML manually.")

    print("Using DATA_YAML:", DATA_YAML.resolve())

    # Enable Ultralytics MLflow autologging (alongside manual custom logging below).
    AUTOLOG_ULTRALYTICS = True
    yolo_settings.update({"mlflow": AUTOLOG_ULTRALYTICS})
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
    os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT
    print("Ultralytics setting mlflow:", yolo_settings.get("mlflow"))

    # Configure tracking
    if MLFLOW_TRACKING_URI:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    else:
        LOCAL_MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
        mlflow.set_tracking_uri(LOCAL_MLFLOW_DIR.resolve().as_uri())

    active_experiment = set_or_restore_experiment(MLFLOW_EXPERIMENT)
    print("MLflow tracking URI:", mlflow.get_tracking_uri())
    print("Experiment:", getattr(active_experiment, "name", MLFLOW_EXPERIMENT))
    print("Experiment ID:", getattr(active_experiment, "experiment_id", "unknown"))

    summary_rows = []

    for ckpt in MODELS_TO_TRAIN:
        model_tag = Path(ckpt).stem
        run_name = f"{model_tag}_{RUN_SUFFIX}"
        print(f"\n=== Training {ckpt} ===")

        run_id = None
        train_dir = None

        closed_before = close_active_mlflow_runs()
        if closed_before:
            print(f"[INFO] Closed {closed_before} stale active MLflow run(s) before {ckpt}")

        try:
            with mlflow.start_run(run_name=run_name) as active_run:
                run_id = active_run.info.run_id
                train_ckpt = resolve_checkpoint_for_training(ckpt)

                mlflow.set_tags(
                    {
                        "framework": "ultralytics",
                        "task": "segment",
                        "dataset": str(DATA_YAML),
                        "model_ckpt": ckpt,
                        "offline_aug": "true",
                        "online_aug": "false",
                    }
                )
                mlflow.log_metric("run_started", 1.0, step=0)
                mlflow.log_params(
                    {
                        "model": ckpt,
                        "model_effective": train_ckpt,
                        "data_yaml": str(DATA_YAML),
                        "imgsz": TRAIN_IMGSZ,
                        "epochs": TRAIN_EPOCHS,
                        "batch": TRAIN_BATCH,
                        "patience": TRAIN_PATIENCE,
                        "project": MLFLOW_PROJECT,
                        "run_name": run_name,
                        "tracking_uri": mlflow.get_tracking_uri(),
                        "predict_conf": PRED_CONF,
                        "predict_iou": PRED_IOU,
                        "predict_max_det": PRED_MAX_DET,
                    }
                )

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    print("CUDA cache cleared")

                expected_train_dir = Path(MLFLOW_PROJECT) / run_name
                live_results_candidates = [
                    expected_train_dir / "results.csv",
                    Path("runs") / "segment" / expected_train_dir / "results.csv",
                    Path("runs") / "segment" / run_name / "results.csv",
                ]
                live_stop_event = threading.Event()
                live_thread = threading.Thread(
                    target=stream_train_metrics_live,
                    args=(run_id, live_results_candidates, live_stop_event),
                    kwargs={"poll_seconds": LIVE_LOG_POLL_SEC},
                    daemon=True,
                )
                live_thread.start()

                model = YOLO(train_ckpt)
                try:
                    train_results = model.train(
                        data=str(DATA_YAML),
                        task="segment",
                        imgsz=TRAIN_IMGSZ,
                        epochs=TRAIN_EPOCHS,
                        batch=TRAIN_BATCH,
                        patience=TRAIN_PATIENCE,
                        device=TRAIN_DEVICE,
                        project=MLFLOW_PROJECT,
                        name=run_name,
                        exist_ok=True,
                        plots=True,
                        # No online augmentation (offline-augmented dataset only)
                        degrees=0.0,
                        translate=0.0,
                        scale=0.0,
                        shear=0.0,
                        perspective=0.0,
                        fliplr=0.0,
                        flipud=0.0,
                        hsv_h=0.0,
                        hsv_s=0.0,
                        hsv_v=0.0,
                        mosaic=0.0,
                        mixup=0.0,
                        copy_paste=0.0,
                        erasing=0.0,
                    )
                finally:
                    live_stop_event.set()
                    live_thread.join(timeout=15)

                train_dir = Path(train_results.save_dir)
                raw_results_csv = train_dir / "results.csv"
                results_csv = train_dir / RESULTS_CSV_NAME
                if raw_results_csv.exists():
                    try:
                        results_csv.write_bytes(raw_results_csv.read_bytes())
                    except Exception:
                        results_csv = raw_results_csv
                elif not results_csv.exists():
                    results_csv = raw_results_csv

                best_w = train_dir / "weights" / "best.pt"
                last_w = train_dir / "weights" / "last.pt"

                # Log train artifacts
                maybe_log_artifact(train_dir / "args.yaml", "train")
                maybe_log_artifact(results_csv, "train")
                maybe_log_artifact(train_dir / "results.png", "train")
                maybe_log_artifact(train_dir / "confusion_matrix.png", "train")
                maybe_log_artifact(train_dir / "confusion_matrix_normalized.png", "train")
                for nm in [
                    "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png", "BoxF1_curve.png",
                    "MaskPR_curve.png", "MaskP_curve.png", "MaskR_curve.png", "MaskF1_curve.png",
                ]:
                    maybe_log_artifact(train_dir / nm, "train")
                maybe_log_artifact(best_w, "weights")
                maybe_log_artifact(last_w, "weights")

                # Log last epoch train metrics from results_736_4.csv (fallback results.csv)
                r_csv = results_csv
                if r_csv.exists():
                    train_df = pd.read_csv(r_csv)
                    if len(train_df) > 0:
                        last_row = train_df.iloc[-1].to_dict()
                        train_metrics = {}
                        for k, v in last_row.items():
                            vv = to_float(v)
                            if np.isfinite(vv):
                                train_metrics[f"train_{safe_metric_name(k)}"] = vv
                        if train_metrics:
                            mlflow.log_metrics(train_metrics)

                if not best_w.exists():
                    raise FileNotFoundError(f"best.pt not found for {ckpt}: {best_w}")

                # Evaluate on test split
                best_model = YOLO(str(best_w))
                test_results = best_model.val(
                    data=str(DATA_YAML),
                    split="test",
                    imgsz=TRAIN_IMGSZ,
                    project=MLFLOW_PROJECT,
                    name=f"{run_name}_test",
                    exist_ok=True,
                    plots=True,
                    save_json=True,
                )

                test_dir = Path(test_results.save_dir)

                metrics_summary = {}
                metrics_summary.update(summarize_section(getattr(test_results, "box", None), "B"))
                metrics_summary.update(summarize_section(getattr(test_results, "seg", None), "M"))
                metrics_summary["fitness"] = to_float(getattr(test_results, "fitness", float("nan")))

                # Custom post-hoc metrics on test split
                custom_metrics, per_image_df = compute_test_custom_metrics(
                    best_model=best_model,
                    data_yaml=Path(DATA_YAML),
                    imgsz=TRAIN_IMGSZ,
                    device=TRAIN_DEVICE,
                )
                metrics_summary.update(custom_metrics)

                per_image_csv = test_dir / "test_per_image_metrics.csv"
                per_image_df.to_csv(per_image_csv, index=False)

                # Persist + log test metrics json
                test_json = test_dir / "test_metrics_summary.json"
                with test_json.open("w", encoding="utf-8") as f:
                    json.dump(metrics_summary, f, indent=2)

                mlflow_metrics = {k: v for k, v in metrics_summary.items() if np.isfinite(v)}
                if mlflow_metrics:
                    mlflow.log_metrics(mlflow_metrics)

                # Log test artifacts
                maybe_log_artifact(test_json, "test")
                maybe_log_artifact(per_image_csv, "test")
                maybe_log_artifact(test_dir / "predictions.json", "test")
                maybe_log_artifact(test_dir / "confusion_matrix.png", "test")
                maybe_log_artifact(test_dir / "confusion_matrix_normalized.png", "test")
                for nm in [
                    "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png", "BoxF1_curve.png",
                    "MaskPR_curve.png", "MaskP_curve.png", "MaskR_curve.png", "MaskF1_curve.png",
                    "PR_curve.png", "P_curve.png", "R_curve.png", "F1_curve.png",
                ]:
                    maybe_log_artifact(test_dir / nm, "test")

                mlflow.log_params(
                    {
                        "ultralytics_train_dir": str(train_dir.resolve()),
                        "ultralytics_test_dir": str(test_dir.resolve()),
                        "best_weights": str(best_w.resolve()),
                    }
                )

                row = {
                    "model": ckpt,
                    "train_dir": str(train_dir),
                    "test_dir": str(test_dir),
                    **metrics_summary,
                }
                summary_rows.append(row)

                print(f"Done: {ckpt}")
                print("  train_dir:", train_dir)
                print("  test_dir :", test_dir)

                # Free references before next model
                del model
                del best_model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        except Exception as exc:
            print(f"[ERROR] {ckpt}: {exc}")

            try:
                partial_csv = find_train_results_csv(train_dir, MLFLOW_PROJECT, run_name)
                if run_id:
                    client = MlflowClient()
                    client.set_tag(run_id, "error_message", str(exc)[:1000])
                    if partial_csv is not None:
                        logged = log_partial_train_metrics_to_run(run_id, partial_csv)
                        if logged:
                            print(f"[INFO] Logged partial train metrics from: {partial_csv}")
                        else:
                            print(f"[INFO] Partial results found but no numeric metrics: {partial_csv}")
            except Exception as log_exc:
                print(f"[WARN] Failed to log partial metrics: {log_exc}")

            summary_rows.append({"model": ckpt, "error": str(exc)})
            if STOP_ON_ERROR:
                raise
        finally:
            closed_after = close_active_mlflow_runs()
            if closed_after:
                print(f"[INFO] Closed {closed_after} dangling MLflow run(s) after {ckpt}")

    summary_df = pd.DataFrame(summary_rows)
    try:
        display(summary_df)
    except Exception:
        print(summary_df)

    summary_csv = Path("mlflow_multi_model_summary.csv")
    summary_df.to_csv(summary_csv, index=False)
    print("Saved summary:", summary_csv.resolve())


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow.ipynb, cell 4
# This block runs only when RUN_COLONY_SEGMENTATION_TRAINING is True.
if RUN_COLONY_SEGMENTATION_TRAINING:
    # X-only: train yolo26x-seg.pt in same MLflow experiment/project
    from pathlib import Path
    from datetime import datetime
    import os, sys
    import numpy as np
    import pandas as pd
    import torch
    import mlflow
    from mlflow.tracking import MlflowClient
    from mlflow.entities import ViewType
    from mlflow.exceptions import MlflowException
    from ultralytics import YOLO, settings as yolo_settings

    os.environ.setdefault("PYTHONIOENCODING", "utf-8")
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

    MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
    MLFLOW_EXPERIMENT = "colony_yolo_seg"
    MLFLOW_PROJECT = "runs/colony_seg_mlflow"

    CKPT_X = "yolo26x-seg.pt"  # если файла нет, Ultralytics попытается скачать
    RUN_NAME = f"yolo26x-seg_cropped736_offline_aug_xonly_{datetime.now():%Y%m%d_%H%M%S}"

    TRAIN_EPOCHS = 50
    TRAIN_BATCH = 8
    TRAIN_IMGSZ = 736
    TRAIN_PATIENCE = 80
    TRAIN_DEVICE = None
    RESULTS_CSV_NAME = "results_736_4.csv"

    def to_float(v):
        try:
            return float(v)
        except Exception:
            return float("nan")

    def safe_metric_name(name: str) -> str:
        s = str(name).strip().replace(" ", "_")
        for bad in ["(", ")", "/", "\\", ":", ",", "|", "-", "."]:
            s = s.replace(bad, "_")
        while "__" in s:
            s = s.replace("__", "_")
        return s.strip("_")

    def summarize_section(section, prefix):
        if section is None:
            return {
                f"precision_{prefix}": float("nan"),
                f"recall_{prefix}": float("nan"),
                f"mAP50_{prefix}": float("nan"),
                f"mAP50_95_{prefix}": float("nan"),
            }
        return {
            f"precision_{prefix}": to_float(getattr(section, "mp", float("nan"))),
            f"recall_{prefix}": to_float(getattr(section, "mr", float("nan"))),
            f"mAP50_{prefix}": to_float(getattr(section, "map50", float("nan"))),
            f"mAP50_95_{prefix}": to_float(getattr(section, "map", float("nan"))),
        }

    def find_experiment_any_state(client: MlflowClient, experiment_name: str):
        for exp in client.search_experiments(view_type=ViewType.ALL):
            if exp.name == experiment_name:
                return exp
        return None

    def set_or_restore_experiment(experiment_name: str):
        client = MlflowClient()
        exp = find_experiment_any_state(client, experiment_name)
        if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
            client.restore_experiment(exp.experiment_id)
        try:
            return mlflow.set_experiment(experiment_name)
        except MlflowException as e:
            if "deleted experiment" in str(e).lower():
                exp = find_experiment_any_state(client, experiment_name)
                if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
                    client.restore_experiment(exp.experiment_id)
                    return mlflow.set_experiment(experiment_name)
            raise


    def close_active_mlflow_runs(max_iters: int = 10):
        """Best-effort cleanup of dangling active MLflow runs."""
        closed = 0
        for _ in range(max_iters):
            active = mlflow.active_run()
            if active is None:
                break
            try:
                mlflow.end_run()
                closed += 1
            except Exception as e:
                print(f"[WARN] Failed to end active MLflow run: {e}")
                break
        return closed


    # DATA_YAML fallback
    if "DATA_YAML" not in globals() or DATA_YAML is None:
        DATA_YAML = Path("cropped_736_aug_leaky/dataset/data.yaml")
    else:
        DATA_YAML = Path(DATA_YAML)

    if not DATA_YAML.exists():
        raise FileNotFoundError(f"DATA_YAML not found: {DATA_YAML}")

    # enable built-in Ultralytics mlflow autologging
    AUTOLOG_ULTRALYTICS = True
    yolo_settings.update({"mlflow": AUTOLOG_ULTRALYTICS})
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
    os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    set_or_restore_experiment(MLFLOW_EXPERIMENT)

    print("DATA_YAML:", DATA_YAML.resolve())
    print("Run name :", RUN_NAME)

    closed_before = close_active_mlflow_runs()
    if closed_before:
        print(f"[INFO] Closed {closed_before} stale active MLflow run(s) before {RUN_NAME}")

    with mlflow.start_run(run_name=RUN_NAME) as run:
        run_id = run.info.run_id
        mlflow.log_metric("run_started", 1.0, step=0)
        mlflow.log_params({
            "model": CKPT_X,
            "data_yaml": str(DATA_YAML),
            "imgsz": TRAIN_IMGSZ,
            "epochs": TRAIN_EPOCHS,
            "batch": TRAIN_BATCH,
            "patience": TRAIN_PATIENCE,
            "project": MLFLOW_PROJECT,
            "run_name": RUN_NAME,
            "tracking_uri": mlflow.get_tracking_uri(),
            "online_aug": "false",
        })
        mlflow.set_tags({
            "framework": "ultralytics",
            "task": "segment",
            "dataset": str(DATA_YAML),
            "model_ckpt": CKPT_X,
            "offline_aug": "true",
            "online_aug": "false",
        })

        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            model = YOLO(CKPT_X)
            train_results = model.train(
                data=str(DATA_YAML),
                task="segment",
                imgsz=TRAIN_IMGSZ,
                epochs=TRAIN_EPOCHS,
                batch=TRAIN_BATCH,
                patience=TRAIN_PATIENCE,
                device=TRAIN_DEVICE,
                project=MLFLOW_PROJECT,
                name=RUN_NAME,
                exist_ok=False,
                plots=True,
                degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
                fliplr=0.0, flipud=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
                mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0,
            )

            train_dir = Path(train_results.save_dir)
            raw_results_csv = train_dir / "results.csv"
            results_csv = train_dir / RESULTS_CSV_NAME
            if raw_results_csv.exists():
                try:
                    results_csv.write_bytes(raw_results_csv.read_bytes())
                except Exception:
                    results_csv = raw_results_csv
            elif not results_csv.exists():
                results_csv = raw_results_csv

            best_w = train_dir / "weights" / "best.pt"
            last_w = train_dir / "weights" / "last.pt"

            # online-like logging from csv (all epochs)
            if results_csv.exists():
                df = pd.read_csv(results_csv)
                for i, row in df.iterrows():
                    step_raw = row.get("epoch", i)
                    step = int(to_float(step_raw)) if np.isfinite(to_float(step_raw)) else int(i)
                    for k, v in row.items():
                        vv = to_float(v)
                        if np.isfinite(vv):
                            mlflow.log_metric(f"train_{safe_metric_name(k)}", vv, step=step)
                mlflow.log_artifact(str(results_csv), artifact_path="train")

            # train artifacts
            for p in [
                train_dir / "args.yaml",
                train_dir / "results.png",
                train_dir / "confusion_matrix.png",
                train_dir / "confusion_matrix_normalized.png",
                best_w, last_w
            ]:
                if p.exists():
                    mlflow.log_artifact(str(p), artifact_path="train" if p.suffix != ".pt" else "weights")

            if not best_w.exists():
                raise FileNotFoundError(f"best.pt not found: {best_w}")

            # test validation
            best_model = YOLO(str(best_w))
            test_results = best_model.val(
                data=str(DATA_YAML),
                split="test",
                imgsz=TRAIN_IMGSZ,
                project=MLFLOW_PROJECT,
                name=f"{RUN_NAME}_test",
                exist_ok=True,
                plots=True,
                save_json=True,
            )
            test_dir = Path(test_results.save_dir)

            metrics_summary = {}
            metrics_summary.update(summarize_section(getattr(test_results, "box", None), "B"))
            metrics_summary.update(summarize_section(getattr(test_results, "seg", None), "M"))
            metrics_summary["fitness"] = to_float(getattr(test_results, "fitness", float("nan")))

            mlflow.log_metrics({k: v for k, v in metrics_summary.items() if np.isfinite(v)})

            test_json = test_dir / "test_metrics_summary.json"
            import json
            with test_json.open("w", encoding="utf-8") as f:
                json.dump(metrics_summary, f, indent=2)

            for p in [
                test_json,
                test_dir / "predictions.json",
                test_dir / "confusion_matrix.png",
                test_dir / "confusion_matrix_normalized.png",
            ]:
                if p.exists():
                    mlflow.log_artifact(str(p), artifact_path="test")

            mlflow.log_params({
                "ultralytics_train_dir": str(train_dir.resolve()),
                "ultralytics_test_dir": str(test_dir.resolve()),
                "best_weights": str(best_w.resolve()),
            })

            print("DONE:", RUN_NAME)
            print("train_dir:", train_dir)
            print("test_dir :", test_dir)

        except Exception as e:
            MlflowClient().set_tag(run_id, "error_message", str(e)[:1000])
            print("[ERROR]", e)
            raise

    closed_after = close_active_mlflow_runs()
    if closed_after:
        print(f"[INFO] Closed {closed_after} dangling MLflow run(s) after {RUN_NAME}")


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow.ipynb, cell 7
# This block runs only when RUN_COLONY_SEGMENTATION_TRAINING is True.
if RUN_COLONY_SEGMENTATION_TRAINING:
    # Summary table: all metrics for YOLO models from MLflow
    import re
    from datetime import datetime, timezone
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import mlflow
    from mlflow.tracking import MlflowClient
    from mlflow.entities import ViewType

    MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
    MLFLOW_EXPERIMENT = "colony_yolo_seg"
    MODEL_ORDER = [
        "yolo26n-seg.pt",
        "yolo26s-seg.pt",
        "yolo26m-seg.pt",
        "yolo26l-seg.pt",
        "yolo26x-seg.pt",
    ]
    PREFER_FINISHED = True  # if True, pick latest FINISHED run per model; fallback to latest any status


    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    client = MlflowClient()
    exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
    if exp is None:
        raise FileNotFoundError(f"MLflow experiment not found: {MLFLOW_EXPERIMENT}")

    runs = client.search_runs(
        [exp.experiment_id],
        run_view_type=ViewType.ALL,
        max_results=1000,
        order_by=["attributes.start_time DESC"],
    )


    def extract_model_key(run):
        model_param = str(run.data.params.get("model", "")).lower()
        run_name = str(run.data.tags.get("mlflow.runName", "")).lower()

        m = re.search(r"(yolo26[nsmlx]-seg\.pt)", model_param)
        if m:
            return m.group(1)

        m = re.search(r"(yolo26[nsmlx]-seg)", run_name)
        if m:
            return m.group(1) + ".pt"

        return None


    by_model_any = {}
    by_model_finished = {}
    for run in runs:
        key = extract_model_key(run)
        if key not in MODEL_ORDER:
            continue
        if key not in by_model_any:
            by_model_any[key] = run
        if run.info.status == "FINISHED" and key not in by_model_finished:
            by_model_finished[key] = run

    selected = {}
    for key in MODEL_ORDER:
        if PREFER_FINISHED and key in by_model_finished:
            selected[key] = by_model_finished[key]
        elif key in by_model_any:
            selected[key] = by_model_any[key]

    if not selected:
        raise RuntimeError("No matching YOLO runs found in MLflow.")

    metric_keys = sorted({k for run in selected.values() for k in run.data.metrics.keys()})
    rows = []
    now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)

    for key in MODEL_ORDER:
        run = selected.get(key)
        if run is None:
            rows.append({"model": key, "status": "NOT_FOUND"})
            continue

        start_ms = run.info.start_time or np.nan
        end_ms = run.info.end_time if run.info.end_time is not None else now_ms

        row = {
            "model": key,
            "run_name": run.data.tags.get("mlflow.runName", ""),
            "status": run.info.status,
            "run_id": run.info.run_id,
            "start_time": pd.to_datetime(start_ms, unit="ms", utc=True) if pd.notna(start_ms) else pd.NaT,
            "duration_min": (end_ms - start_ms) / 60000.0 if pd.notna(start_ms) else np.nan,
            "error_message": run.data.tags.get("error_message", ""),
        }

        for mk in metric_keys:
            row[mk] = run.data.metrics.get(mk, np.nan)

        rows.append(row)

    summary_df = pd.DataFrame(rows)

    base_cols = ["model", "run_name", "status", "duration_min", "run_id", "start_time", "error_message"]
    metric_cols = [c for c in summary_df.columns if c not in base_cols]
    summary_df = summary_df[base_cols + metric_cols]

    # Optional rounded view for readability
    display_df = summary_df.copy()
    for c in display_df.columns:
        if pd.api.types.is_numeric_dtype(display_df[c]):
            display_df[c] = display_df[c].round(6)

    display(display_df)

    out_csv = Path("mlflow_models_metrics_summary_latest.csv")
    summary_df.to_csv(out_csv, index=False, encoding="utf-8")
    print("Saved:", out_csv.resolve())


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow.ipynb, cell 9
# This block runs only when RUN_COLONY_SEGMENTATION_TRAINING is True.
if RUN_COLONY_SEGMENTATION_TRAINING:
    # Recalculate custom metrics for yolo26x-seg.pt without retraining (Dice/MAE/RMSE/MAPE)
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    import cv2
    import yaml
    import mlflow
    from mlflow.tracking import MlflowClient
    from mlflow.entities import ViewType
    from ultralytics import YOLO

    MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
    MLFLOW_EXPERIMENT = "colony_yolo_seg"
    X_MODEL_KEY = "yolo26x-seg.pt"
    LOG_RECALC_TO_MLFLOW = True  # set False if you only need local files/print

    PRED_CONF = 0.25
    PRED_IOU = 0.7
    PRED_MAX_DET = 300
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


    def resolve_data_yaml_local():
        if "DATA_YAML" in globals() and DATA_YAML is not None:
            p = Path(DATA_YAML)
            if p.exists():
                return p

        for c in [
            Path("cropped_736_aug_leaky/dataset/data.yaml"),
            Path("C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml"),
            Path("cropped_736/dataset/data.yaml"),
            Path("C:/ColonyNet/cropped_736/dataset/data.yaml"),
        ]:
            if c.exists():
                return c

        found = sorted(Path(".").rglob("cropped_736_aug_leaky/dataset/data.yaml"))
        if not found:
            found = sorted(Path(".").rglob("cropped_736/dataset/data.yaml"))
        if found:
            return found[0]

        raise FileNotFoundError("data.yaml not found")


    def resolve_latest_x_run_and_weights():
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        client = MlflowClient()
        exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
        if exp is None:
            raise FileNotFoundError(f"Experiment not found: {MLFLOW_EXPERIMENT}")

        runs = client.search_runs(
            [exp.experiment_id],
            run_view_type=ViewType.ACTIVE_ONLY,
            max_results=1000,
            order_by=["attributes.start_time DESC"],
        )

        selected = None
        for r in runs:
            model_param = str(r.data.params.get("model", "")).lower()
            run_name = str(r.data.tags.get("mlflow.runName", "")).lower()
            if X_MODEL_KEY in model_param or "yolo26x-seg" in run_name:
                selected = r
                break

        if selected is None:
            raise RuntimeError("No MLflow run found for yolo26x-seg")

        candidates = []
        train_dir_param = selected.data.params.get("ultralytics_train_dir")
        if train_dir_param:
            candidates.append(Path(train_dir_param) / "weights" / "best.pt")

        for base in [
            Path("runs/segment/runs/colony_seg_mlflow"),
            Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
        ]:
            if not base.exists():
                continue
            for d in sorted(base.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
                if not d.is_dir() or d.name.endswith("_test"):
                    continue
                if "yolo26x-seg" not in d.name.lower():
                    continue
                candidates.append(d / "weights" / "best.pt")

        for c in candidates:
            if c.exists():
                return selected, c

        raise FileNotFoundError("best.pt for yolo26x-seg not found")


    def resolve_test_split_dirs(data_yaml_path: Path):
        data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8")) or {}
        test_raw = data.get("test")
        if not test_raw:
            raise KeyError("`test` path is missing in data.yaml")

        test_img_dir = Path(str(test_raw).strip().strip('"').strip("'"))
        if not test_img_dir.is_absolute():
            test_img_dir = (data_yaml_path.parent / test_img_dir).resolve()

        split_name = test_img_dir.name
        label_candidates = [data_yaml_path.parent / "labels" / split_name]
        if test_img_dir.parent.name == "images":
            label_candidates.append(test_img_dir.parent.parent / "labels" / split_name)
        label_candidates.append(Path(str(test_img_dir).replace("\\images\\", "\\labels\\").replace("/images/", "/labels/")))

        for cand in label_candidates:
            if cand.exists():
                return test_img_dir, cand

        return test_img_dir, label_candidates[0]


    def read_yolo_seg_polygons(label_path: Path):
        if not label_path.exists():
            return []

        txt = label_path.read_text(encoding="utf-8", errors="ignore")
        if not txt.strip():
            return []

        txt = txt.replace("\r", "").replace("\\n", "\n")

        polygons = []
        for raw_line in txt.splitlines():
            line = raw_line.strip()
            if not line:
                continue

            parts = line.split()
            if len(parts) < 7:
                continue

            coords = []
            for token in parts[1:]:
                try:
                    coords.append(float(token))
                except Exception:
                    for sub in token.replace(",", " ").replace(";", " ").split():
                        try:
                            coords.append(float(sub))
                        except Exception:
                            pass

            if len(coords) < 6:
                continue
            if len(coords) % 2 == 1:
                coords = coords[:-1]

            pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
            pts = np.clip(pts, 0.0, 1.0)
            if pts.shape[0] >= 3:
                polygons.append(pts)

        return polygons


    def polygons_norm_to_mask(polygons_norm, h: int, w: int):
        mask = np.zeros((h, w), dtype=np.uint8)
        if h <= 0 or w <= 0:
            return mask

        for pts in polygons_norm:
            arr = np.asarray(pts, dtype=np.float32)
            if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
                continue

            arr_px = np.empty_like(arr)
            arr_px[:, 0] = np.clip(arr[:, 0] * (w - 1), 0, w - 1)
            arr_px[:, 1] = np.clip(arr[:, 1] * (h - 1), 0, h - 1)
            cv2.fillPoly(mask, [np.round(arr_px).astype(np.int32)], 1)

        return mask


    def result_to_pred_mask(result, h: int, w: int):
        mask = np.zeros((h, w), dtype=np.uint8)
        pred_count = 0

        masks_obj = getattr(result, "masks", None)
        if masks_obj is None:
            return mask, pred_count

        polys = getattr(masks_obj, "xy", None)
        if polys is not None and len(polys) > 0:
            pred_count = int(len(polys))
            for poly in polys:
                arr = np.asarray(poly, dtype=np.float32)
                if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
                    continue
                arr[:, 0] = np.clip(arr[:, 0], 0, w - 1)
                arr[:, 1] = np.clip(arr[:, 1], 0, h - 1)
                cv2.fillPoly(mask, [np.round(arr).astype(np.int32)], 1)
            return mask, pred_count

        data = getattr(masks_obj, "data", None)
        if data is None:
            return mask, pred_count

        arr = data.detach().cpu().numpy()
        pred_count = int(arr.shape[0])
        if arr.size == 0:
            return mask, pred_count

        union = (arr > 0.5).any(axis=0).astype(np.uint8)
        if union.shape != (h, w):
            union = cv2.resize(union, (w, h), interpolation=cv2.INTER_NEAREST)
        return union.astype(np.uint8), pred_count


    def dice_score(pred_mask, gt_mask, eps: float = 1e-7):
        a = pred_mask.astype(bool)
        b = gt_mask.astype(bool)
        sa = float(a.sum(dtype=np.float64))
        sb = float(b.sum(dtype=np.float64))
        if sa == 0.0 and sb == 0.0:
            return 1.0
        inter = float(np.logical_and(a, b).sum(dtype=np.float64))
        return float((2.0 * inter + eps) / (sa + sb + eps))


    def compute_custom_metrics(best_model, data_yaml: Path, imgsz: int = 736):
        test_img_dir, test_lbl_dir = resolve_test_split_dirs(data_yaml)
        test_images = [p for p in sorted(test_img_dir.iterdir()) if p.is_file() and p.suffix.lower() in IMG_EXTS]
        if not test_images:
            raise RuntimeError(f"No test images in: {test_img_dir}")

        pred_iter = best_model.predict(
            source=str(test_img_dir),
            imgsz=imgsz,
            conf=PRED_CONF,
            iou=PRED_IOU,
            max_det=PRED_MAX_DET,
            stream=True,
            verbose=False,
            save=False,
        )

        rows = []
        for res in pred_iter:
            image_path = Path(res.path)
            h, w = map(int, res.orig_shape)

            label_path = test_lbl_dir / f"{image_path.stem}.txt"
            gt_polys = read_yolo_seg_polygons(label_path)
            gt_mask = polygons_norm_to_mask(gt_polys, h, w)

            pred_mask, pred_count = result_to_pred_mask(res, h, w)
            gt_count = int(len(gt_polys))
            count_error = int(pred_count - gt_count)
            abs_error = abs(count_error)
            ape = (abs_error / gt_count) if gt_count > 0 else np.nan

            rows.append({
                "image": image_path.name,
                "label_exists": int(label_path.exists()),
                "gt_count": gt_count,
                "pred_count": int(pred_count),
                "count_error": count_error,
                "count_abs_error": abs_error,
                "count_ape": float(ape),
                "dice": dice_score(pred_mask, gt_mask),
            })

        df = pd.DataFrame(rows)
        sq = np.square(df["count_error"].to_numpy(dtype=np.float64))
        ape_valid = df["count_ape"].dropna()

        metrics = {
            "dice_M_mean": float(df["dice"].mean()),
            "dice_M_median": float(df["dice"].median()),
            "mae_count": float(df["count_abs_error"].mean()),
            "rmse_count": float(np.sqrt(sq.mean())),
            "mape_count_nonzero": float(ape_valid.mean() * 100.0) if len(ape_valid) else np.nan,
            "test_images_eval": int(len(df)),
            "test_images_missing_labels": int((df["label_exists"] == 0).sum()),
        }
        return metrics, df, test_img_dir


    # ---- run ----
    data_yaml = resolve_data_yaml_local()
    run_x, best_weights_x = resolve_latest_x_run_and_weights()
    print("Using DATA_YAML:", data_yaml.resolve())
    print("Using X run:", run_x.data.tags.get("mlflow.runName"), run_x.info.run_id)
    print("Using weights:", best_weights_x)

    best_model_x = YOLO(str(best_weights_x))
    custom_metrics, per_image_df, used_test_dir = compute_custom_metrics(best_model_x, data_yaml, imgsz=736)

    print("\nCustom metrics (X, recalculated):")
    for k, v in custom_metrics.items():
        print(f"- {k}: {v}")

    try:
        display(per_image_df.head(10))
    except Exception:
        print(per_image_df.head(10))

    # Save local artifacts
    out_dir = Path("runs/segment/runs/colony_seg_mlflow/x_metrics_recalc")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_json = out_dir / "x_custom_metrics_recalc.json"
    out_csv = out_dir / "x_per_image_metrics_recalc.csv"
    out_json.write_text(json.dumps(custom_metrics, indent=2), encoding="utf-8")
    per_image_df.to_csv(out_csv, index=False, encoding="utf-8")
    print("Saved:", out_json.resolve())
    print("Saved:", out_csv.resolve())

    # Optionally log to MLflow as separate run (no retraining)
    if LOG_RECALC_TO_MLFLOW:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        mlflow.set_experiment(MLFLOW_EXPERIMENT)
        recalc_name = f"{run_x.data.tags.get('mlflow.runName','yolo26x')}_custom_metrics_recalc"
        with mlflow.start_run(run_name=recalc_name):
            mlflow.set_tags({
                "task": "segment_custom_metrics_recalc",
                "source_run_id": run_x.info.run_id,
                "source_model": X_MODEL_KEY,
                "retrain": "false",
            })
            mlflow.log_params({
                "data_yaml": str(data_yaml),
                "weights": str(best_weights_x),
                "test_dir": str(used_test_dir),
                "pred_conf": PRED_CONF,
                "pred_iou": PRED_IOU,
                "pred_max_det": PRED_MAX_DET,
            })
            mlflow.log_metrics({k: float(v) for k, v in custom_metrics.items() if pd.notna(v)})
            mlflow.log_artifact(str(out_json), artifact_path="recalc")
            mlflow.log_artifact(str(out_csv), artifact_path="recalc")

        print("Logged recalculated metrics to MLflow as separate run.")


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow.ipynb, cell 12
# This block runs only when RUN_COLONY_SEGMENTATION_TRAINING is True.
if RUN_COLONY_SEGMENTATION_TRAINING:
    # Validation summary table + comparison charts + training curves
    from pathlib import Path
    from datetime import datetime
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from ultralytics import YOLO
    import yaml

    RUN_ROOT_CANDIDATES = [
        Path("runs/segment/runs/segment/runs/colony_seg_mlflow_736"),
        Path("runs/segment/runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/segment/runs/colony_seg_mlflow_736"),
        Path("C:/ColonyNet/runs/segment/runs/segment/runs/colony_seg_mlflow"),
    ]

    DATA_YAML_CANDIDATES = [
        Path("cropped_736_aug_leaky/dataset/data.yaml"),
        Path("cropped_736/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736/dataset/data.yaml"),
    ]

    VAL_IMGSZ = 736
    VAL_CONF = 0.25
    VAL_IOU = 0.6
    VAL_BATCH = 1
    OUT_DIR = Path("runs/segment/runs/colony_seg_compare_736") / f"metrics_{datetime.now():%Y%m%d_%H%M%S}"


    def find_model_runs():
        model_dirs = []
        seen = set()

        for root in RUN_ROOT_CANDIDATES:
            if not root.exists():
                continue

            for d in sorted(root.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
                if not d.is_dir():
                    continue
                name = d.name.lower()
                if name.endswith("_test"):
                    continue
                if "yolo26" not in name or "-seg" not in name:
                    continue
                if "cropped736" not in name:
                    continue

                best_pt = d / "weights" / "best.pt"
                if not best_pt.exists():
                    continue

                key = str(best_pt.resolve()).lower()
                if key in seen:
                    continue
                seen.add(key)
                model_dirs.append(d)

        if not model_dirs:
            raise FileNotFoundError("No cropped736 model folders with weights/best.pt found.")

        return sorted(model_dirs, key=lambda x: x.name.lower())


    def resolve_data_yaml():
        for p in DATA_YAML_CANDIDATES:
            if p.exists():
                return p
        raise FileNotFoundError("Could not find data.yaml for cropped_736 dataset.")


    def resolve_eval_split(data_yaml_path):
        cfg = yaml.safe_load(Path(data_yaml_path).read_text(encoding="utf-8"))
        for split in ["test", "val"]:
            split_path = cfg.get(split)
            if split_path and Path(split_path).exists():
                return split
        return "val"


    def safe_metric(obj, attr_chain, default=np.nan):
        cur = obj
        try:
            for part in attr_chain.split("."):
                cur = getattr(cur, part)
            return float(cur)
        except Exception:
            return default


    def resolve_results_csv(model_dir):
        candidates = [
            model_dir / "results_736_4.csv",
            model_dir / "results.csv",
        ]
        for p in candidates:
            if p.exists():
                return p
        return None


    def short_model_label(name):
        name = str(name)
        for tag in ["yolo26n", "yolo26s", "yolo26m", "yolo26l", "yolo26x"]:
            if tag in name.lower():
                return tag
        return name


    model_dirs = find_model_runs()
    data_yaml = resolve_data_yaml()
    eval_split = resolve_eval_split(data_yaml)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Using DATA_YAML: {data_yaml.resolve()}")
    print(f"Using split: {eval_split}")

    rows = []
    curve_tables = {}

    for model_dir in model_dirs:
        model_name = model_dir.name
        model_label = short_model_label(model_name)
        weights_path = model_dir / "weights" / "best.pt"
        print(f"\n=== Validating {model_label} ===")

        model = YOLO(str(weights_path))
        metrics = model.val(
            data=str(data_yaml),
            split=eval_split,
            imgsz=VAL_IMGSZ,
            conf=VAL_CONF,
            iou=VAL_IOU,
            batch=VAL_BATCH,
            plots=False,
            verbose=False,
        )

        row = {
            "model": model_name,
            "model_short": model_label,
            "weights": str(weights_path.resolve()),
            "box_precision": safe_metric(metrics, "box.mp"),
            "box_recall": safe_metric(metrics, "box.mr"),
            "box_mAP50": safe_metric(metrics, "box.map50"),
            "box_mAP50_95": safe_metric(metrics, "box.map"),
            "seg_precision": safe_metric(metrics, "seg.mp"),
            "seg_recall": safe_metric(metrics, "seg.mr"),
            "seg_mAP50": safe_metric(metrics, "seg.map50"),
            "seg_mAP50_95": safe_metric(metrics, "seg.map"),
            "fitness": safe_metric(metrics, "fitness"),
            "preprocess_ms": np.nan,
            "inference_ms": np.nan,
            "postprocess_ms": np.nan,
        }

        try:
            speed = getattr(metrics, "speed", None)
            if isinstance(speed, dict):
                row["preprocess_ms"] = float(speed.get("preprocess", np.nan))
                row["inference_ms"] = float(speed.get("inference", np.nan))
                row["postprocess_ms"] = float(speed.get("postprocess", np.nan))
        except Exception:
            pass

        results_csv = resolve_results_csv(model_dir)
        row["results_csv"] = str(results_csv.resolve()) if results_csv else ""

        if results_csv is not None:
            df_curve = pd.read_csv(results_csv)
            df_curve.columns = [c.strip() for c in df_curve.columns]
            curve_tables[model_label] = df_curve

            if "epoch" in df_curve.columns:
                row["epochs_logged"] = int(df_curve["epoch"].max()) + 1
            else:
                row["epochs_logged"] = len(df_curve)

            last_row = df_curve.iloc[-1]

            for col in [
                "train/box_loss", "train/seg_loss", "train/cls_loss", "train/dfl_loss",
                "val/box_loss", "val/seg_loss", "val/cls_loss", "val/dfl_loss",
                "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)",
                "metrics/precision(M)", "metrics/recall(M)", "metrics/mAP50(M)", "metrics/mAP50-95(M)",
                "lr/pg0", "lr/pg1", "lr/pg2",
            ]:
                row[col] = float(last_row[col]) if col in df_curve.columns else np.nan
        else:
            row["epochs_logged"] = np.nan

        rows.append(row)

    summary_df = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
    display(summary_df)

    summary_csv = OUT_DIR / "model_metrics_summary_736.csv"
    summary_xlsx = OUT_DIR / "model_metrics_summary_736.xlsx"
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    summary_df.to_excel(summary_xlsx, index=False)

    print(f"Saved summary CSV:  {summary_csv.resolve()}")
    print(f"Saved summary XLSX: {summary_xlsx.resolve()}")

    # Comparison bar charts
    bar_metrics = [
        "seg_mAP50_95",
        "seg_mAP50",
        "seg_precision",
        "seg_recall",
        "box_mAP50_95",
        "inference_ms",
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    axes = axes.reshape(-1)

    for ax, metric_name in zip(axes, bar_metrics):
        plot_df = summary_df[["model_short", metric_name]].dropna().sort_values(metric_name, ascending=(metric_name == "inference_ms"))
        if plot_df.empty:
            ax.set_title(f"{metric_name} (no data)")
            ax.axis("off")
            continue

        ax.bar(plot_df["model_short"], plot_df[metric_name])
        ax.set_title(metric_name)
        ax.tick_params(axis="x", rotation=35)
        ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.show()

    # Training curves from results.csv / results_736_4.csv
    curve_specs = [
        ("train/seg_loss", "Train Seg Loss"),
        ("val/seg_loss", "Val Seg Loss"),
        ("metrics/mAP50(M)", "Mask mAP50"),
        ("metrics/mAP50-95(M)", "Mask mAP50-95"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.reshape(-1)

    for ax, (col, title) in zip(axes, curve_specs):
        plotted = False
        for model_name, df_curve in curve_tables.items():
            if col not in df_curve.columns:
                continue
            x = df_curve["epoch"] if "epoch" in df_curve.columns else np.arange(len(df_curve))
            ax.plot(x, df_curve[col], marker="o", linewidth=2, label=model_name)
            plotted = True

        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.25)
        if plotted:
            ax.legend()
        else:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)

    plt.tight_layout()
    plt.show()

    # Optional ranking by main mask metric
    rank_df = summary_df.sort_values(["seg_mAP50_95", "seg_mAP50"], ascending=False).reset_index(drop=True)
    display(rank_df[[
        "model",
        "seg_mAP50_95",
        "seg_mAP50",
        "seg_precision",
        "seg_recall",
        "box_mAP50_95",
        "inference_ms",
        "epochs_logged",
    ]])


# YOLO Colony Segmentation Metrics Report

Источник метрик: `C:\ColonyNet\mlflow_models_metrics_summary_latest.csv`

## Что означает каждая метрика

- `mAP50_95_M`: основная метрика качества сегментации масок. Чем выше, тем лучше. Это средний AP по IoU от 0.50 до 0.95.
- `mAP50_M`: более мягкая метрика качества масок при IoU = 0.50. Обычно выше, чем `mAP50_95_M`.
- `precision_M`: доля корректных масок среди предсказанных масок.
- `recall_M`: доля найденных объектов среди всех размеченных объектов.
- `mAP50_95_B`, `mAP50_B`, `precision_B`, `recall_B`: те же метрики, но для боксов, а не для масок.
- `dice_M_mean`: средний Dice по маскам. Удобен как дополнительная метрика overlap.
- `dice_M_median`: медианный Dice. Менее чувствителен к сильным выбросам.
- `mae_count`: средняя абсолютная ошибка по числу найденных колоний.
- `rmse_count`: среднеквадратичная ошибка по числу найденных колоний. Сильнее штрафует большие ошибки.
- `mape_count_nonzero`: относительная ошибка по числу объектов, рассчитанная на изображениях, где истинное число объектов не равно нулю.
- `fitness`: агрегированная служебная метрика Ultralytics для ранжирования модели. Полезна как общий ориентир, но для вашей задачи важнее смотреть прежде всего на mask-метрики.
- `duration_min`: время обучения в минутах.

## Сводка по моделям

| Model | mAP50-95 Mask | mAP50 Mask | Precision Mask | Recall Mask | Dice Mean | MAE Count | RMSE Count | MAPE Count | Duration, min |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| yolo26n-seg.pt | 0.3002 | 0.5684 | 0.8859 | 0.3932 | 0.7834 | 191.4688 | 394.5202 | 18.7775 | 37.4509 |
| yolo26s-seg.pt | 0.3836 | 0.6541 | 0.9362 | 0.4042 | 0.7886 | 191.2344 | 396.1689 | 17.7441 | 59.8995 |
| yolo26m-seg.pt | 0.4227 | 0.6713 | 0.9586 | 0.4160 | 0.7903 | 190.3125 | 394.6725 | 17.4073 | 322.1299 |
| yolo26l-seg.pt | 0.4314 | 0.6808 | 0.9650 | 0.4148 | 0.7901 | 190.4688 | 395.1548 | 17.3974 | 502.6380 |
| yolo26x-seg.pt | 0.4545 | 0.6942 | 0.9777 | 0.4201 | 0.7904 | 190.0625 | 394.4362 | 17.2524 | 1331.8361 |

## Вывод

- Лучшая модель по основной метрике сегментации `mAP50_95_M`: `yolo26x-seg.pt` (`0.4545`).
- Лучшая модель по `mAP50_M`: `yolo26x-seg.pt` (`0.6942`).
- Лучшая модель по `precision_M`: `yolo26x-seg.pt` (`0.9777`).
- Лучшая модель по `recall_M`: `yolo26x-seg.pt` (`0.4201`), но прирост recall относительно `m/l` уже небольшой.
- Лучшая модель по Dice: `yolo26x-seg.pt` (`0.7904`), но разница с `m` и `l` минимальна.
- По метрикам счета объектов `x` тоже лучшая, но выигрыш там уже небольшой.

## Практическая интерпретация

- Линейка `n -> s -> m -> l -> x` дает ожидаемый рост качества.
- Самый заметный скачок качества идет от `n/s` к `m`.
- Переход `m -> l -> x` улучшает метрики, но уже с убывающей отдачей.
- `yolo26x-seg.pt` дает лучший результат, но цена очень высокая: обучение заняло примерно `1331.8` мин, то есть около `22.2` часов.
- `yolo26m-seg.pt` выглядит наиболее разумным компромиссом между качеством и временем.

## Что смотреть в первую очередь для этой задачи

- Для сегментации колоний главный приоритет: `mAP50_95_M`.
- Затем: `mAP50_M`, `precision_M`, `recall_M`.
- `Dice` полезен как дополнительная проверка качества формы масок.
- Box-метрики вторичны, потому что ваша цель не боксы, а именно instance segmentation.



## 5. Segmentation Predictions and Visual Checks on Training Dataset

Source notebook: `C:/ColonyNet/yolo_colonies_mlflow.ipynb`.

Code cells in this section are guarded by `RUN_TRAINING_SET_PREDICTIONS`.


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow.ipynb, cell 5
# This block runs only when RUN_TRAINING_SET_PREDICTIONS is True.
if RUN_TRAINING_SET_PREDICTIONS:
    # Preview segmentation masks and contours only (no bounding boxes) for all trained YOLO models
    from pathlib import Path
    import re
    import colorsys

    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    import torch
    from ultralytics import YOLO

    IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    PREVIEW_IMAGE_PATHS = [
        Path("cropped_736/images/train/1127763_IMG_4363.jpg"),
        Path("cropped_736/images/train/1127763_IMG_4615.jpg"),
        Path("cropped_736/images/train/1127763_IMG_4672.jpg"),
        Path("cropped_736/images/train/1127763_IMG_6137.jpg"),
    ]
    PRED_CONF = 0.25
    PRED_IOU = 0.60
    MASK_ALPHA = 0.55
    COLS = 2
    CONTOUR_THICKNESS = 2
    SMOOTH_SIGMA = 0.9
    SAVE_PREVIEW_PNG = True
    PREVIEW_SAVE_DIR = Path("runs/segment/runs/colony_seg_mlflow/preview_masks_only")


    MODEL_ORDER = ["yolo26n-seg", "yolo26s-seg", "yolo26m-seg", "yolo26l-seg", "yolo26x-seg"]
    MODEL_TAG_RE = re.compile(r"^(yolo26[a-z]-seg)", re.IGNORECASE)


    if SAVE_PREVIEW_PNG:
        PREVIEW_SAVE_DIR.mkdir(parents=True, exist_ok=True)


    def resolve_preview_images():
        selected = []
        missing = []

        for rel_path in PREVIEW_IMAGE_PATHS:
            p = Path(rel_path)
            if p.exists():
                selected.append(p)
                continue

            p_abs = Path("C:/ColonyNet") / p
            if p_abs.exists():
                selected.append(p_abs)
            else:
                missing.append(str(rel_path))

        if missing:
            raise FileNotFoundError("Missing preview images:" + "\n" + "\n".join(missing))

        return selected



    def collect_trained_best_weights():
        by_model = {}

        # Include current globals if available
        extra_candidates = []
        if "best_weights" in globals():
            try:
                p = Path(best_weights)
                if p.exists():
                    extra_candidates.append(p)
            except Exception:
                pass

        if "run_dir" in globals():
            try:
                p = Path(run_dir) / "weights" / "best.pt"
                if p.exists():
                    extra_candidates.append(p)
            except Exception:
                pass

        for p in extra_candidates:
            run_name = p.parent.parent.name
            m = MODEL_TAG_RE.match(run_name)
            if not m:
                continue
            model_tag = m.group(1).lower()
            prev = by_model.get(model_tag)
            if prev is None or p.stat().st_mtime > prev[1].stat().st_mtime:
                by_model[model_tag] = (run_name, p)

        bases = [
            Path("runs/segment/runs/colony_seg_mlflow"),
            Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
        ]

        for base in bases:
            if not base.exists():
                continue
            for d in base.iterdir():
                if not d.is_dir() or d.name.endswith("_test"):
                    continue
                best = d / "weights" / "best.pt"
                if not best.exists():
                    continue

                m = MODEL_TAG_RE.match(d.name)
                if not m:
                    continue
                model_tag = m.group(1).lower()

                prev = by_model.get(model_tag)
                if prev is None or best.stat().st_mtime > prev[1].stat().st_mtime:
                    by_model[model_tag] = (d.name, best)

        if not by_model:
            raise FileNotFoundError("No trained best.pt found in colony_seg_mlflow runs.")

        ordered = []
        for tag in MODEL_ORDER:
            if tag in by_model:
                run_name, best = by_model[tag]
                ordered.append((tag, run_name, best))

        # Append any extra tags not in MODEL_ORDER
        for tag, (run_name, best) in sorted(by_model.items()):
            if tag not in MODEL_ORDER:
                ordered.append((tag, run_name, best))

        return ordered


    def color_for_idx(i, n_total=1):
        # High-contrast deterministic instance colors.
        if n_total < 1:
            n_total = 1
        t = ((i * 37) % n_total) / max(1, n_total - 1)
        hue = (0.02 + 0.96 * t) % 1.0
        sat = 0.95
        val = 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
        return np.array([255.0 * r, 255.0 * g, 255.0 * b], dtype=np.float32)


    def render_masks_only(image_path, result):
        bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if bgr is None:
            return None, 0

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
        overlay = rgb.copy()
        h, w = overlay.shape[:2]

        n_masks = 0
        if result.masks is not None and getattr(result.masks, "data", None) is not None:
            mask_data = result.masks.data
            if hasattr(mask_data, "detach"):
                mask_stack = mask_data.detach().cpu().numpy()
            else:
                mask_stack = np.asarray(mask_data)

            n_total = len(mask_stack)
            for k, mask_prob in enumerate(mask_stack):
                if mask_prob is None:
                    continue

                mask_prob = np.asarray(mask_prob, dtype=np.float32)
                if mask_prob.ndim != 2:
                    continue

                if mask_prob.shape != (h, w):
                    mask_prob = cv2.resize(mask_prob, (w, h), interpolation=cv2.INTER_LINEAR)

                # Soft mask edges for smoother visualization.
                mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=SMOOTH_SIGMA, sigmaY=SMOOTH_SIGMA)
                mask_alpha = np.clip(mask_prob, 0.0, 1.0) * MASK_ALPHA
                if float(mask_alpha.max()) < 0.01:
                    continue

                color = color_for_idx(k, n_total=n_total)
                for ch in range(3):
                    overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

                binary = (mask_prob >= 0.5).astype(np.uint8)
                if binary.any():
                    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                    if contours:
                        cv2.drawContours(overlay, contours, -1, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                    n_masks += 1

        elif result.masks is not None and getattr(result.masks, "xy", None) is not None:
            # Fallback to polygon mode if data masks are unavailable.
            polys = result.masks.xy
            n_total = len(polys)
            for k, poly in enumerate(polys):
                if poly is None or len(poly) < 3:
                    continue

                pts = np.round(poly).astype(np.int32)
                color = color_for_idx(k, n_total=n_total)

                mask = np.zeros((h, w), dtype=np.uint8)
                cv2.fillPoly(mask, [pts], 255)
                mask_alpha = (mask.astype(np.float32) / 255.0) * MASK_ALPHA

                for ch in range(3):
                    overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

                cv2.polylines(overlay, [pts], True, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                n_masks += 1

        out = np.clip(overlay, 0, 255).astype(np.uint8)
        return out, n_masks


    def id_from_name(path: Path) -> str:
        m = re.search(r"(\d+)(?!.*\d)", path.stem)
        return m.group(1) if m else path.stem


    weights_info = collect_trained_best_weights()
    preview_images = resolve_preview_images()

    print("Selected images:")
    for p in preview_images:
        print(f"- {p}")

    print("Models for preview:")
    for tag, run_name, w in weights_info:
        print(f"- {tag}: {w} (run: {run_name})")

    id_tag = "_".join(id_from_name(p) for p in preview_images)

    for tag, run_name, weights_path in weights_info:
        print(f"\nPreviewing {tag} from: {weights_path}")
        model = YOLO(str(weights_path))
        results = model.predict(
            source=[str(p) for p in preview_images],
            conf=PRED_CONF,
            iou=PRED_IOU,
            save=False,
            retina_masks=True,
            verbose=False,
        )

        n = len(preview_images)
        rows = int(np.ceil(n / COLS))
        fig, axes = plt.subplots(rows, COLS, figsize=(6 * COLS, 4.5 * rows))
        axes = np.array(axes).reshape(-1)

        for ax in axes:
            ax.axis("off")

        for ax, img_path, res in zip(axes, preview_images, results):
            out, n_masks = render_masks_only(img_path, res)
            if out is None:
                ax.set_title(f"{img_path.name} (read error)", fontsize=9)
                continue
            ax.imshow(out)
            ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
            ax.axis("off")

        fig.suptitle(f"{tag} | {run_name}", fontsize=14)
        plt.tight_layout()

        if SAVE_PREVIEW_PNG:
            out_png = PREVIEW_SAVE_DIR / f"{tag}_{run_name}_ids_{id_tag}_preview_v2.png"
            fig.savefig(out_png, dpi=180, bbox_inches="tight")
            print(f"Saved preview: {out_png}")

        plt.show()

        # Free memory between models
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow.ipynb, cell 10
# This block runs only when RUN_TRAINING_SET_PREDICTIONS is True.
if RUN_TRAINING_SET_PREDICTIONS:
    # Predict on full cropped_736/images/train using latest yolo26x-seg best weights
    from pathlib import Path
    from datetime import datetime
    import colorsys
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    from ultralytics import YOLO

    IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    SOURCE_DIR = Path("cropped_736/images/train")
    TARGET_PREVIEW_FILES = [
        "1127763_IMG_4363.jpg",
        "1127763_IMG_4615.jpg",
        "1127763_IMG_4672.jpg",
        "1127763_IMG_6137.jpg",
    ]
    PRED_CONF = 0.25
    PRED_IOU = 0.6
    PRED_IMGSZ = 736
    MASK_ALPHA = 0.55
    CONTOUR_THICKNESS = 2
    SMOOTH_SIGMA = 0.9


    def resolve_x_best_weights_for_predict():
        candidates = []

        # from previous cells
        for g in ["best_weights_x", "best_weights"]:
            if g in globals():
                try:
                    p = Path(globals()[g])
                    if p.exists() and p.name == "best.pt":
                        candidates.append(p)
                except Exception:
                    pass

        # latest x run directory
        for base in [
            Path("runs/segment/runs/colony_seg_mlflow"),
            Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
        ]:
            if not base.exists():
                continue
            for d in sorted(base.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
                if not d.is_dir() or d.name.endswith("_test"):
                    continue
                if "yolo26x-seg" not in d.name.lower():
                    continue
                p = d / "weights" / "best.pt"
                if p.exists():
                    candidates.append(p)

        if not candidates:
            raise FileNotFoundError("Could not find best.pt for yolo26x-seg")

        # newest first
        candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
        return candidates[0]


    def color_for_idx(i, n_total=1):
        # High-contrast deterministic instance colors.
        if n_total < 1:
            n_total = 1
        t = ((i * 37) % n_total) / max(1, n_total - 1)
        hue = (0.02 + 0.96 * t) % 1.0
        sat = 0.95
        val = 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
        return np.array([255.0 * r, 255.0 * g, 255.0 * b], dtype=np.float32)


    def render_masks_only(image_path, result):
        bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if bgr is None:
            return None, 0

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
        overlay = rgb.copy()
        h, w = overlay.shape[:2]

        n_masks = 0
        if result.masks is not None and getattr(result.masks, "data", None) is not None:
            mask_data = result.masks.data
            if hasattr(mask_data, "detach"):
                mask_stack = mask_data.detach().cpu().numpy()
            else:
                mask_stack = np.asarray(mask_data)

            n_total = len(mask_stack)
            for k, mask_prob in enumerate(mask_stack):
                if mask_prob is None:
                    continue

                mask_prob = np.asarray(mask_prob, dtype=np.float32)
                if mask_prob.ndim != 2:
                    continue

                if mask_prob.shape != (h, w):
                    mask_prob = cv2.resize(mask_prob, (w, h), interpolation=cv2.INTER_LINEAR)

                # Soft mask edges for smoother visualization.
                mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=SMOOTH_SIGMA, sigmaY=SMOOTH_SIGMA)
                mask_alpha = np.clip(mask_prob, 0.0, 1.0) * MASK_ALPHA
                if float(mask_alpha.max()) < 0.01:
                    continue

                color = color_for_idx(k, n_total=n_total)
                for ch in range(3):
                    overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

                binary = (mask_prob >= 0.5).astype(np.uint8)
                if binary.any():
                    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                    if contours:
                        cv2.drawContours(overlay, contours, -1, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                    n_masks += 1

        elif result.masks is not None and getattr(result.masks, "xy", None) is not None:
            # Fallback to polygon mode if data masks are unavailable.
            polys = result.masks.xy
            n_total = len(polys)
            for k, poly in enumerate(polys):
                if poly is None or len(poly) < 3:
                    continue

                pts = np.round(poly).astype(np.int32)
                color = color_for_idx(k, n_total=n_total)

                mask = np.zeros((h, w), dtype=np.uint8)
                cv2.fillPoly(mask, [pts], 255)
                mask_alpha = (mask.astype(np.float32) / 255.0) * MASK_ALPHA

                for ch in range(3):
                    overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

                cv2.polylines(overlay, [pts], True, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                n_masks += 1

        out = np.clip(overlay, 0, 255).astype(np.uint8)
        return out, n_masks


    if not SOURCE_DIR.exists():
        raise FileNotFoundError(f"Source folder not found: {SOURCE_DIR}")

    images = sorted([p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    if not images:
        raise RuntimeError(f"No images found in: {SOURCE_DIR}")

    weights_path = resolve_x_best_weights_for_predict()
    print(f"Using weights: {weights_path}")
    print(f"Source folder: {SOURCE_DIR.resolve()}")
    print(f"Images count: {len(images)}")

    pred_project = Path("runs/colony_seg_mlflow")
    pred_name = f"x_predict_train_{datetime.now():%Y%m%d_%H%M%S}"

    model = YOLO(str(weights_path))
    results = model.predict(
        source=str(SOURCE_DIR),
        conf=PRED_CONF,
        iou=PRED_IOU,
        imgsz=PRED_IMGSZ,
        save=True,
        show_boxes=False,
        retina_masks=True,
        project=str(pred_project),
        name=pred_name,
        exist_ok=True,
        verbose=True,
    )

    # Resolve real save directory from Ultralytics output (robust to internal path prefixes)
    pred_dir = None
    if results:
        try:
            sd = getattr(results[0], "save_dir", None)
            if sd:
                pred_dir = Path(sd)
        except Exception:
            pass

    if pred_dir is None or not pred_dir.exists():
        # fallback search by folder name
        matches = sorted(Path("runs").rglob(pred_name), key=lambda x: x.stat().st_mtime, reverse=True)
        if matches:
            pred_dir = matches[0]

    if pred_dir is None or not pred_dir.exists():
        raise FileNotFoundError(f"Prediction output folder not found for: {pred_name}")

    print(f"Saved predictions to: {pred_dir.resolve()}")

    # Preview ONLY requested files with custom per-instance colors
    preview_paths = [SOURCE_DIR / fn for fn in TARGET_PREVIEW_FILES]
    missing = [str(p) for p in preview_paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing preview files:" + "\n" + "\n".join(missing))

    print("Preview files:")
    for p in preview_paths:
        print(f"- {p}")

    preview_results = model.predict(
        source=[str(p) for p in preview_paths],
        conf=PRED_CONF,
        iou=PRED_IOU,
        imgsz=PRED_IMGSZ,
        save=False,
        show_boxes=False,
        retina_masks=True,
        verbose=False,
    )

    cols = 2
    rows = int(np.ceil(len(preview_paths) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.8 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")

    for ax, img_path, res in zip(axes, preview_paths, preview_results):
        out, n_masks = render_masks_only(img_path, res)
        if out is None:
            ax.set_title(f"{img_path.name} (read error)", fontsize=9)
            continue
        ax.imshow(out)
        ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    # Save this custom preview panel
    custom_png = pred_dir / "x_custom_preview_4363_4615_4672_6137_masks_only.png"
    fig.savefig(custom_png, dpi=180, bbox_inches="tight")
    print(f"Saved custom preview: {custom_png.resolve()}")


In [ ]:
# Source: C:\ColonyNet\yolo_colonies_mlflow.ipynb, cell 11
# This block runs only when RUN_TRAINING_SET_PREDICTIONS is True.
if RUN_TRAINING_SET_PREDICTIONS:
    # Predict + preview for all cropped736 models
    from pathlib import Path
    from datetime import datetime
    import colorsys
    import cv2
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from ultralytics import YOLO

    IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    SOURCE_DIR = Path("cropped_736/images/train")
    TARGET_PREVIEW_FILES = [
        "1127763_IMG_4363.jpg",
        "1127763_IMG_4615.jpg",
        "1127763_IMG_4672.jpg",
        "1127763_IMG_6137.jpg",
    ]

    RUN_ROOT_CANDIDATES = [
        Path("runs/segment/runs/segment/runs/colony_seg_mlflow_736"),
        Path("runs/segment/runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/segment/runs/colony_seg_mlflow_736"),
        Path("C:/ColonyNet/runs/segment/runs/segment/runs/colony_seg_mlflow"),
    ]

    PRED_CONF = 0.25
    PRED_IOU = 0.6
    PRED_IMGSZ = 736
    MASK_ALPHA = 0.55
    CONTOUR_THICKNESS = 2
    SMOOTH_SIGMA = 0.9
    PRED_PROJECT = Path("runs/segment/runs/colony_seg_predicts_736")


    def find_model_runs():
        model_dirs = []
        seen = set()

        for root in RUN_ROOT_CANDIDATES:
            if not root.exists():
                continue

            for d in sorted(root.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
                if not d.is_dir():
                    continue
                name = d.name.lower()
                if name.endswith("_test"):
                    continue
                if "yolo26" not in name or "-seg" not in name:
                    continue
                if "cropped736" not in name:
                    continue

                best_pt = d / "weights" / "best.pt"
                if not best_pt.exists():
                    continue

                key = str(best_pt.resolve()).lower()
                if key in seen:
                    continue
                seen.add(key)

                model_dirs.append(d)

        if not model_dirs:
            raise FileNotFoundError(
                "No model folders with cropped736 and weights/best.pt were found in:\n"
                + "\n".join(str(p.resolve()) for p in RUN_ROOT_CANDIDATES if p.exists())
            )

        model_dirs = sorted(model_dirs, key=lambda x: x.name.lower())
        return model_dirs


    def color_for_idx(i, n_total=1):
        if n_total < 1:
            n_total = 1
        t = ((i * 37) % n_total) / max(1, n_total - 1)
        hue = (0.02 + 0.96 * t) % 1.0
        sat = 0.95
        val = 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
        return np.array([255.0 * r, 255.0 * g, 255.0 * b], dtype=np.float32)


    def render_masks_only(image_path, result):
        bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if bgr is None:
            return None, 0

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
        overlay = rgb.copy()
        h, w = overlay.shape[:2]
        n_masks = 0

        if result.masks is not None and getattr(result.masks, "data", None) is not None:
            mask_data = result.masks.data
            mask_stack = mask_data.detach().cpu().numpy() if hasattr(mask_data, "detach") else np.asarray(mask_data)
            n_total = len(mask_stack)

            for k, mask_prob in enumerate(mask_stack):
                if mask_prob is None:
                    continue

                mask_prob = np.asarray(mask_prob, dtype=np.float32)
                if mask_prob.ndim != 2:
                    continue

                if mask_prob.shape != (h, w):
                    mask_prob = cv2.resize(mask_prob, (w, h), interpolation=cv2.INTER_LINEAR)

                mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=SMOOTH_SIGMA, sigmaY=SMOOTH_SIGMA)
                mask_alpha = np.clip(mask_prob, 0.0, 1.0) * MASK_ALPHA
                if float(mask_alpha.max()) < 0.01:
                    continue

                color = color_for_idx(k, n_total=n_total)
                for ch in range(3):
                    overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

                binary = (mask_prob >= 0.5).astype(np.uint8)
                if binary.any():
                    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                    if contours:
                        cv2.drawContours(overlay, contours, -1, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                    n_masks += 1

        elif result.masks is not None and getattr(result.masks, "xy", None) is not None:
            polys = result.masks.xy
            n_total = len(polys)

            for k, poly in enumerate(polys):
                if poly is None or len(poly) < 3:
                    continue

                pts = np.round(poly).astype(np.int32)
                color = color_for_idx(k, n_total=n_total)

                mask = np.zeros((h, w), dtype=np.uint8)
                cv2.fillPoly(mask, [pts], 255)
                mask_alpha = (mask.astype(np.float32) / 255.0) * MASK_ALPHA

                for ch in range(3):
                    overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

                cv2.polylines(overlay, [pts], True, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                n_masks += 1

        out = np.clip(overlay, 0, 255).astype(np.uint8)
        return out, n_masks


    if not SOURCE_DIR.exists():
        raise FileNotFoundError(f"Source folder not found: {SOURCE_DIR}")

    all_images = sorted([p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    if not all_images:
        raise RuntimeError(f"No images found in: {SOURCE_DIR}")

    preview_paths = [SOURCE_DIR / fn for fn in TARGET_PREVIEW_FILES]
    missing = [str(p) for p in preview_paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing preview files:\n" + "\n".join(missing))

    model_dirs = find_model_runs()
    print("Found models:")
    for d in model_dirs:
        print("-", d)

    PRED_PROJECT.mkdir(parents=True, exist_ok=True)
    predict_rows = []

    for model_dir in model_dirs:
        weights_path = model_dir / "weights" / "best.pt"
        model_name = model_dir.name
        pred_name = f"{model_name}_predict_train_{datetime.now():%Y%m%d_%H%M%S}"

        print(f"\n=== Predicting with {model_name} ===")
        model = YOLO(str(weights_path))

        results = model.predict(
            source=str(SOURCE_DIR),
            conf=PRED_CONF,
            iou=PRED_IOU,
            imgsz=PRED_IMGSZ,
            save=True,
            show_boxes=False,
            retina_masks=True,
            project=str(PRED_PROJECT),
            name=pred_name,
            exist_ok=True,
            verbose=True,
        )

        pred_dir = None
        if results:
            try:
                sd = getattr(results[0], "save_dir", None)
                if sd:
                    pred_dir = Path(sd)
            except Exception:
                pass

        if pred_dir is None or not pred_dir.exists():
            matches = sorted(PRED_PROJECT.rglob(pred_name), key=lambda x: x.stat().st_mtime, reverse=True)
            if matches:
                pred_dir = matches[0]

        if pred_dir is None or not pred_dir.exists():
            raise FileNotFoundError(f"Prediction output folder not found for: {pred_name}")

        preview_results = model.predict(
            source=[str(p) for p in preview_paths],
            conf=PRED_CONF,
            iou=PRED_IOU,
            imgsz=PRED_IMGSZ,
            save=False,
            show_boxes=False,
            retina_masks=True,
            verbose=False,
        )

        cols = 2
        rows = int(np.ceil(len(preview_paths) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.8 * rows))
        axes = np.array(axes).reshape(-1)
        for ax in axes:
            ax.axis("off")

        preview_mask_counts = {}
        for ax, img_path, res in zip(axes, preview_paths, preview_results):
            out, n_masks = render_masks_only(img_path, res)
            preview_mask_counts[img_path.name] = n_masks
            if out is None:
                ax.set_title(f"{img_path.name} (read error)", fontsize=9)
                continue
            ax.imshow(out)
            ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
            ax.axis("off")

        plt.tight_layout()
        plt.show()

        custom_png = pred_dir / f"{model_name}_preview_4363_4615_4672_6137_masks_only.png"
        fig.savefig(custom_png, dpi=180, bbox_inches="tight")
        plt.close(fig)

        predict_rows.append(
            {
                "model": model_name,
                "weights": str(weights_path.resolve()),
                "pred_dir": str(pred_dir.resolve()),
                "preview_png": str(custom_png.resolve()),
                "masks_4363": preview_mask_counts.get("1127763_IMG_4363.jpg"),
                "masks_4615": preview_mask_counts.get("1127763_IMG_4615.jpg"),
                "masks_4672": preview_mask_counts.get("1127763_IMG_4672.jpg"),
                "masks_6137": preview_mask_counts.get("1127763_IMG_6137.jpg"),
            }
        )

    predict_df = pd.DataFrame(predict_rows).sort_values("model").reset_index(drop=True)
    display(predict_df)

    predict_summary_csv = PRED_PROJECT / "predict_summary_cropped736.csv"
    predict_summary_xlsx = PRED_PROJECT / "predict_summary_cropped736.xlsx"
    predict_df.to_csv(predict_summary_csv, index=False, encoding="utf-8-sig")
    predict_df.to_excel(predict_summary_xlsx, index=False)

    print(f"Saved summary CSV:  {predict_summary_csv.resolve()}")
    print(f"Saved summary XLSX: {predict_summary_xlsx.resolve()}")


## 6. Final Two-Model Prediction and Anomaly Detection

The next code cell embeds the standalone pipeline generated in `full_pipline/full_pipeline.py`. It contains model loading, Petri crop/resize, YOLO26x-seg inference, mask cleaning, feature extraction, anomaly scoring, reports, and visualization.


In [ ]:
# Auto-generated standalone full pipeline from notebooks/colony_anomaly_detection_yolo_x_improved.ipynb.
# Runs: Petri detector -> crop/resize 736x736 -> YOLO26x-seg -> colony anomaly analysis.

import os
import math
import json
import colorsys
import warnings
from pathlib import Path
from dataclasses import dataclass, replace
from typing import Any

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from scipy.spatial import distance
from scipy.spatial import cKDTree
from scipy import stats
from scipy.ndimage import binary_fill_holes

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from skimage.measure import regionprops, label, find_contours
from skimage.morphology import binary_dilation, disk
from skimage.feature import local_binary_pattern
from skimage import exposure

try:
    from skimage.feature import graycomatrix, graycoprops
except ImportError:
    from skimage.feature import greycomatrix as graycomatrix
    from skimage.feature import greycoprops as graycoprops

from ultralytics import YOLO

warnings.filterwarnings("ignore")

try:
    PIPELINE_DIR = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    PIPELINE_DIR = _cwd / "full_pipline" if (_cwd / "full_pipline").exists() else _cwd
MODELS_DIR = PIPELINE_DIR / "models"
DEFAULT_PETRI_DETECTOR_WEIGHTS = MODELS_DIR / "petri_detector_yolo26s_best.pt"
DEFAULT_COLONY_SEG_WEIGHTS = MODELS_DIR / "colony_yolo26x_seg_best.pt"

@dataclass
class AnomalyDetectionConfig:
    # Preprocessing mode: already_cropped_736 / detect_petri / resize_only
    preprocess_mode: str = "already_cropped_736"
    use_petri_detector: bool = False

    # Optional Petri dish detector for raw photos
    petri_detector_weights_path: str = str(DEFAULT_PETRI_DETECTOR_WEIGHTS)
    petri_det_conf: float = 0.25
    petri_det_iou: float = 0.45
    petri_det_max_det: int = 10
    petri_bbox_margin: float = 0.01

    # YOLO26x-seg colony segmentation model
    model_dir: str = str(MODELS_DIR)
    model_weights_path: str | None = str(DEFAULT_COLONY_SEG_WEIGHTS)

    input_path: str = r"data\colonies_736"
    output_dir: str = r"outputs\colony_anomaly_detection"

    # YOLO inference
    imgsz: int = 736
    conf: float = 0.10
    iou: float = 0.65
    max_det: int = 1000
    retina_masks: bool = True
    classes: tuple[int, ...] | None = None
    device: str | None = None

    # Mask filtering
    min_area: int = 25
    min_area_relative_to_median: float = 0.03
    max_area_fraction: float = 0.10
    max_area_relative_to_median: float = 10.0
    min_yolo_conf_for_analysis: float = 0.05
    edge_margin_px: int = 2
    exclude_border_touching_from_anomaly: bool = True
    exclude_technical_from_anomaly: bool = True
    enforce_non_overlap: bool = True
    suspicious_aspect_ratio_threshold: float = 4.0

    # Mask cleanup
    clean_masks: bool = True
    keep_largest_component: bool = True
    fill_mask_holes: bool = True
    min_component_area: int = 10

    # Local background and neighbors
    local_background_radius: int = 12
    local_density_radius: float = 35.0
    neighbor_k: int = 5

    # Texture
    glcm_levels: int = 32
    glcm_distances: tuple[int, ...] = (1, 2)
    lbp_p: int = 8
    lbp_r: int = 1

    # Histograms
    use_histogram_features: bool = True
    hist_bins: int = 32
    hist_use_lab: bool = True
    hist_use_intensity: bool = True
    hist_use_contrast: bool = True

    # Morphotypes
    use_morphotype_clustering: bool = True
    morphotype_method: str = "kmeans"
    max_morphotypes: int = 4
    min_morphotype_size: int = 5
    rare_morphotype_fraction: float = 0.08
    min_morphotype_silhouette: float = 0.20

    # Anomaly scoring
    lof_neighbors: int = 20
    contamination: float = 0.05
    dbscan_eps: float = 1.8
    dbscan_min_samples: int | None = None
    use_group_scores: bool = True
    use_weighted_final_score: bool = True
    random_state: int = 42

    # Objective selection controls
    min_independent_evidence_count: int = 2
    min_reliability_score: float = 0.50
    min_final_score_raw: float = 0.65
    min_absolute_evidence_strength: float = 0.35
    min_consensus_score: float = 0.30
    clip_relative_features: float = 5.0
    neighbor_difference_weight: float = 0.05
    spatial_context_weight: float = 0.03

    # Stability / ablation
    use_ablation_stability: bool = True
    use_perturbation_stability: bool = False

    # Selection
    top_k: int | None = None
    top_percent: float | None = None
    min_score: float | None = None

    # Saving
    save_visualizations: bool = True
    save_csv: bool = True
    save_xlsx: bool = True
    save_colony_crops: bool = True
    save_feature_space_plot: bool = True
    save_feature_correlation_report: bool = True


config = AnomalyDetectionConfig()

def find_best_yolo_weights(model_dir: str | Path) -> Path:
    model_dir = Path(model_dir)
    if not model_dir.exists():
        raise FileNotFoundError(f"Model directory not found: {model_dir}")

    candidates = [
        model_dir / "weights" / "best.pt",
        model_dir / "best.pt",
        model_dir / "weights" / "last.pt",
        model_dir / "last.pt",
    ]

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate

    pt_files = [p for p in model_dir.rglob("*.pt") if p.is_file()]
    if not pt_files:
        raise FileNotFoundError(f"No .pt files found in model directory: {model_dir}")

    pt_files = sorted(
        pt_files,
        key=lambda p: ("yolo26x" not in str(p).lower(), str(p).lower()),
    )

    best_named = [p for p in pt_files if "best" in p.name.lower()]
    if best_named:
        return best_named[0]

    return max(pt_files, key=lambda p: p.stat().st_mtime)

def collect_image_paths(input_path: str | Path) -> list[Path]:
    input_path = Path(input_path)
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    if not input_path.exists():
        raise FileNotFoundError(f"Input path not found: {input_path}")

    if input_path.is_file():
        if input_path.suffix.lower() not in image_exts:
            raise ValueError(f"File is not a supported image: {input_path}")
        return [input_path]

    image_paths = [
        p for p in input_path.rglob("*") if p.is_file() and p.suffix.lower() in image_exts
    ]
    image_paths = sorted(image_paths)

    if not image_paths:
        raise FileNotFoundError(
            f"No images found in directory {input_path} with extensions {sorted(image_exts)}"
        )

    return image_paths


def read_image_rgb(path: str | Path) -> np.ndarray:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f"Failed to read image: {path}")

    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def resize_to_config(image_rgb: np.ndarray, config: AnomalyDetectionConfig) -> np.ndarray:
    h, w = image_rgb.shape[:2]
    if (h, w) == (config.imgsz, config.imgsz):
        return image_rgb.copy()
    interpolation = cv2.INTER_AREA if max(h, w) > int(config.imgsz) else cv2.INTER_LINEAR
    return cv2.resize(image_rgb, (int(config.imgsz), int(config.imgsz)), interpolation=interpolation)


def detect_petri_and_resize(
    image_rgb: np.ndarray,
    detector_model,
    config: AnomalyDetectionConfig,
) -> tuple[np.ndarray, dict[str, Any]]:
    h, w = image_rgb.shape[:2]
    meta: dict[str, Any] = {
        "original_shape": [int(h), int(w)],
        "preprocess_mode": config.preprocess_mode,
        "petri_detected": False,
        "detector_conf": np.nan,
        "crop_xyxy": [0, 0, int(w), int(h)],
        "detection_error": None,
    }

    crop = image_rgb

    if detector_model is not None:
        try:
            det_results = detector_model.predict(
                source=image_rgb,
                conf=float(config.petri_det_conf),
                iou=float(config.petri_det_iou),
                max_det=int(config.petri_det_max_det),
                device=config.device,
                verbose=False,
            )

            if det_results is not None and len(det_results) > 0:
                det_res = det_results[0]
                if det_res.boxes is not None and det_res.boxes.xyxy is not None and len(det_res.boxes.xyxy) > 0:
                    boxes = det_res.boxes.xyxy.cpu().numpy().astype(float)
                    confs = (
                        det_res.boxes.conf.cpu().numpy().astype(float)
                        if det_res.boxes.conf is not None
                        else np.full((boxes.shape[0],), np.nan, dtype=float)
                    )

                    best_idx = int(np.nanargmax(confs)) if confs.size and np.isfinite(confs).any() else 0
                    x1, y1, x2, y2 = boxes[best_idx]
                    conf_best = float(confs[best_idx]) if confs.size > best_idx else np.nan

                    bw = max(1.0, x2 - x1)
                    bh = max(1.0, y2 - y1)
                    mx = int(round(bw * float(config.petri_bbox_margin)))
                    my = int(round(bh * float(config.petri_bbox_margin)))

                    x1i = max(0, int(np.floor(x1)) - mx)
                    y1i = max(0, int(np.floor(y1)) - my)
                    x2i = min(w, int(np.ceil(x2)) + mx)
                    y2i = min(h, int(np.ceil(y2)) + my)

                    if x2i > x1i and y2i > y1i:
                        crop_candidate = image_rgb[y1i:y2i, x1i:x2i]
                        if crop_candidate.size > 0:
                            crop = crop_candidate
                            meta["petri_detected"] = True
                            meta["detector_conf"] = conf_best
                            meta["crop_xyxy"] = [int(x1i), int(y1i), int(x2i), int(y2i)]
        except Exception as exc:
            meta["detection_error"] = str(exc)

    ch, cw = crop.shape[:2]
    prepared = resize_to_config(crop, config)
    meta["crop_shape"] = [int(ch), int(cw)]
    meta["prepared_shape"] = [int(prepared.shape[0]), int(prepared.shape[1])]
    return prepared, meta


def prepare_image_for_segmentation(
    image_rgb: np.ndarray,
    detector_model,
    config: AnomalyDetectionConfig,
) -> tuple[np.ndarray, dict[str, Any]]:
    mode = str(config.preprocess_mode).lower()
    h, w = image_rgb.shape[:2]

    if mode == "already_cropped_736":
        prepared = resize_to_config(image_rgb, config)
        return prepared, {
            "original_shape": [int(h), int(w)],
            "preprocess_mode": mode,
            "petri_detected": False,
            "crop_xyxy": [0, 0, int(w), int(h)],
            "crop_shape": [int(h), int(w)],
            "prepared_shape": [int(prepared.shape[0]), int(prepared.shape[1])],
        }

    if mode == "resize_only":
        prepared = resize_to_config(image_rgb, config)
        return prepared, {
            "original_shape": [int(h), int(w)],
            "preprocess_mode": mode,
            "petri_detected": False,
            "crop_xyxy": [0, 0, int(w), int(h)],
            "crop_shape": [int(h), int(w)],
            "prepared_shape": [int(prepared.shape[0]), int(prepared.shape[1])],
        }

    if mode == "detect_petri":
        if not config.use_petri_detector:
            raise ValueError("preprocess_mode='detect_petri' requires config.use_petri_detector=True")
        return detect_petri_and_resize(image_rgb, detector_model, config)

    raise ValueError(
        "Unsupported preprocess_mode. Use 'already_cropped_736', 'resize_only', or 'detect_petri'."
    )

def _component_count(mask_bool: np.ndarray) -> int:
    mask_bool = np.asarray(mask_bool).astype(bool)
    if mask_bool.ndim != 2 or not mask_bool.any():
        return 0
    return int(len(regionprops(label(mask_bool.astype(np.uint8)))))


def clean_instance_mask(mask: np.ndarray, config: AnomalyDetectionConfig) -> tuple[np.ndarray, dict[str, Any]]:
    mask = np.asarray(mask).astype(bool)
    diagnostics: dict[str, Any] = {
        "original_area": float(mask.sum()) if mask.ndim == 2 else 0.0,
        "cleaned_area": 0.0,
        "area_loss_fraction": 1.0,
        "n_components_before": 0,
        "n_components_after": 0,
        "mask_fragment_after_overlap": False,
    }

    if mask.ndim != 2:
        return np.zeros_like(mask, dtype=bool), diagnostics

    diagnostics["n_components_before"] = _component_count(mask)
    if not mask.any():
        return np.zeros_like(mask, dtype=bool), diagnostics

    if not bool(getattr(config, "clean_masks", True)):
        cleaned = mask.astype(bool)
        cleaned_area = float(cleaned.sum())
        diagnostics.update(
            cleaned_area=cleaned_area,
            area_loss_fraction=0.0,
            n_components_after=_component_count(cleaned),
            mask_fragment_after_overlap=bool(cleaned_area < float(config.min_component_area)),
        )
        return cleaned, diagnostics

    cleaned = mask.copy()

    if config.min_component_area and int(config.min_component_area) > 1:
        labeled = label(cleaned.astype(np.uint8))
        keep = np.zeros_like(cleaned, dtype=bool)
        for region in regionprops(labeled):
            if region.area >= int(config.min_component_area):
                keep[labeled == region.label] = True
        cleaned = keep

    if config.keep_largest_component and cleaned.any():
        labeled = label(cleaned.astype(np.uint8))
        regions = regionprops(labeled)
        if regions:
            largest = max(regions, key=lambda r: r.area)
            cleaned = labeled == largest.label

    if config.fill_mask_holes and cleaned.any():
        try:
            cleaned = binary_fill_holes(cleaned).astype(bool)
        except Exception:
            pass

    original_area = max(float(diagnostics["original_area"]), 1.0)
    cleaned_area = float(cleaned.sum())
    area_loss_fraction = float(np.clip(1.0 - cleaned_area / original_area, 0.0, 1.0))
    diagnostics.update(
        cleaned_area=cleaned_area,
        area_loss_fraction=area_loss_fraction,
        n_components_after=_component_count(cleaned),
        mask_fragment_after_overlap=bool(
            cleaned_area < float(config.min_component_area)
            or (diagnostics["original_area"] >= float(config.min_area) and cleaned_area < 0.5 * diagnostics["original_area"])
        ),
    )
    return cleaned.astype(bool), diagnostics


def make_masks_non_overlapping(
    masks: list[np.ndarray],
    confs: np.ndarray,
    boxes: np.ndarray,
    config: AnomalyDetectionConfig,
    mask_meta: list[dict[str, Any]] | None = None,
) -> tuple[list[np.ndarray], np.ndarray, np.ndarray, list[dict[str, Any]]]:
    if mask_meta is None:
        mask_meta = []
        for m in masks:
            area = float(np.asarray(m).sum())
            mask_meta.append(
                {
                    "mask_area_raw": area,
                    "mask_area_cleaned": area,
                    "mask_area_after_overlap": area,
                    "mask_area_loss_fraction": 0.0,
                    "original_area": area,
                    "cleaned_area": area,
                    "area_loss_fraction": 0.0,
                    "n_components_before": _component_count(m),
                    "n_components_after": _component_count(m),
                    "mask_fragment_after_overlap": False,
                }
            )

    if len(masks) <= 1 or not config.enforce_non_overlap:
        for i, m in enumerate(masks):
            area = float(np.asarray(m).sum())
            before = float(mask_meta[i].get("mask_area_cleaned", area))
            loss = 1.0 - area / max(before, 1.0)
            mask_meta[i]["mask_area_after_overlap"] = area
            mask_meta[i]["mask_area_loss_fraction"] = float(np.clip(loss, 0.0, 1.0))
            mask_meta[i]["mask_fragment_after_overlap"] = bool(
                mask_meta[i].get("mask_fragment_after_overlap", False)
                or area < float(config.min_component_area)
                or (before >= float(config.min_area) and area < 0.5 * before)
            )
        keep_indices = [i for i, m in enumerate(masks) if np.asarray(m).any()]
        return (
            [masks[i] for i in keep_indices],
            confs[keep_indices] if len(keep_indices) else np.array([], dtype=float),
            boxes[keep_indices] if len(keep_indices) else np.empty((0, 4), dtype=float),
            [mask_meta[i] for i in keep_indices],
        )

    h, w = masks[0].shape
    conf_priority = np.nan_to_num(confs, nan=-np.inf)
    priority_indices = np.argsort(conf_priority)[::-1]
    occupied = np.zeros((h, w), dtype=bool)
    exclusive_masks: list[np.ndarray] = [np.zeros((h, w), dtype=bool) for _ in range(len(masks))]

    for idx in priority_indices:
        before_area = float(np.asarray(masks[idx]).sum())
        unique_pixels = masks[idx] & (~occupied)
        if config.clean_masks:
            unique_pixels, diag = clean_instance_mask(unique_pixels, config)
            mask_meta[idx]["n_components_after"] = diag.get("n_components_after", mask_meta[idx].get("n_components_after", np.nan))
        after_area = float(np.asarray(unique_pixels).sum())
        loss_fraction = 1.0 - after_area / max(before_area, 1.0)
        mask_meta[idx]["mask_area_after_overlap"] = after_area
        mask_meta[idx]["mask_area_loss_fraction"] = float(np.clip(loss_fraction, 0.0, 1.0))
        mask_meta[idx]["mask_fragment_after_overlap"] = bool(
            mask_meta[idx].get("mask_fragment_after_overlap", False)
            or after_area < float(config.min_component_area)
            or (before_area >= float(config.min_area) and after_area < 0.5 * before_area)
        )
        exclusive_masks[idx] = unique_pixels
        occupied |= unique_pixels

    keep_indices = [i for i, m in enumerate(exclusive_masks) if np.asarray(m).any()]
    masks = [exclusive_masks[i] for i in keep_indices]
    confs = confs[keep_indices] if len(keep_indices) else np.array([], dtype=float)
    boxes = boxes[keep_indices] if len(keep_indices) else np.empty((0, 4), dtype=float)
    mask_meta = [mask_meta[i] for i in keep_indices]
    return masks, confs, boxes, mask_meta


def predict_colony_masks(
    model,
    image_rgb: np.ndarray,
    config: AnomalyDetectionConfig,
) -> tuple[list[np.ndarray], pd.DataFrame]:
    h, w = image_rgb.shape[:2]

    detections_columns = [
        "colony_id",
        "yolo_conf",
        "x1",
        "y1",
        "x2",
        "y2",
        "bbox_area_yolo",
        "mask_area_raw",
        "mask_area_cleaned",
        "mask_area_after_overlap",
        "mask_area_loss_fraction",
        "original_area",
        "cleaned_area",
        "area_loss_fraction",
        "n_components_before",
        "n_components_after",
        "mask_fragment_after_overlap",
    ]

    predict_kwargs = dict(
        source=image_rgb,
        imgsz=config.imgsz,
        conf=config.conf,
        iou=config.iou,
        max_det=config.max_det,
        device=config.device,
        verbose=False,
        retina_masks=config.retina_masks,
    )
    if config.classes is not None:
        predict_kwargs["classes"] = list(config.classes)

    results = model.predict(**predict_kwargs)

    if results is None or len(results) == 0:
        return [], pd.DataFrame(columns=detections_columns)

    result = results[0]
    if result.masks is None or result.masks.data is None or len(result.masks.data) == 0:
        return [], pd.DataFrame(columns=detections_columns)

    masks_np = result.masks.data.cpu().numpy()
    if masks_np.ndim == 2:
        masks_np = masks_np[None, ...]

    masks: list[np.ndarray] = []
    raw_keep_indices: list[int] = []
    mask_meta: list[dict[str, Any]] = []
    for raw_idx, m in enumerate(masks_np):
        m_bool = m.astype(np.float32) > 0.5
        if m_bool.shape != (h, w):
            m_bool = cv2.resize(m_bool.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
        raw_area = float(m_bool.sum())
        cleaned, diag = clean_instance_mask(m_bool, config)
        cleaned_area = float(cleaned.sum())
        if cleaned.any():
            masks.append(cleaned)
            raw_keep_indices.append(raw_idx)
            mask_meta.append(
                {
                    "mask_area_raw": raw_area,
                    "mask_area_cleaned": cleaned_area,
                    "mask_area_after_overlap": cleaned_area,
                    "mask_area_loss_fraction": float(diag.get("area_loss_fraction", 0.0)),
                    "original_area": float(diag.get("original_area", raw_area)),
                    "cleaned_area": float(diag.get("cleaned_area", cleaned_area)),
                    "area_loss_fraction": float(diag.get("area_loss_fraction", 0.0)),
                    "n_components_before": int(diag.get("n_components_before", 0)),
                    "n_components_after": int(diag.get("n_components_after", 0)),
                    "mask_fragment_after_overlap": bool(diag.get("mask_fragment_after_overlap", False)),
                }
            )

    n_masks = len(masks)
    if n_masks == 0:
        return [], pd.DataFrame(columns=detections_columns)

    confs = result.boxes.conf.cpu().numpy().astype(float) if result.boxes is not None and result.boxes.conf is not None else np.array([], dtype=float)
    boxes = result.boxes.xyxy.cpu().numpy().astype(float) if result.boxes is not None and result.boxes.xyxy is not None else np.empty((0, 4), dtype=float)

    if raw_keep_indices and confs.shape[0] > max(raw_keep_indices):
        confs = confs[raw_keep_indices]
    if raw_keep_indices and boxes.shape[0] > max(raw_keep_indices):
        boxes = boxes[raw_keep_indices]

    if confs.shape[0] < n_masks:
        confs = np.pad(confs, (0, n_masks - confs.shape[0]), constant_values=np.nan)
    elif confs.shape[0] > n_masks:
        confs = confs[:n_masks]

    if boxes.shape[0] < n_masks:
        pad_rows = n_masks - boxes.shape[0]
        boxes = np.vstack([boxes, np.full((pad_rows, 4), np.nan)]) if boxes.size else np.full((n_masks, 4), np.nan)
    elif boxes.shape[0] > n_masks:
        boxes = boxes[:n_masks]

    masks, confs, boxes, mask_meta = make_masks_non_overlapping(masks, confs, boxes, config, mask_meta)
    n_masks = len(masks)
    if n_masks == 0:
        return [], pd.DataFrame(columns=detections_columns)

    meta_df = pd.DataFrame(mask_meta)
    if len(meta_df) != n_masks:
        meta_df = pd.DataFrame(index=range(n_masks))
    defaults = {
        "mask_area_raw": np.nan,
        "mask_area_cleaned": np.nan,
        "mask_area_after_overlap": np.nan,
        "mask_area_loss_fraction": np.nan,
        "original_area": np.nan,
        "cleaned_area": np.nan,
        "area_loss_fraction": np.nan,
        "n_components_before": np.nan,
        "n_components_after": np.nan,
        "mask_fragment_after_overlap": False,
    }
    for col, default in defaults.items():
        if col not in meta_df.columns:
            meta_df[col] = default

    detections_df = pd.DataFrame(
        {
            "colony_id": np.arange(1, n_masks + 1, dtype=int),
            "yolo_conf": confs,
            "x1": boxes[:, 0],
            "y1": boxes[:, 1],
            "x2": boxes[:, 2],
            "y2": boxes[:, 3],
            "mask_area_raw": meta_df["mask_area_raw"].to_numpy(dtype=float),
            "mask_area_cleaned": meta_df["mask_area_cleaned"].to_numpy(dtype=float),
            "mask_area_after_overlap": meta_df["mask_area_after_overlap"].to_numpy(dtype=float),
            "mask_area_loss_fraction": meta_df["mask_area_loss_fraction"].to_numpy(dtype=float),
            "original_area": meta_df["original_area"].to_numpy(dtype=float),
            "cleaned_area": meta_df["cleaned_area"].to_numpy(dtype=float),
            "area_loss_fraction": meta_df["area_loss_fraction"].to_numpy(dtype=float),
            "n_components_before": meta_df["n_components_before"].to_numpy(dtype=float),
            "n_components_after": meta_df["n_components_after"].to_numpy(dtype=float),
            "mask_fragment_after_overlap": meta_df["mask_fragment_after_overlap"].fillna(False).astype(bool).to_numpy(),
        }
    )

    bbox_w = np.maximum(0.0, detections_df["x2"] - detections_df["x1"])
    bbox_h = np.maximum(0.0, detections_df["y2"] - detections_df["y1"])
    detections_df["bbox_area_yolo"] = bbox_w * bbox_h

    return masks, detections_df[detections_columns]

def normalize_masks_input(masks) -> list[np.ndarray]:
    if masks is None:
        return []

    if isinstance(masks, list):
        normalized = []
        for m in masks:
            arr = np.asarray(m)
            if arr.ndim != 2:
                continue
            arr_bool = arr.astype(bool)
            if arr_bool.any():
                normalized.append(arr_bool)
        return normalized

    arr = np.asarray(masks)

    if arr.ndim == 3:
        normalized = []
        for i in range(arr.shape[0]):
            mask_i = arr[i].astype(bool)
            if mask_i.any():
                normalized.append(mask_i)
        return normalized

    if arr.ndim == 2:
        if arr.dtype == bool:
            return [arr] if arr.any() else []

        unique_values = np.unique(arr)
        non_zero_ids = [int(v) for v in unique_values if v > 0]

        if len(non_zero_ids) == 0:
            return []

        if set(unique_values.tolist()).issubset({0, 1}):
            arr_bool = arr.astype(bool)
            return [arr_bool] if arr_bool.any() else []

        return [(arr == idx) for idx in sorted(non_zero_ids)]

    raise ValueError(
        "Неподдерживаемый формат masks. Ожидалось: list[np.ndarray], (N,H,W) или labeled mask (H,W)."
    )

def compute_local_background_ring(
    mask: np.ndarray,
    all_masks: list[np.ndarray],
    radius: int = 12,
) -> np.ndarray:
    mask = np.asarray(mask).astype(bool)
    if mask.ndim != 2:
        raise ValueError("mask должен иметь форму (H, W)")

    radius = max(1, int(radius))

    dilated = binary_dilation(mask, disk(radius))
    ring = dilated & ~mask

    other_objects = np.zeros_like(mask, dtype=bool)
    for other in normalize_masks_input(all_masks):
        if other.shape == mask.shape:
            other_objects |= other

    ring = ring & ~other_objects
    return ring.astype(bool)

def _safe_ratio(numerator: float, denominator: float, eps: float = 1e-8) -> float:
    if not np.isfinite(numerator) or not np.isfinite(denominator) or abs(denominator) <= eps:
        return np.nan
    return float(numerator / (denominator + eps))


def _hist_entropy_from_values(values: np.ndarray, bins: int = 32, value_range: tuple[float, float] = (0, 256)) -> float:
    values = np.asarray(values)
    if values.size == 0:
        return np.nan
    counts, _ = np.histogram(values, bins=bins, range=value_range)
    total = counts.sum()
    if total <= 0:
        return np.nan
    p = counts.astype(float) / float(total)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def _finite_mad(values: np.ndarray, eps: float = 1e-8) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    med = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values - med))
    return float(max(mad, eps))


def _robust_neighbor_delta(value: float, neighbor_values: np.ndarray, eps: float = 1e-8) -> float:
    if not np.isfinite(value):
        return np.nan
    neighbor_values = np.asarray(neighbor_values, dtype=float)
    neighbor_values = neighbor_values[np.isfinite(neighbor_values)]
    if neighbor_values.size == 0:
        return np.nan
    med = np.nanmedian(neighbor_values)
    mad = _finite_mad(neighbor_values, eps=eps)
    if not np.isfinite(med) or not np.isfinite(mad):
        return np.nan
    return float((value - med) / (1.4826 * mad + eps))


def _mask_contour_perimeter(mask_bool: np.ndarray) -> float:
    if mask_bool is None:
        return np.nan
    mask_bool = np.asarray(mask_bool).astype(bool)
    if mask_bool.ndim != 2 or not mask_bool.any():
        return np.nan
    if mask_bool.shape[0] < 2 or mask_bool.shape[1] < 2:
        return np.nan
    try:
        contours = find_contours(mask_bool.astype(float), level=0.5)
    except ValueError:
        return np.nan
    lengths: list[float] = []
    for contour in contours:
        if contour.shape[0] < 2:
            continue
        diffs = np.diff(contour, axis=0)
        lengths.append(float(np.sum(np.sqrt(np.sum(diffs * diffs, axis=1)))))
    return float(np.sum(lengths)) if lengths else np.nan


def _radial_contour_features(mask_bool: np.ndarray, centroid_x: float, centroid_y: float) -> tuple[float, float, float]:
    mask_arr = np.asarray(mask_bool).astype(float)
    if mask_arr.ndim != 2 or mask_arr.shape[0] < 2 or mask_arr.shape[1] < 2:
        return np.nan, np.nan, np.nan
    try:
        contours = find_contours(mask_arr, level=0.5)
    except ValueError:
        return np.nan, np.nan, np.nan
    if not contours:
        return np.nan, np.nan, np.nan
    points = np.vstack([c for c in contours if c.shape[0] >= 3]) if any(c.shape[0] >= 3 for c in contours) else np.empty((0, 2))
    if points.shape[0] < 3:
        return np.nan, np.nan, np.nan
    distances = np.sqrt((points[:, 1] - centroid_x) ** 2 + (points[:, 0] - centroid_y) ** 2)
    distances = distances[np.isfinite(distances)]
    if distances.size == 0:
        return np.nan, np.nan, np.nan
    mean_v = float(np.mean(distances))
    std_v = float(np.std(distances))
    cv_v = std_v / (mean_v + 1e-8) if mean_v > 0 else np.nan
    return mean_v, std_v, float(cv_v)


def _polygon_area(points: np.ndarray) -> float:
    if points is None or len(points) < 3:
        return np.nan
    x = points[:, 0]
    y = points[:, 1]
    return float(0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))))


def _compute_voronoi_areas(coords: np.ndarray, image_shape: tuple[int, int]) -> np.ndarray:
    coords = np.asarray(coords, dtype=float)
    n = len(coords)
    areas = np.full(n, np.nan, dtype=float)
    if n < 4:
        return areas
    try:
        from scipy.spatial import Voronoi

        vor = Voronoi(coords)
        h, w = image_shape
        max_reasonable_area = float(h * w)
        for i, region_idx in enumerate(vor.point_region):
            region = vor.regions[region_idx]
            if not region or any(v < 0 for v in region):
                continue
            vertices = vor.vertices[region]
            if vertices.size == 0:
                continue
            if np.any(vertices[:, 0] < 0) or np.any(vertices[:, 0] > w) or np.any(vertices[:, 1] < 0) or np.any(vertices[:, 1] > h):
                continue
            area = _polygon_area(vertices)
            if np.isfinite(area) and 0 <= area <= max_reasonable_area:
                areas[i] = area
    except Exception:
        pass
    return areas


def _add_feature_knn_distance(features_df: pd.DataFrame, config: AnomalyDetectionConfig) -> pd.DataFrame:
    """Средняя дистанция до ближайших соседей в пространстве признаков.

    Важно: расстояние считается только среди валидных объектов. Технические маски
    не должны становиться соседями и искажать локальную норму.
    """
    base_cols = [
        "log_area",
        "circularity",
        "eccentricity",
        "solidity",
        "intensity_mean",
        "intensity_iqr",
        "intensity_entropy",
        "glcm_contrast",
        "lbp_std",
        "local_color_delta_lab",
        "intensity_hist_js_to_plate_median",
    ]
    cols = [c for c in base_cols if c in features_df.columns and pd.api.types.is_numeric_dtype(features_df[c])]
    features_df["feature_knn_distance"] = np.nan
    if len(features_df) < 2 or not cols:
        return features_df

    valid_mask = pd.Series(True, index=features_df.index)
    if "valid_for_anomaly" in features_df.columns:
        valid_mask &= features_df["valid_for_anomaly"].fillna(False).astype(bool)
    if "technical_warning" in features_df.columns:
        valid_mask &= ~features_df["technical_warning"].fillna(False).astype(bool)
    valid_idx = np.where(valid_mask.to_numpy())[0]
    if valid_idx.size < 2:
        return features_df

    try:
        valid_df = features_df.iloc[valid_idx]
        X = valid_df[cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
        X = SimpleImputer(strategy="median").fit_transform(X)
        X = RobustScaler().fit_transform(X)
        tree = cKDTree(X)
        k = min(int(config.neighbor_k) + 1, len(valid_idx))
        dists, _ = tree.query(X, k=k)
        dists = np.atleast_2d(dists)
        distances = []
        for i in range(len(valid_idx)):
            row_dists = np.asarray(dists[i], dtype=float)
            row_dists = row_dists[np.isfinite(row_dists)]
            row_dists = row_dists[row_dists > 1e-12]
            distances.append(float(np.mean(row_dists[: max(1, k - 1)])) if row_dists.size else np.nan)
        features_df.loc[valid_idx, "feature_knn_distance"] = distances
    except Exception:
        features_df["feature_knn_distance"] = np.nan
    return features_df

def _build_technical_reason(row: pd.Series) -> str:
    reasons: list[str] = []
    checks = [
        ("is_too_small", "too_small"),
        ("is_too_large", "too_large"),
        ("touches_image_border", "touches_border"),
        ("suspicious_aspect_ratio", "suspicious_aspect_ratio"),
        ("low_yolo_conf", "low_yolo_conf"),
        ("critical_poor_local_background", "critical_poor_local_background"),
        ("mask_fragment_after_overlap", "mask_fragment_after_overlap"),
        ("invalid_geometry", "invalid_geometry"),
        ("possible_segmentation_artifact", "possible_segmentation_artifact"),
    ]
    for col, reason in checks:
        try:
            if bool(row.get(col, False)):
                reasons.append(reason)
        except Exception:
            pass
    return ";".join(reasons)



def _normalize_histogram(hist: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    hist = np.asarray(hist, dtype=float)
    hist = np.nan_to_num(hist, nan=0.0, posinf=0.0, neginf=0.0)
    hist = np.maximum(hist, 0.0)
    total = float(hist.sum())
    if total <= eps:
        if hist.size == 0:
            return hist
        return np.full(hist.shape, 1.0 / hist.size, dtype=float)
    return hist / total


def _safe_jensen_shannon(p: np.ndarray, q: np.ndarray) -> float:
    try:
        p = _normalize_histogram(p)
        q = _normalize_histogram(q)
        if p.size == 0 or q.size == 0 or p.size != q.size:
            return np.nan
        value = distance.jensenshannon(p, q, base=2.0)
        return float(value) if np.isfinite(value) else np.nan
    except Exception:
        return np.nan


def _safe_wasserstein_hist(p: np.ndarray, q: np.ndarray) -> float:
    try:
        p = _normalize_histogram(p)
        q = _normalize_histogram(q)
        if p.size == 0 or q.size == 0 or p.size != q.size:
            return np.nan
        x = np.arange(p.size, dtype=float)
        return float(stats.wasserstein_distance(x, x, u_weights=p, v_weights=q))
    except Exception:
        return np.nan


def _hist_width_fraction(hist: np.ndarray) -> float:
    hist = _normalize_histogram(hist)
    if hist.size == 0:
        return np.nan
    nz = np.where(hist > 0.01 * max(float(hist.max()), 1e-12))[0]
    if nz.size == 0:
        return 0.0
    return float((nz.max() - nz.min() + 1) / max(1, hist.size))


def _robust_z_value(value: float, values: np.ndarray, eps: float = 1e-8) -> float:
    try:
        value = float(value)
    except Exception:
        return np.nan
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 3 or not np.isfinite(value):
        return np.nan
    med = np.nanmedian(values)
    mad = _finite_mad(values, eps=eps)
    if not np.isfinite(mad) or mad <= 0:
        return np.nan
    return float((value - med) / (1.4826 * mad + eps))


def _compute_histograms_for_features(
    features_df: pd.DataFrame,
    image_rgb: np.ndarray,
    masks_by_colony_id: dict[int, np.ndarray],
    gray: np.ndarray,
    lab: np.ndarray,
    config: AnomalyDetectionConfig,
) -> tuple[pd.DataFrame, dict[int, dict[str, np.ndarray]]]:
    bins = int(max(4, getattr(config, "hist_bins", 32)))
    eps = 1e-8
    hist_store: dict[int, dict[str, np.ndarray]] = {}

    hist_cols = [
        "hist_intensity_entropy",
        "hist_intensity_peak_bin",
        "hist_intensity_peak_value",
        "hist_intensity_width",
        "hist_intensity_skewness",
        "hist_intensity_kurtosis",
        "dark_fraction",
        "bright_fraction",
        "contrast_hist_entropy",
        "contrast_dark_fraction",
        "contrast_bright_fraction",
        "intensity_hist_js_to_plate_median",
        "intensity_hist_wasserstein_to_plate_median",
        "intensity_hist_js_to_neighbors",
        "L_hist_js_to_plate_median",
        "L_hist_js_to_neighbors",
        "contrast_hist_js_to_plate_median",
        "relative_histogram_vs_neighbors",
    ]
    for c in hist_cols:
        if c not in features_df.columns:
            features_df[c] = np.nan

    if not bool(getattr(config, "use_histogram_features", True)) or features_df.empty:
        return features_df, hist_store

    all_valid_pixels: list[np.ndarray] = []
    for _, row in features_df.iterrows():
        if not bool(row.get("valid_for_anomaly", True)):
            continue
        mask = masks_by_colony_id.get(int(row["colony_id"]))
        if mask is None or not np.asarray(mask).any():
            continue
        vals = gray[mask].astype(float)
        if vals.size:
            all_valid_pixels.append(vals)
    if all_valid_pixels:
        valid_pixels = np.concatenate(all_valid_pixels)
        plate_p25 = float(np.nanpercentile(valid_pixels, 25))
        plate_p75 = float(np.nanpercentile(valid_pixels, 75))
    else:
        plate_p25, plate_p75 = 64.0, 192.0

    for idx, row in features_df.iterrows():
        cid = int(row["colony_id"])
        mask = masks_by_colony_id.get(cid)
        if mask is None or not np.asarray(mask).any():
            continue
        pix_gray = gray[mask].astype(float)
        pix_lab = lab[mask].astype(float)
        if pix_gray.size == 0:
            continue

        intensity_counts, _ = np.histogram(pix_gray, bins=bins, range=(0, 256))
        intensity_hist = _normalize_histogram(intensity_counts)
        L_counts, _ = np.histogram(pix_lab[:, 0], bins=bins, range=(0, 256))
        L_hist = _normalize_histogram(L_counts)
        contrast_center = row.get("local_background_intensity_mean", np.nan)
        if not np.isfinite(contrast_center):
            contrast_center = np.nanmedian(pix_gray)
        contrast_values = np.clip(pix_gray - float(contrast_center) + 128.0, 0, 255)
        contrast_counts, _ = np.histogram(contrast_values, bins=bins, range=(0, 256))
        contrast_hist = _normalize_histogram(contrast_counts)

        hist_store[cid] = {
            "intensity": intensity_hist,
            "L": L_hist,
            "contrast": contrast_hist,
        }
        features_df.at[idx, "hist_intensity_entropy"] = _hist_entropy_from_values(pix_gray, bins=bins, value_range=(0, 256))
        features_df.at[idx, "hist_intensity_peak_bin"] = int(np.argmax(intensity_hist))
        features_df.at[idx, "hist_intensity_peak_value"] = float(np.max(intensity_hist))
        features_df.at[idx, "hist_intensity_width"] = _hist_width_fraction(intensity_hist)
        features_df.at[idx, "hist_intensity_skewness"] = float(stats.skew(pix_gray, nan_policy="omit")) if pix_gray.size >= 3 else np.nan
        features_df.at[idx, "hist_intensity_kurtosis"] = float(stats.kurtosis(pix_gray, nan_policy="omit")) if pix_gray.size >= 4 else np.nan
        features_df.at[idx, "dark_fraction"] = float(np.mean(pix_gray < plate_p25))
        features_df.at[idx, "bright_fraction"] = float(np.mean(pix_gray > plate_p75))
        features_df.at[idx, "contrast_hist_entropy"] = _hist_entropy_from_values(contrast_values, bins=bins, value_range=(0, 256))
        features_df.at[idx, "contrast_dark_fraction"] = float(np.mean(contrast_values < 96))
        features_df.at[idx, "contrast_bright_fraction"] = float(np.mean(contrast_values > 160))

    valid_ids = features_df.loc[features_df.get("valid_for_anomaly", True).astype(bool), "colony_id"].astype(int).tolist() if "valid_for_anomaly" in features_df.columns else features_df["colony_id"].astype(int).tolist()
    valid_ids = [cid for cid in valid_ids if cid in hist_store]
    if not valid_ids:
        return features_df, hist_store

    median_hists: dict[str, np.ndarray] = {}
    for key in ["intensity", "L", "contrast"]:
        stack = np.vstack([hist_store[cid][key] for cid in valid_ids])
        median_hists[key] = _normalize_histogram(np.nanmedian(stack, axis=0))

    coords = features_df[["centroid_x", "centroid_y"]].to_numpy(dtype=float) if {"centroid_x", "centroid_y"}.issubset(features_df.columns) else None
    tree = cKDTree(coords) if coords is not None and len(features_df) >= 2 else None

    for idx, row in features_df.iterrows():
        cid = int(row["colony_id"])
        if cid not in hist_store:
            continue
        features_df.at[idx, "intensity_hist_js_to_plate_median"] = _safe_jensen_shannon(hist_store[cid]["intensity"], median_hists["intensity"])
        features_df.at[idx, "intensity_hist_wasserstein_to_plate_median"] = _safe_wasserstein_hist(hist_store[cid]["intensity"], median_hists["intensity"])
        features_df.at[idx, "L_hist_js_to_plate_median"] = _safe_jensen_shannon(hist_store[cid]["L"], median_hists["L"])
        features_df.at[idx, "contrast_hist_js_to_plate_median"] = _safe_jensen_shannon(hist_store[cid]["contrast"], median_hists["contrast"])

        if tree is not None:
            k = min(int(config.neighbor_k) + 1, len(features_df))
            _, idxs = tree.query(coords[idx], k=k)
            idxs = np.atleast_1d(idxs).astype(int)
            neighbor_ids = [int(features_df.iloc[j]["colony_id"]) for j in idxs if j != idx and int(features_df.iloc[j]["colony_id"]) in hist_store]
            if neighbor_ids:
                nn_intensity = _normalize_histogram(np.nanmedian(np.vstack([hist_store[nid]["intensity"] for nid in neighbor_ids]), axis=0))
                nn_L = _normalize_histogram(np.nanmedian(np.vstack([hist_store[nid]["L"] for nid in neighbor_ids]), axis=0))
                features_df.at[idx, "intensity_hist_js_to_neighbors"] = _safe_jensen_shannon(hist_store[cid]["intensity"], nn_intensity)
                features_df.at[idx, "L_hist_js_to_neighbors"] = _safe_jensen_shannon(hist_store[cid]["L"], nn_L)

    if "intensity_hist_js_to_neighbors" in features_df.columns:
        vals = features_df["intensity_hist_js_to_neighbors"].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
        med = np.nanmedian(vals) if np.isfinite(vals).any() else np.nan
        mad = _finite_mad(vals)
        if np.isfinite(med) and np.isfinite(mad):
            rel = (vals - med) / (1.4826 * mad + eps)
            features_df["relative_histogram_vs_neighbors"] = np.clip(rel, -float(config.clip_relative_features), float(config.clip_relative_features))

    return features_df, hist_store


def _recompute_neighbor_relative_features(features_df: pd.DataFrame, config: AnomalyDetectionConfig) -> pd.DataFrame:
    """Локальные relative-признаки относительно ближайших валидных соседей.

    Нормирование выполняется через глобальную MAD по валидным объектам, а не через
    MAD только нескольких соседей. Это защищает от искусственно огромных значений,
    когда у ближайших соседей почти нулевой разброс.
    """
    if len(features_df) < 2 or not {"centroid_x", "centroid_y"}.issubset(features_df.columns):
        return features_df

    valid_mask = pd.Series(True, index=features_df.index)
    if "valid_for_anomaly" in features_df.columns:
        valid_mask &= features_df["valid_for_anomaly"].fillna(False).astype(bool)
    if "technical_warning" in features_df.columns:
        valid_mask &= ~features_df["technical_warning"].fillna(False).astype(bool)
    valid_idx = np.where(valid_mask.to_numpy())[0]
    if valid_idx.size < 2:
        return features_df

    coords_valid = features_df.iloc[valid_idx][["centroid_x", "centroid_y"]].to_numpy(dtype=float)
    tree = cKDTree(coords_valid)
    feature_map = {
        "relative_area_vs_neighbors": "area",
        "relative_intensity_vs_neighbors": "intensity_mean",
        "relative_texture_vs_neighbors": "glcm_contrast",
        "relative_circularity_vs_neighbors": "circularity",
        "relative_solidity_vs_neighbors": "solidity",
        "relative_color_delta_lab_vs_neighbors": "local_color_delta_lab",
        "relative_local_contrast_vs_neighbors": "local_contrast",
        "relative_entropy_vs_neighbors": "intensity_entropy",
        "relative_histogram_vs_neighbors": "intensity_hist_js_to_neighbors",
    }
    clip = float(getattr(config, "clip_relative_features", 5.0))
    for out_col, source_col in feature_map.items():
        if source_col not in features_df.columns:
            continue
        valid_values = features_df.loc[valid_idx, source_col].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
        global_mad = _finite_mad(valid_values)
        if not np.isfinite(global_mad) or global_mad <= 0:
            continue
        for pos, global_i in enumerate(valid_idx):
            k = min(int(config.neighbor_k) + 1, len(valid_idx))
            _, local_idxs = tree.query(coords_valid[pos], k=k)
            local_idxs = np.atleast_1d(local_idxs).astype(int)
            nn_global = [valid_idx[j] for j in local_idxs if valid_idx[j] != global_i]
            if not nn_global:
                continue
            cur = float(features_df.at[global_i, source_col])
            nn_values = features_df.loc[nn_global, source_col].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
            nn_values = nn_values[np.isfinite(nn_values)]
            if not np.isfinite(cur) or nn_values.size == 0:
                continue
            med_nn = np.nanmedian(nn_values)
            value = (cur - med_nn) / (1.4826 * global_mad + 1e-8)
            features_df.at[global_i, out_col] = float(np.clip(value, -clip, clip))
    return features_df


def _augment_features_after_extraction(
    features_df: pd.DataFrame,
    image_rgb: np.ndarray,
    masks_by_colony_id: dict[int, np.ndarray],
    gray: np.ndarray,
    lab: np.ndarray,
    config: AnomalyDetectionConfig,
    image_area: float,
) -> pd.DataFrame:
    if features_df.empty:
        return features_df
    eps = 1e-8
    plate_area_median = np.nanmedian(features_df["area"].to_numpy(dtype=float))
    plate_equiv_median = np.nanmedian(features_df["equivalent_diameter"].to_numpy(dtype=float))
    if not np.isfinite(plate_equiv_median) or plate_equiv_median <= eps:
        plate_equiv_median = np.nan

    features_df["log_area"] = np.log1p(features_df["area"].astype(float))
    features_df["area_to_median_ratio"] = features_df["area"].astype(float) / (plate_area_median + eps) if np.isfinite(plate_area_median) else np.nan
    features_df["equivalent_diameter_to_median_ratio"] = features_df["equivalent_diameter"].astype(float) / (plate_equiv_median + eps) if np.isfinite(plate_equiv_median) else np.nan
    features_df["size_global_z"] = [
        _robust_z_value(v, features_df["log_area"].to_numpy(dtype=float)) for v in features_df["log_area"].to_numpy(dtype=float)
    ]

    features_df["possible_merged_colony"] = (
        (features_df["aspect_ratio"].astype(float) > 2.2)
        | ((features_df["eccentricity"].astype(float) > 0.88) & (features_df["circularity"].astype(float) < 0.70))
        | ((features_df["area_to_median_ratio"].astype(float) > 2.0) & (features_df["convexity_defect_ratio"].astype(float) > 0.15))
    )
    features_df["possible_merged_colony_score"] = features_df["possible_merged_colony"].astype(float)

    if len(features_df) >= 2:
        coords = features_df[["centroid_x", "centroid_y"]].to_numpy(dtype=float)
        radii = np.sqrt(np.maximum(features_df["area"].astype(float).to_numpy(), 0.0) / math.pi)
        tree = cKDTree(coords)
        edge_nearest = np.full(len(features_df), np.nan, dtype=float)
        mean_edge = np.full(len(features_df), np.nan, dtype=float)
        size_vs_neighbors = np.full(len(features_df), np.nan, dtype=float)
        for i in range(len(features_df)):
            k = min(int(config.neighbor_k) + 1, len(features_df))
            dists, idxs = tree.query(coords[i], k=k)
            dists = np.atleast_1d(dists).astype(float)
            idxs = np.atleast_1d(idxs).astype(int)
            keep = idxs != i
            nd = dists[keep]
            ni = idxs[keep]
            if nd.size:
                edge_d = nd - radii[i] - radii[ni]
                edge_nearest[i] = float(edge_d[0])
                mean_edge[i] = float(np.mean(edge_d[:5]))
                neighbor_log_area = features_df.loc[ni[: int(config.neighbor_k)], "log_area"].to_numpy(dtype=float)
                size_vs_neighbors[i] = _robust_neighbor_delta(float(features_df.at[i, "log_area"]), neighbor_log_area)
        features_df["edge_nearest_distance"] = edge_nearest
        features_df["mean_5nn_edge_distance"] = mean_edge
        features_df["size_vs_neighbors_z"] = np.clip(size_vs_neighbors, -float(config.clip_relative_features), float(config.clip_relative_features))
    else:
        features_df["edge_nearest_distance"] = np.nan
        features_df["mean_5nn_edge_distance"] = np.nan
        features_df["size_vs_neighbors_z"] = np.nan

    features_df["nearest_distance_to_median_diameter_ratio"] = features_df["nearest_neighbor_distance"].astype(float) / (plate_equiv_median + eps) if np.isfinite(plate_equiv_median) else np.nan
    features_df["edge_distance_to_median_diameter_ratio"] = features_df["edge_nearest_distance"].astype(float) / (plate_equiv_median + eps) if np.isfinite(plate_equiv_median) else np.nan

    # Update technical flags after derived mask/shape diagnostics.
    features_df["possible_segmentation_artifact"] = (
        features_df.get("mask_fragment_after_overlap", False).astype(bool)
        | features_df.get("invalid_geometry", False).astype(bool)
        | (features_df["possible_merged_colony"].astype(bool) & (features_df.get("mask_area_loss_fraction", 0).fillna(0).astype(float) > 0.30))
    )

    min_area_threshold = float(config.min_area)
    if np.isfinite(plate_area_median):
        min_area_threshold = max(min_area_threshold, float(plate_area_median) * float(config.min_area_relative_to_median))
    max_area_threshold = float(config.max_area_fraction) * image_area
    if np.isfinite(plate_area_median) and plate_area_median > 0:
        max_area_threshold = min(max_area_threshold, float(plate_area_median) * float(config.max_area_relative_to_median))
    features_df["is_too_small"] = features_df["area"].astype(float) < min_area_threshold
    features_df["is_too_large"] = features_df["area"].astype(float) > max_area_threshold
    if "yolo_conf" not in features_df.columns:
        features_df["yolo_conf"] = np.nan
    features_df["low_yolo_conf"] = features_df["yolo_conf"].fillna(1.0).astype(float) < float(config.min_yolo_conf_for_analysis)

    technical_flags = (
        features_df["is_too_small"].fillna(True).astype(bool)
        | features_df["is_too_large"].fillna(True).astype(bool)
        | features_df.get("suspicious_aspect_ratio", False).fillna(False).astype(bool)
        | features_df["low_yolo_conf"].fillna(False).astype(bool)
        | features_df.get("critical_poor_local_background", False).fillna(False).astype(bool)
        | features_df.get("mask_fragment_after_overlap", False).fillna(False).astype(bool)
        | features_df.get("invalid_geometry", False).fillna(False).astype(bool)
        | features_df["possible_segmentation_artifact"].fillna(False).astype(bool)
    )
    if config.exclude_border_touching_from_anomaly:
        technical_flags = technical_flags | features_df.get("touches_image_border", False).fillna(False).astype(bool)
    features_df["technical_warning"] = technical_flags.astype(bool)
    features_df["valid_for_anomaly"] = ~features_df["technical_warning"].astype(bool)

    features_df, hist_store = _compute_histograms_for_features(features_df, image_rgb, masks_by_colony_id, gray, lab, config)
    features_df = _recompute_neighbor_relative_features(features_df, config)
    features_df = _add_feature_knn_distance(features_df, config)
    features_df["technical_reason"] = features_df.apply(_build_technical_reason, axis=1)
    features_df.loc[~features_df["technical_warning"].astype(bool), "technical_reason"] = ""
    return features_df


def extract_colony_features(
    image_rgb: np.ndarray,
    masks,
    detections_df: pd.DataFrame | None = None,
    plate_mask: np.ndarray | None = None,
    config: AnomalyDetectionConfig | None = None,
) -> pd.DataFrame:
    if config is None:
        config = AnomalyDetectionConfig()

    image_rgb = np.asarray(image_rgb)
    if image_rgb.ndim != 3 or image_rgb.shape[2] != 3:
        raise ValueError("Ожидалось RGB-изображение формы (H, W, 3)")

    h, w = image_rgb.shape[:2]
    masks_list = normalize_masks_input(masks)

    prepared_masks: list[tuple[int, np.ndarray, dict[str, Any]]] = []
    for source_colony_id, m in enumerate(masks_list, start=1):
        m_bool = np.asarray(m).astype(bool)
        if m_bool.shape != (h, w):
            m_bool = cv2.resize(
                m_bool.astype(np.uint8),
                (w, h),
                interpolation=cv2.INTER_NEAREST,
            ).astype(bool)
        raw_area = float(m_bool.sum())
        cleaned, diag = clean_instance_mask(m_bool, config)
        cleaned_area = float(cleaned.sum())
        loss_fraction = float(diag.get("area_loss_fraction", 1.0 - cleaned_area / max(raw_area, 1.0)))
        mask_meta = {
            "mask_area_input": raw_area,
            "mask_area_after_feature_cleaning": cleaned_area,
            "mask_area_feature_loss_fraction": float(np.clip(loss_fraction, 0.0, 1.0)),
            "original_area": float(diag.get("original_area", raw_area)),
            "cleaned_area": float(diag.get("cleaned_area", cleaned_area)),
            "area_loss_fraction": float(diag.get("area_loss_fraction", loss_fraction)),
            "n_components_before": int(diag.get("n_components_before", 0)),
            "n_components_after": int(diag.get("n_components_after", 0)),
            "mask_fragment_after_overlap": bool(diag.get("mask_fragment_after_overlap", False)),
        }
        if cleaned.any():
            feature_mask = cleaned.astype(bool)
        else:
            # Preserve a fully discarded fragment as a technical warning row.
            # Otherwise it disappears from technical_warnings.csv and quality control.
            feature_mask = m_bool.astype(bool)
            mask_meta["invalid_geometry"] = True
            mask_meta["mask_fragment_after_overlap"] = True
        if feature_mask.any():
            prepared_masks.append((source_colony_id, feature_mask, mask_meta))

    if len(prepared_masks) == 0:
        return pd.DataFrame()
    analysis_masks = [m for _, m, _ in prepared_masks]
    masks_by_colony_id = {int(cid): m for cid, m, _ in prepared_masks}

    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)

    eps = 1e-8
    image_area = float(h * w)

    if plate_mask is None:
        plate_mask_bool = np.ones((h, w), dtype=bool)
        plate_center_x = (w - 1) / 2.0
        plate_center_y = (h - 1) / 2.0
        plate_dist_map = None
    else:
        plate_mask_bool = np.asarray(plate_mask).astype(bool)
        if plate_mask_bool.shape != (h, w):
            plate_mask_bool = cv2.resize(
                plate_mask_bool.astype(np.uint8),
                (w, h),
                interpolation=cv2.INTER_NEAREST,
            ).astype(bool)
        if not plate_mask_bool.any():
            plate_mask_bool = np.ones((h, w), dtype=bool)

        ys_pm, xs_pm = np.where(plate_mask_bool)
        if len(xs_pm) > 0:
            plate_center_x = float(np.mean(xs_pm))
            plate_center_y = float(np.mean(ys_pm))
        else:
            plate_center_x = (w - 1) / 2.0
            plate_center_y = (h - 1) / 2.0

        plate_dist_map = cv2.distanceTransform(
            plate_mask_bool.astype(np.uint8), cv2.DIST_L2, 3
        )

    detections_lookup: dict[int, dict[str, float]] = {}
    if detections_df is not None and not detections_df.empty and "colony_id" in detections_df.columns:
        for _, row in detections_df.iterrows():
            colony_id = int(row["colony_id"])
            detections_lookup[colony_id] = row.to_dict()

    rows: list[dict[str, Any]] = []

    for colony_id, mask, mask_meta in prepared_masks:
        labeled_mask = label(mask.astype(np.uint8))
        props = regionprops(labeled_mask)
        if not props:
            continue

        prop = max(props, key=lambda p: p.area)
        area = float(prop.area)
        if area <= 0:
            continue

        perimeter = float(prop.perimeter)
        equivalent_diameter = float(prop.equivalent_diameter)

        min_row, min_col, max_row, max_col = prop.bbox
        bbox_width = float(max_col - min_col)
        bbox_height = float(max_row - min_row)
        aspect_ratio = bbox_width / bbox_height if bbox_height > 0 else np.nan
        circularity = (
            4.0 * math.pi * area / (perimeter**2) if perimeter > 0 else np.nan
        )

        eccentricity = float(getattr(prop, "eccentricity", np.nan))
        solidity = float(getattr(prop, "solidity", np.nan))
        extent = float(getattr(prop, "extent", np.nan))
        major_axis_length = float(getattr(prop, "major_axis_length", np.nan))
        minor_axis_length = float(getattr(prop, "minor_axis_length", np.nan))
        orientation = float(getattr(prop, "orientation", np.nan))
        convex_area = float(getattr(prop, "convex_area", np.nan))
        convexity_defect_area = convex_area - area if np.isfinite(convex_area) else np.nan
        convexity_defect_ratio = _safe_ratio(convexity_defect_area, area, eps=eps)
        convex_perimeter = _mask_contour_perimeter(getattr(prop, "image_convex", None))
        boundary_roughness = _safe_ratio(perimeter, convex_perimeter, eps=eps)

        centroid_y, centroid_x = prop.centroid
        centroid_x = float(centroid_x)
        centroid_y = float(centroid_y)
        centroid_x_norm = centroid_x / max(1.0, float(w - 1))
        centroid_y_norm = centroid_y / max(1.0, float(h - 1))

        distance_to_plate_center = float(
            np.hypot(centroid_x - plate_center_x, centroid_y - plate_center_y)
        )

        if plate_dist_map is None:
            distance_to_plate_edge = float(
                min(
                    centroid_x,
                    centroid_y,
                    (w - 1) - centroid_x,
                    (h - 1) - centroid_y,
                )
            )
        else:
            cyi = int(np.clip(round(centroid_y), 0, h - 1))
            cxi = int(np.clip(round(centroid_x), 0, w - 1))
            distance_to_plate_edge = float(plate_dist_map[cyi, cxi])
            if not np.isfinite(distance_to_plate_edge) or distance_to_plate_edge <= 0:
                distance_to_plate_edge = float(
                    min(
                        centroid_x,
                        centroid_y,
                        (w - 1) - centroid_x,
                        (h - 1) - centroid_y,
                    )
                )

        radial_contour_mean, radial_contour_std, radial_contour_cv = _radial_contour_features(
            mask, centroid_x=centroid_x, centroid_y=centroid_y
        )

        pix_rgb = image_rgb[mask]
        pix_gray = gray[mask]
        pix_hsv = hsv[mask]
        pix_lab = lab[mask]

        if pix_gray.size == 0:
            continue

        mean_R, mean_G, mean_B = [float(v) for v in np.mean(pix_rgb, axis=0)]
        std_R, std_G, std_B = [float(v) for v in np.std(pix_rgb, axis=0)]

        mean_H, mean_S, mean_V = [float(v) for v in np.mean(pix_hsv, axis=0)]
        std_H, std_S, std_V = [float(v) for v in np.std(pix_hsv, axis=0)]
        hue_rad = pix_hsv[:, 0].astype(float) / 180.0 * 2.0 * np.pi
        mean_H_sin = float(np.mean(np.sin(hue_rad))) if hue_rad.size else np.nan
        mean_H_cos = float(np.mean(np.cos(hue_rad))) if hue_rad.size else np.nan

        mean_L_lab, mean_a_lab, mean_b_lab = [float(v) for v in np.mean(pix_lab, axis=0)]
        std_L_lab, std_a_lab, std_b_lab = [float(v) for v in np.std(pix_lab, axis=0)]

        intensity_mean = float(np.mean(pix_gray))
        intensity_median = float(np.median(pix_gray))
        intensity_std = float(np.std(pix_gray))
        intensity_min = float(np.min(pix_gray))
        intensity_max = float(np.max(pix_gray))
        intensity_range = intensity_max - intensity_min
        intensity_cv = intensity_std / (abs(intensity_mean) + eps)
        intensity_p05, intensity_p25, intensity_p75, intensity_p95 = [
            float(v) for v in np.percentile(pix_gray, [5, 25, 75, 95])
        ]
        intensity_iqr = intensity_p75 - intensity_p25
        intensity_p95_p05_range = intensity_p95 - intensity_p05
        intensity_entropy = _hist_entropy_from_values(pix_gray, bins=32, value_range=(0, 256))

        ys_mask, xs_mask = np.where(mask)
        radial_distances = np.sqrt((xs_mask.astype(float) - centroid_x) ** 2 + (ys_mask.astype(float) - centroid_y) ** 2)
        max_radius = float(np.max(radial_distances)) if radial_distances.size else np.nan
        if radial_distances.size >= 3 and np.isfinite(max_radius) and max_radius > eps:
            normalized_radius = radial_distances / max_radius
            pix_gray_float = pix_gray.astype(float)
            center_vals = pix_gray_float[normalized_radius <= 0.35]
            rim_vals = pix_gray_float[normalized_radius >= 0.70]
            center_intensity_mean = float(np.mean(center_vals)) if center_vals.size else np.nan
            rim_intensity_mean = float(np.mean(rim_vals)) if rim_vals.size else np.nan
            center_rim_intensity_delta = center_intensity_mean - rim_intensity_mean if np.isfinite(center_intensity_mean) and np.isfinite(rim_intensity_mean) else np.nan
            center_rim_intensity_ratio = _safe_ratio(center_intensity_mean, rim_intensity_mean, eps=eps)
            radial_intensity_std = float(np.std(pix_gray_float)) if pix_gray_float.size else np.nan
            try:
                radial_intensity_slope = float(np.polyfit(normalized_radius, pix_gray_float, deg=1)[0])
            except Exception:
                radial_intensity_slope = np.nan
        else:
            center_intensity_mean = np.nan
            rim_intensity_mean = np.nan
            center_rim_intensity_delta = np.nan
            center_rim_intensity_ratio = np.nan
            radial_intensity_slope = np.nan
            radial_intensity_std = np.nan

        y1, y2 = int(min_row), int(max_row)
        x1, x2 = int(min_col), int(max_col)
        gray_crop = gray[y1:y2, x1:x2].copy()
        mask_crop = mask[y1:y2, x1:x2]

        if gray_crop.size == 0 or not mask_crop.any():
            laplacian_var = np.nan
            glcm_contrast = np.nan
            glcm_homogeneity = np.nan
            glcm_energy = np.nan
            glcm_correlation = np.nan
            glcm_dissimilarity = np.nan
            glcm_ASM = np.nan
            lbp_mean = np.nan
            lbp_std = np.nan
            lbp_entropy = np.nan
        else:
            inside_vals = gray_crop[mask_crop]
            fill_val = float(np.median(inside_vals)) if inside_vals.size else float(np.median(gray_crop))
            gray_crop_filled = gray_crop.astype(np.float32)
            gray_crop_filled[~mask_crop] = fill_val
            gray_crop_u8 = np.clip(gray_crop_filled, 0, 255).astype(np.uint8)

            laplacian_var = float(cv2.Laplacian(gray_crop_u8, cv2.CV_32F).var())

            try:
                glcm_levels = int(max(2, min(256, config.glcm_levels)))
                gray_crop_q = np.floor(gray_crop_u8.astype(np.float32) * glcm_levels / 256.0).astype(np.uint8)
                gray_crop_q = np.clip(gray_crop_q, 0, glcm_levels - 1)
                glcm = graycomatrix(
                    gray_crop_q,
                    distances=list(config.glcm_distances),
                    angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                    levels=glcm_levels,
                    symmetric=True,
                    normed=True,
                )
                glcm_contrast = float(np.mean(graycoprops(glcm, "contrast")))
                glcm_homogeneity = float(np.mean(graycoprops(glcm, "homogeneity")))
                glcm_energy = float(np.mean(graycoprops(glcm, "energy")))
                glcm_correlation = float(np.mean(graycoprops(glcm, "correlation")))
                glcm_dissimilarity = float(np.mean(graycoprops(glcm, "dissimilarity")))
                glcm_ASM = float(np.mean(graycoprops(glcm, "ASM")))
            except Exception:
                glcm_contrast = np.nan
                glcm_homogeneity = np.nan
                glcm_energy = np.nan
                glcm_correlation = np.nan
                glcm_dissimilarity = np.nan
                glcm_ASM = np.nan

            try:
                lbp = local_binary_pattern(gray_crop_u8, P=config.lbp_p, R=config.lbp_r, method="uniform")
                lbp_vals = lbp[mask_crop] if mask_crop.any() else lbp.ravel()
                lbp_mean = float(np.mean(lbp_vals)) if lbp_vals.size else np.nan
                lbp_std = float(np.std(lbp_vals)) if lbp_vals.size else np.nan
                lbp_entropy = _hist_entropy_from_values(lbp_vals, bins=int(config.lbp_p) + 2, value_range=(0, int(config.lbp_p) + 2)) if lbp_vals.size else np.nan
            except Exception:
                lbp_mean = np.nan
                lbp_std = np.nan
                lbp_entropy = np.nan

        raw_background_ring = binary_dilation(mask, disk(config.local_background_radius)) & (~mask)
        ring = compute_local_background_ring(
            mask=mask,
            all_masks=analysis_masks,
            radius=config.local_background_radius,
        )
        local_background_area = int(ring.sum())
        raw_background_area = int(raw_background_ring.sum())
        local_background_valid_fraction = _safe_ratio(local_background_area, raw_background_area, eps=eps)

        if ring.any():
            bg_gray = gray[ring]
            bg_rgb = image_rgb[ring]
            bg_lab = lab[ring]

            local_background_intensity_mean = float(np.mean(bg_gray))
            local_background_intensity_std = float(np.std(bg_gray))
            local_contrast = float(abs(intensity_mean - local_background_intensity_mean))

            mean_obj_rgb = np.mean(pix_rgb, axis=0)
            mean_bg_rgb = np.mean(bg_rgb, axis=0)
            local_color_delta_rgb = float(np.linalg.norm(mean_obj_rgb - mean_bg_rgb))

            local_background_L_mean, local_background_a_mean, local_background_b_mean = [float(v) for v in np.mean(bg_lab, axis=0)]
            local_delta_L = mean_L_lab - local_background_L_mean
            local_delta_a = mean_a_lab - local_background_a_mean
            local_delta_b = mean_b_lab - local_background_b_mean
            local_color_delta_lab = float(np.sqrt(local_delta_L**2 + local_delta_a**2 + local_delta_b**2))
        else:
            local_background_intensity_mean = np.nan
            local_background_intensity_std = np.nan
            local_contrast = np.nan
            local_color_delta_rgb = np.nan
            local_background_L_mean = np.nan
            local_background_a_mean = np.nan
            local_background_b_mean = np.nan
            local_delta_L = np.nan
            local_delta_a = np.nan
            local_delta_b = np.nan
            local_color_delta_lab = np.nan

        poor_local_background = bool(local_background_area < max(20, 0.1 * area))
        critical_poor_local_background = bool(local_background_area < 5)

        edge_margin = max(0, int(config.edge_margin_px))
        if edge_margin == 0:
            touches_image_border = bool(mask[0, :].any() or mask[-1, :].any() or mask[:, 0].any() or mask[:, -1].any())
        else:
            touches_image_border = bool(
                mask[:edge_margin, :].any()
                or mask[-edge_margin:, :].any()
                or mask[:, :edge_margin].any()
                or mask[:, -edge_margin:].any()
            )
        is_too_small = bool(area < config.min_area)
        is_too_large = bool(area > config.max_area_fraction * image_area)

        if np.isfinite(aspect_ratio) and aspect_ratio > 0:
            inv_ar = 1.0 / max(aspect_ratio, eps)
            suspicious_aspect_ratio = bool(
                max(aspect_ratio, inv_ar) > config.suspicious_aspect_ratio_threshold
            )
        else:
            suspicious_aspect_ratio = False

        technical_warning = bool(
            touches_image_border
            or is_too_small
            or is_too_large
            or suspicious_aspect_ratio
        )

        row = {
            "colony_id": int(colony_id),
            "area": area,
            "perimeter": perimeter,
            "equivalent_diameter": equivalent_diameter,
            "bbox_width": bbox_width,
            "bbox_height": bbox_height,
            "aspect_ratio": aspect_ratio,
            "circularity": circularity,
            "eccentricity": eccentricity,
            "solidity": solidity,
            "extent": extent,
            "major_axis_length": major_axis_length,
            "minor_axis_length": minor_axis_length,
            "orientation": orientation,
            "convex_area": convex_area,
            "convexity_defect_area": convexity_defect_area,
            "convexity_defect_ratio": convexity_defect_ratio,
            "convex_perimeter": convex_perimeter,
            "boundary_roughness": boundary_roughness,
            "radial_contour_mean": radial_contour_mean,
            "radial_contour_std": radial_contour_std,
            "radial_contour_cv": radial_contour_cv,
            "mean_R": mean_R,
            "mean_G": mean_G,
            "mean_B": mean_B,
            "std_R": std_R,
            "std_G": std_G,
            "std_B": std_B,
            "mean_H": mean_H,
            "mean_H_sin": mean_H_sin,
            "mean_H_cos": mean_H_cos,
            "mean_S": mean_S,
            "mean_V": mean_V,
            "std_H": std_H,
            "std_S": std_S,
            "std_V": std_V,
            "mean_L_lab": mean_L_lab,
            "mean_a_lab": mean_a_lab,
            "mean_b_lab": mean_b_lab,
            "std_L_lab": std_L_lab,
            "std_a_lab": std_a_lab,
            "std_b_lab": std_b_lab,
            "intensity_mean": intensity_mean,
            "intensity_median": intensity_median,
            "intensity_std": intensity_std,
            "intensity_min": intensity_min,
            "intensity_max": intensity_max,
            "intensity_range": intensity_range,
            "intensity_cv": intensity_cv,
            "intensity_p05": intensity_p05,
            "intensity_p25": intensity_p25,
            "intensity_p75": intensity_p75,
            "intensity_p95": intensity_p95,
            "intensity_iqr": intensity_iqr,
            "intensity_p95_p05_range": intensity_p95_p05_range,
            "intensity_entropy": intensity_entropy,
            "center_intensity_mean": center_intensity_mean,
            "rim_intensity_mean": rim_intensity_mean,
            "center_rim_intensity_delta": center_rim_intensity_delta,
            "center_rim_intensity_ratio": center_rim_intensity_ratio,
            "radial_intensity_slope": radial_intensity_slope,
            "radial_intensity_std": radial_intensity_std,
            "laplacian_var": laplacian_var,
            "glcm_contrast": glcm_contrast,
            "glcm_homogeneity": glcm_homogeneity,
            "glcm_energy": glcm_energy,
            "glcm_correlation": glcm_correlation,
            "glcm_dissimilarity": glcm_dissimilarity,
            "glcm_ASM": glcm_ASM,
            "lbp_mean": lbp_mean,
            "lbp_std": lbp_std,
            "lbp_entropy": lbp_entropy,
            "centroid_x": centroid_x,
            "centroid_y": centroid_y,
            "centroid_x_norm": centroid_x_norm,
            "centroid_y_norm": centroid_y_norm,
            "distance_to_plate_center": distance_to_plate_center,
            "distance_to_plate_edge": distance_to_plate_edge,
            "nearest_neighbor_distance": np.nan,
            "mean_5nn_distance": np.nan,
            "log_nearest_neighbor_distance": np.nan,
            "log_mean_5nn_distance": np.nan,
            "edge_nearest_distance": np.nan,
            "mean_5nn_edge_distance": np.nan,
            "nearest_distance_to_median_diameter_ratio": np.nan,
            "edge_distance_to_median_diameter_ratio": np.nan,
            "local_density_r": np.nan,
            "voronoi_area": np.nan,
            "local_background_intensity_mean": local_background_intensity_mean,
            "local_background_intensity_std": local_background_intensity_std,
            "local_background_area": local_background_area,
            "local_background_valid_fraction": local_background_valid_fraction,
            "local_background_L_mean": local_background_L_mean,
            "local_background_a_mean": local_background_a_mean,
            "local_background_b_mean": local_background_b_mean,
            "local_delta_L": local_delta_L,
            "local_delta_a": local_delta_a,
            "local_delta_b": local_delta_b,
            "local_contrast": local_contrast,
            "local_color_delta_rgb": local_color_delta_rgb,
            "local_color_delta_lab": local_color_delta_lab,
            "relative_area_vs_plate_median": np.nan,
            "relative_intensity_vs_plate_median": np.nan,
            "relative_texture_vs_plate_median": np.nan,
            "relative_area_vs_neighbors": np.nan,
            "relative_intensity_vs_neighbors": np.nan,
            "relative_texture_vs_neighbors": np.nan,
            "relative_circularity_vs_neighbors": np.nan,
            "relative_solidity_vs_neighbors": np.nan,
            "relative_color_delta_lab_vs_neighbors": np.nan,
            "relative_local_contrast_vs_neighbors": np.nan,
            "relative_entropy_vs_neighbors": np.nan,
            "relative_histogram_vs_neighbors": np.nan,
            "feature_knn_distance": np.nan,
            "touches_image_border": touches_image_border,
            "is_too_small": is_too_small,
            "is_too_large": is_too_large,
            "suspicious_aspect_ratio": suspicious_aspect_ratio,
            "poor_local_background": poor_local_background,
            "critical_poor_local_background": critical_poor_local_background,
            "low_yolo_conf": False,
            "mask_fragment_after_overlap": bool(mask_meta.get("mask_fragment_after_overlap", False)),
            "possible_merged_colony": False,
            "possible_merged_colony_score": 0.0,
            "possible_segmentation_artifact": False,
            "invalid_geometry": bool(
                (not np.isfinite(area))
                or area <= 0
                or (not np.isfinite(perimeter))
                or perimeter <= 0
                or (np.isfinite(circularity) and circularity > 1.5)
                or (not np.isfinite(aspect_ratio))
            ),
            "mask_area_input": float(mask_meta.get("mask_area_input", np.nan)),
            "mask_area_after_feature_cleaning": float(mask_meta.get("mask_area_after_feature_cleaning", np.nan)),
            "mask_area_feature_loss_fraction": float(mask_meta.get("mask_area_feature_loss_fraction", np.nan)),
            "original_area": float(mask_meta.get("original_area", np.nan)),
            "cleaned_area": float(mask_meta.get("cleaned_area", np.nan)),
            "area_loss_fraction": float(mask_meta.get("area_loss_fraction", np.nan)),
            "n_components_before": float(mask_meta.get("n_components_before", np.nan)),
            "n_components_after": float(mask_meta.get("n_components_after", np.nan)),
            "technical_warning": technical_warning,
            "technical_reason": "",
        }

        if detections_df is not None:
            det_row = detections_lookup.get(int(colony_id), {})
            row["yolo_conf"] = float(det_row.get("yolo_conf", np.nan))
            row["x1"] = float(det_row.get("x1", np.nan))
            row["y1"] = float(det_row.get("y1", np.nan))
            row["x2"] = float(det_row.get("x2", np.nan))
            row["y2"] = float(det_row.get("y2", np.nan))
            row["bbox_area_yolo"] = float(det_row.get("bbox_area_yolo", np.nan))
            row["mask_area_raw"] = float(det_row.get("mask_area_raw", np.nan))
            row["mask_area_cleaned"] = float(det_row.get("mask_area_cleaned", np.nan))
            row["mask_area_after_overlap"] = float(det_row.get("mask_area_after_overlap", np.nan))
            row["mask_area_loss_fraction"] = float(det_row.get("mask_area_loss_fraction", np.nan))
            row["original_area"] = float(det_row.get("original_area", row.get("original_area", np.nan)))
            row["cleaned_area"] = float(det_row.get("cleaned_area", row.get("cleaned_area", np.nan)))
            row["area_loss_fraction"] = float(det_row.get("area_loss_fraction", row.get("area_loss_fraction", np.nan)))
            row["n_components_before"] = float(det_row.get("n_components_before", row.get("n_components_before", np.nan)))
            row["n_components_after"] = float(det_row.get("n_components_after", row.get("n_components_after", np.nan)))
            row["mask_fragment_after_overlap"] = bool(
                row.get("mask_fragment_after_overlap", False) or det_row.get("mask_fragment_after_overlap", False)
            )

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    features_df = pd.DataFrame(rows).reset_index(drop=True)

    n = len(features_df)
    if n >= 2:
        coords = features_df[["centroid_x", "centroid_y"]].to_numpy(dtype=float)
        tree = cKDTree(coords)

        for i in range(n):
            k = min(config.neighbor_k + 1, n)
            dists, idxs = tree.query(coords[i], k=k)
            dists = np.atleast_1d(dists).astype(float)
            idxs = np.atleast_1d(idxs).astype(int)

            mask_non_self = idxs != i
            neighbor_dists = dists[mask_non_self]
            neighbor_idxs = idxs[mask_non_self]

            features_df.at[i, "nearest_neighbor_distance"] = (
                float(neighbor_dists[0]) if neighbor_dists.size else np.nan
            )
            features_df.at[i, "mean_5nn_distance"] = (
                float(np.mean(neighbor_dists[:5])) if neighbor_dists.size else np.nan
            )

            density_count = len(
                tree.query_ball_point(coords[i], r=float(config.local_density_radius))
            ) - 1
            features_df.at[i, "local_density_r"] = float(max(0, density_count))

            k_rel = min(config.neighbor_k, n - 1)
            if k_rel >= 1 and neighbor_idxs.size > 0:
                nn = neighbor_idxs[:k_rel]
                neighbor_feature_map = {
                    "relative_area_vs_neighbors": "area",
                    "relative_intensity_vs_neighbors": "intensity_mean",
                    "relative_texture_vs_neighbors": "glcm_contrast",
                    "relative_circularity_vs_neighbors": "circularity",
                    "relative_solidity_vs_neighbors": "solidity",
                    "relative_color_delta_lab_vs_neighbors": "local_color_delta_lab",
                    "relative_local_contrast_vs_neighbors": "local_contrast",
                    "relative_entropy_vs_neighbors": "intensity_entropy",
                }
                for out_col, source_col in neighbor_feature_map.items():
                    if source_col not in features_df.columns:
                        continue
                    cur_value = float(features_df.at[i, source_col])
                    neighbor_values = features_df.loc[nn, source_col].to_numpy(dtype=float)
                    features_df.at[i, out_col] = _robust_neighbor_delta(cur_value, neighbor_values, eps=eps)

    plate_area_median = np.nanmedian(features_df["area"].to_numpy(dtype=float))
    plate_equivalent_diameter_median = np.nanmedian(features_df["equivalent_diameter"].to_numpy(dtype=float))
    plate_intensity_median = np.nanmedian(
        features_df["intensity_mean"].to_numpy(dtype=float)
    )
    plate_texture_median = np.nanmedian(
        features_df["glcm_contrast"].to_numpy(dtype=float)
    )

    features_df["log_area"] = np.log1p(features_df["area"].astype(float))
    features_df["area_to_median_ratio"] = (
        features_df["area"].astype(float) / (plate_area_median + eps)
        if np.isfinite(plate_area_median) and abs(plate_area_median) > eps
        else np.nan
    )
    features_df["equivalent_diameter_to_median_ratio"] = (
        features_df["equivalent_diameter"].astype(float) / (plate_equivalent_diameter_median + eps)
        if np.isfinite(plate_equivalent_diameter_median) and abs(plate_equivalent_diameter_median) > eps
        else np.nan
    )
    features_df["relative_area_vs_plate_median"] = features_df["area_to_median_ratio"]
    features_df["relative_intensity_vs_plate_median"] = (
        features_df["intensity_mean"] / (plate_intensity_median + eps)
        if np.isfinite(plate_intensity_median) and abs(plate_intensity_median) > eps
        else np.nan
    )
    features_df["relative_texture_vs_plate_median"] = (
        features_df["glcm_contrast"] / (plate_texture_median + eps)
        if np.isfinite(plate_texture_median) and abs(plate_texture_median) > eps
        else np.nan
    )

    features_df["log_nearest_neighbor_distance"] = np.log1p(
        features_df["nearest_neighbor_distance"].replace([np.inf, -np.inf], np.nan).astype(float)
    )
    features_df["log_mean_5nn_distance"] = np.log1p(
        features_df["mean_5nn_distance"].replace([np.inf, -np.inf], np.nan).astype(float)
    )

    if n >= 4:
        features_df["voronoi_area"] = _compute_voronoi_areas(
            features_df[["centroid_x", "centroid_y"]].to_numpy(dtype=float),
            image_shape=(h, w),
        )

    features_df = _add_feature_knn_distance(features_df, config)

    median_area = plate_area_median if len(features_df) else np.nan
    min_area_threshold = float(config.min_area)
    if np.isfinite(median_area):
        min_area_threshold = max(
            min_area_threshold,
            float(median_area) * float(config.min_area_relative_to_median),
        )
    max_area_threshold = float(config.max_area_fraction) * image_area
    if np.isfinite(median_area) and median_area > 0:
        max_area_threshold = min(
            max_area_threshold,
            float(median_area) * float(config.max_area_relative_to_median),
        )

    features_df["is_too_small"] = features_df["area"].astype(float) < min_area_threshold
    features_df["is_too_large"] = features_df["area"].astype(float) > max_area_threshold

    if "yolo_conf" not in features_df.columns:
        features_df["yolo_conf"] = np.nan
    features_df["low_yolo_conf"] = (
        features_df["yolo_conf"].fillna(1.0).astype(float) < float(config.min_yolo_conf_for_analysis)
    )

    for col, default in {
        "poor_local_background": False,
        "mask_fragment_after_overlap": False,
        "invalid_geometry": False,
        "touches_image_border": False,
        "suspicious_aspect_ratio": False,
    }.items():
        if col not in features_df.columns:
            features_df[col] = default

    technical_flags = (
        features_df["is_too_small"].fillna(True).astype(bool)
        | features_df["is_too_large"].fillna(True).astype(bool)
        | features_df["suspicious_aspect_ratio"].fillna(False).astype(bool)
        | features_df["low_yolo_conf"].fillna(False).astype(bool)
        | features_df["poor_local_background"].fillna(False).astype(bool)
        | features_df["mask_fragment_after_overlap"].fillna(False).astype(bool)
        | features_df["invalid_geometry"].fillna(False).astype(bool)
    )
    if config.exclude_border_touching_from_anomaly:
        technical_flags = technical_flags | features_df["touches_image_border"].fillna(False).astype(bool)

    features_df["technical_warning"] = technical_flags.astype(bool)
    features_df["technical_reason"] = features_df.apply(_build_technical_reason, axis=1)
    features_df.loc[~features_df["technical_warning"].astype(bool), "technical_reason"] = ""
    features_df["valid_for_anomaly"] = ~features_df["technical_warning"].astype(bool)

    if detections_df is not None:
        for col in [
            "yolo_conf",
            "x1",
            "y1",
            "x2",
            "y2",
            "bbox_area_yolo",
            "mask_area_raw",
            "mask_area_cleaned",
            "mask_area_after_overlap",
            "mask_area_loss_fraction",
            "original_area",
            "cleaned_area",
            "area_loss_fraction",
            "n_components_before",
            "n_components_after",
        ]:
            if col not in features_df.columns:
                features_df[col] = np.nan

    features_df = _augment_features_after_extraction(
        features_df=features_df,
        image_rgb=image_rgb,
        masks_by_colony_id=masks_by_colony_id,
        gray=gray,
        lab=lab,
        config=config,
        image_area=image_area,
    )

    return features_df

FEATURE_GROUPS = {
    "size_shape_score": [
        "log_area",
        "equivalent_diameter",
        "area_to_median_ratio",
        "aspect_ratio",
        "circularity",
        "eccentricity",
        "solidity",
        "convexity_defect_ratio",
        "radial_contour_cv",
    ],
    "color_background_score": [
        "mean_L_lab",
        "mean_a_lab",
        "mean_b_lab",
        "std_L_lab",
        "std_a_lab",
        "std_b_lab",
        "mean_S",
        "mean_V",
        "mean_H_sin",
        "mean_H_cos",
        "local_delta_L",
        "local_delta_a",
        "local_delta_b",
        "local_color_delta_lab",
    ],
    "intensity_score": [
        "intensity_mean",
        "intensity_median",
        "intensity_iqr",
        "intensity_p95_p05_range",
        "intensity_cv",
        "intensity_entropy",
        "center_rim_intensity_delta",
        "center_rim_intensity_ratio",
        "radial_intensity_slope",
        "radial_intensity_std",
    ],
    "texture_score": [
        "laplacian_var",
        "glcm_contrast",
        "glcm_homogeneity",
        "glcm_energy",
        "glcm_correlation",
        "glcm_dissimilarity",
        "glcm_ASM",
        "lbp_mean",
        "lbp_std",
        "lbp_entropy",
    ],
    "histogram_score": [
        "hist_intensity_entropy",
        "hist_intensity_width",
        "hist_intensity_skewness",
        "hist_intensity_kurtosis",
        "dark_fraction",
        "bright_fraction",
        "contrast_hist_entropy",
        "intensity_hist_js_to_plate_median",
        "intensity_hist_wasserstein_to_plate_median",
        "intensity_hist_js_to_neighbors",
        "L_hist_js_to_plate_median",
        "contrast_hist_js_to_plate_median",
    ],
    "spatial_context_score": [
        "log_nearest_neighbor_distance",
        "log_mean_5nn_distance",
        "edge_nearest_distance",
        "mean_5nn_edge_distance",
        "nearest_distance_to_median_diameter_ratio",
        "edge_distance_to_median_diameter_ratio",
        "local_density_r",
    ],
    "neighbor_difference_score": [
        "relative_area_vs_neighbors",
        "relative_intensity_vs_neighbors",
        "relative_texture_vs_neighbors",
        "relative_circularity_vs_neighbors",
        "relative_solidity_vs_neighbors",
        "relative_color_delta_lab_vs_neighbors",
        "relative_local_contrast_vs_neighbors",
        "relative_entropy_vs_neighbors",
        "relative_histogram_vs_neighbors",
        "feature_knn_distance",
    ],
    "morphotype_score": [
        "within_morphotype_anomaly_score",
        "size_within_morphotype_z",
        "texture_within_morphotype_z",
        "color_within_morphotype_z",
        "histogram_within_morphotype_z",
    ],
}

EXCLUDE_FROM_ANOMALY = {
    "colony_id",
    "technical_warning",
    "valid_for_anomaly",
    "selected_for_further_analysis",
    "technical_reason",
    "recommendation_status",
    "anomaly_type",
    "explanations",
    "explanations_text",
    "touches_image_border",
    "is_too_small",
    "is_too_large",
    "suspicious_aspect_ratio",
    "low_yolo_conf",
    "poor_local_background",
    "critical_poor_local_background",
    "mask_fragment_after_overlap",
    "possible_merged_colony",
    "possible_segmentation_artifact",
    "invalid_geometry",
    "rare_morphotype_flag",
    "yolo_conf",
    "x1",
    "y1",
    "x2",
    "y2",
    "bbox_area_yolo",
    "mask_area_raw",
    "mask_area_cleaned",
    "mask_area_after_overlap",
    "mask_area_loss_fraction",
    "mask_area_input",
    "mask_area_after_feature_cleaning",
    "mask_area_feature_loss_fraction",
    "original_area",
    "cleaned_area",
    "area_loss_fraction",
    "n_components_before",
    "n_components_after",
    "centroid_x",
    "centroid_y",
    "centroid_x_norm",
    "centroid_y_norm",
    "orientation",
    "mean_H",
    "std_H",
    "intensity_min",
    "intensity_max",
    "distance_to_plate_center",
    "distance_to_plate_edge",
}


def _prepare_numeric_matrix(df: pd.DataFrame, columns: list[str]) -> tuple[np.ndarray, list[str]]:
    columns = [c for c in columns if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if not columns:
        return np.empty((len(df), 0), dtype=float), []
    X = df[columns].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
    finite_columns = np.any(np.isfinite(X), axis=0)
    columns = [c for c, keep in zip(columns, finite_columns) if keep]
    X = X[:, finite_columns]
    if X.shape[1] == 0:
        return np.empty((len(df), 0), dtype=float), []
    try:
        X = SimpleImputer(strategy="median").fit_transform(X)
        X = RobustScaler().fit_transform(X)
    except Exception:
        return np.empty((len(df), 0), dtype=float), []
    return X, columns


def _robust_outlier_score(X: np.ndarray) -> np.ndarray:
    if X.size == 0 or X.shape[1] == 0:
        return np.zeros(X.shape[0], dtype=float)
    eps = 1e-9
    med = np.nanmedian(X, axis=0)
    mad = np.nanmedian(np.abs(X - med), axis=0)
    mad = np.where(np.isfinite(mad) & (mad >= eps), mad, eps)
    z = np.abs((X - med) / (1.4826 * mad))
    z = np.where(np.isfinite(z), z, 0.0)
    return 0.7 * np.nanmean(z, axis=1) + 0.3 * np.nanmax(z, axis=1)


def _rank01(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    ranks = np.zeros(values.shape[0], dtype=float)
    finite = np.isfinite(values)
    if finite.sum() == 0:
        return ranks
    ranks[finite] = pd.Series(values[finite]).rank(method="average", pct=True).to_numpy(dtype=float)
    return ranks


def _score_has_signal(values: np.ndarray) -> bool:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return bool(values.size >= 2 and np.nanstd(values) >= 1e-8)


def assign_morphotypes(features_df: pd.DataFrame, config: AnomalyDetectionConfig) -> pd.DataFrame:
    """Выделить морфологические подгруппы только при доказуемой кластерной структуре.

    Морфотип здесь не является видом бактерий. Это вычислимая группа колоний,
    похожих по размеру, форме, яркости, цвету и текстуре. Если разделение слабое
    (низкий silhouette), все объекты остаются в одном морфотипе.
    """
    df = features_df.copy()
    n = len(df)
    df["morphotype_id"] = 0
    df["morphotype_size"] = n
    df["morphotype_fraction"] = 1.0 if n else np.nan
    df["rare_morphotype_flag"] = False
    df["within_morphotype_anomaly_score"] = 0.0
    df["size_within_morphotype_z"] = np.nan
    df["texture_within_morphotype_z"] = np.nan
    df["color_within_morphotype_z"] = np.nan
    df["histogram_within_morphotype_z"] = np.nan
    df["morphotype_silhouette"] = np.nan

    if not bool(getattr(config, "use_morphotype_clustering", True)) or n == 0:
        return df

    valid_mask = df.get("valid_for_anomaly", pd.Series(True, index=df.index)).fillna(False).astype(bool)
    if "technical_warning" in df.columns:
        valid_mask = valid_mask & (~df["technical_warning"].fillna(False).astype(bool))
    valid_idx = np.where(valid_mask.to_numpy())[0]
    n_valid = len(valid_idx)
    if n_valid < 10:
        df.loc[valid_idx, "morphotype_size"] = n_valid
        df.loc[valid_idx, "morphotype_fraction"] = 1.0 if n_valid else np.nan
        return df

    core_cols = [
        "log_area",
        "circularity",
        "eccentricity",
        "solidity",
        "intensity_mean",
        "intensity_iqr",
        "mean_L_lab",
        "mean_a_lab",
        "mean_b_lab",
        "glcm_contrast",
        "lbp_entropy",
        "local_color_delta_lab",
        "intensity_hist_js_to_plate_median",
    ]
    valid_df = df.iloc[valid_idx].copy()
    X, used_cols = _prepare_numeric_matrix(valid_df, core_cols)
    if X.shape[1] == 0:
        return df

    max_k = min(int(config.max_morphotypes), max(2, int(np.sqrt(n_valid)) + 1), n_valid - 1)
    best_labels = None
    best_score = -np.inf

    if str(config.morphotype_method).lower() == "dbscan":
        try:
            labels = DBSCAN(eps=float(config.dbscan_eps), min_samples=max(3, int(config.min_morphotype_size))).fit_predict(X)
            non_noise = labels[labels >= 0]
            if len(np.unique(non_noise)) >= 2:
                best_labels = labels
                best_score = float(silhouette_score(X, labels)) if len(np.unique(labels)) >= 2 else -np.inf
        except Exception:
            best_labels = None

    if best_labels is None:
        for k in range(2, max_k + 1):
            try:
                labels = KMeans(n_clusters=k, random_state=config.random_state, n_init=10).fit_predict(X)
                if len(np.unique(labels)) < 2:
                    continue
                score = silhouette_score(X, labels) if n_valid > k else -np.inf
                if np.isfinite(score) and score > best_score:
                    best_score = float(score)
                    best_labels = labels
            except Exception:
                continue

    # Если разделение слабое, не создаём искусственные морфотипы.
    min_sil = float(getattr(config, "min_morphotype_silhouette", 0.20))
    if best_labels is None or not np.isfinite(best_score) or best_score < min_sil:
        df.loc[valid_idx, "morphotype_id"] = 0
        df.loc[valid_idx, "morphotype_size"] = n_valid
        df.loc[valid_idx, "morphotype_fraction"] = 1.0
        df.loc[valid_idx, "morphotype_silhouette"] = best_score if np.isfinite(best_score) else np.nan
        return df

    labels = np.asarray(best_labels, dtype=int)
    if np.any(labels < 0):
        labels = labels.copy()
        labels[labels < 0] = labels.max() + 1

    df.loc[valid_idx, "morphotype_id"] = labels
    df.loc[valid_idx, "morphotype_silhouette"] = best_score

    for lab_id in np.unique(labels):
        member_local = np.where(labels == lab_id)[0]
        member_global = valid_idx[member_local]
        size = len(member_global)
        fraction = size / max(1, n_valid)
        rare = bool(size < int(config.min_morphotype_size) or fraction < float(config.rare_morphotype_fraction))
        df.loc[member_global, "morphotype_size"] = size
        df.loc[member_global, "morphotype_fraction"] = fraction
        df.loc[member_global, "rare_morphotype_flag"] = rare

        if size >= 3:
            X_cluster = X[member_local]
            df.loc[member_global, "within_morphotype_anomaly_score"] = _robust_outlier_score(X_cluster)
            for out_col, source_cols in {
                "size_within_morphotype_z": ["log_area"],
                "texture_within_morphotype_z": ["glcm_contrast", "lbp_entropy"],
                "color_within_morphotype_z": ["local_color_delta_lab", "mean_L_lab", "mean_a_lab", "mean_b_lab"],
                "histogram_within_morphotype_z": ["intensity_hist_js_to_plate_median", "hist_intensity_entropy"],
            }.items():
                vals = []
                for global_i in member_global:
                    z_parts = []
                    for source_col in source_cols:
                        if source_col in df.columns:
                            z = _robust_z_value(float(df.at[global_i, source_col]), df.loc[member_global, source_col].to_numpy(dtype=float))
                            if np.isfinite(z):
                                z_parts.append(abs(z))
                    vals.append(float(np.mean(z_parts)) if z_parts else np.nan)
                df.loc[member_global, out_col] = vals
    return df


def compute_weighted_final_anomaly_score(
    results_df: pd.DataFrame,
    valid_indices: np.ndarray,
    config: AnomalyDetectionConfig,
) -> pd.DataFrame:
    if valid_indices.size == 0:
        return results_df
    weights = {
        "lof_score_rank": 0.18,
        "isolation_forest_score_rank": 0.12,
        "robust_z_score_rank": 0.12,
        "size_shape_score_rank": 0.15,
        "texture_score_rank": 0.15,
        "intensity_score_rank": 0.10,
        "color_background_score_rank": 0.08,
        "histogram_score_rank": 0.08,
        "neighbor_difference_score_rank": float(getattr(config, "neighbor_difference_weight", 0.05)),
        "spatial_context_score_rank": float(getattr(config, "spatial_context_weight", 0.03)),
        "morphotype_score_rank": 0.04,
        "cluster_outlier_score_rank": 0.03,
    }
    used_values = []
    used_weights = []
    for rank_col, weight in weights.items():
        score_col = rank_col[:-5] if rank_col.endswith("_rank") else rank_col
        if rank_col not in results_df.columns:
            continue
        score_values = results_df.loc[valid_indices, score_col].to_numpy(dtype=float) if score_col in results_df.columns else results_df.loc[valid_indices, rank_col].to_numpy(dtype=float)
        if not _score_has_signal(score_values):
            continue
        ranks = results_df.loc[valid_indices, rank_col].to_numpy(dtype=float)
        if not _score_has_signal(ranks):
            continue
        used_values.append(ranks)
        used_weights.append(float(weight))
    if not used_values:
        results_df.loc[valid_indices, "final_anomaly_score_raw"] = 0.0
        results_df.loc[valid_indices, "final_anomaly_score_percentile"] = 0.0
        results_df.loc[valid_indices, "final_anomaly_score"] = 0.0
        return results_df
    w = np.asarray(used_weights, dtype=float)
    w = w / max(float(w.sum()), 1e-12)
    raw = np.average(np.vstack(used_values), axis=0, weights=w) if bool(config.use_weighted_final_score) else np.nanmean(np.vstack(used_values), axis=0)
    raw = np.clip(np.nan_to_num(raw, nan=0.0, posinf=1.0, neginf=0.0), 0.0, 1.0)
    percentile = _rank01(raw) if _score_has_signal(raw) else raw.copy()
    results_df.loc[valid_indices, "final_anomaly_score_raw"] = raw
    results_df.loc[valid_indices, "final_anomaly_score_percentile"] = percentile
    # Use raw for objective selection; percentile is stored only as within-plate context.
    results_df.loc[valid_indices, "final_anomaly_score"] = raw
    return results_df


def _assign_anomaly_types(results_df: pd.DataFrame) -> pd.Series:
    type_map = {
        "size_shape_score_rank": "size_shape",
        "texture_score_rank": "texture",
        "color_background_score_rank": "color_background",
        "intensity_score_rank": "intensity",
        "histogram_score_rank": "histogram",
        "spatial_context_score_rank": "spatial_isolation",
        "neighbor_difference_score_rank": "neighbor_difference",
        "morphotype_score_rank": "morphotype",
    }
    labels: list[str] = []
    for _, row in results_df.iterrows():
        if bool(row.get("technical_warning", False)) or not bool(row.get("valid_for_anomaly", False)):
            labels.append("technical_warning")
            continue
        scored = []
        for col, label_name in type_map.items():
            value = row.get(col, np.nan)
            if pd.notna(value) and np.isfinite(float(value)):
                scored.append((float(value), label_name))
        if not scored:
            labels.append("mixed")
            continue
        scored.sort(reverse=True, key=lambda x: x[0])
        best_value, best_label = scored[0]
        second_value = scored[1][0] if len(scored) > 1 else 0.0
        labels.append(best_label if best_value >= 0.65 and (best_value - second_value) >= 0.12 else "mixed")
    return pd.Series(labels, index=results_df.index, dtype=object)


def _compute_evidence_reliability_and_status(
    results_df: pd.DataFrame,
    valid_indices: np.ndarray,
    config: AnomalyDetectionConfig,
) -> pd.DataFrame:
    """Рассчитать доказательность, согласованность, надёжность и финальный статус.

    В отличие от чистого rank-based top-N, статус select_candidate требует:
    - высокий raw-score;
    - абсолютную выраженность отклонений;
    - минимум несколько независимых групп признаков;
    - достаточную надёжность маски/фона/текстуры;
    - согласованность нескольких методов.
    """
    evidence_rank_cols = [
        "size_shape_score_rank",
        "texture_score_rank",
        "intensity_score_rank",
        "color_background_score_rank",
        "histogram_score_rank",
        "neighbor_difference_score_rank",
        "morphotype_score_rank",
    ]
    evidence_score_cols = [c[:-5] for c in evidence_rank_cols if c.endswith("_rank")]
    consensus_rank_cols = [
        "lof_score_rank",
        "isolation_forest_score_rank",
        "robust_z_score_rank",
        "size_shape_score_rank",
        "texture_score_rank",
        "intensity_score_rank",
        "color_background_score_rank",
        "histogram_score_rank",
    ]

    results_df["absolute_evidence_strength"] = 0.0
    results_df["independent_evidence_count"] = 0
    results_df["consensus_score"] = 0.0
    results_df["ablation_stability_score"] = 0.0
    results_df["segmentation_reliability"] = 0.0
    results_df["texture_reliability"] = 0.0
    results_df["background_reliability"] = 0.0
    results_df["shape_reliability"] = 0.0
    results_df["overall_reliability_score"] = 0.0
    results_df["recommendation_status"] = "technical_exclude"

    if valid_indices.size == 0:
        return results_df

    # Absolute evidence: robust z of group scores among valid objects, clipped to 0..1.
    absolute_components: dict[str, np.ndarray] = {}
    for score_col in evidence_score_cols:
        if score_col not in results_df.columns:
            continue
        vals = results_df.loc[valid_indices, score_col].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
        med = np.nanmedian(vals) if np.isfinite(vals).any() else np.nan
        mad = _finite_mad(vals)
        if not np.isfinite(med) or not np.isfinite(mad) or mad <= 0:
            continue
        z = (vals - med) / (1.4826 * mad + 1e-8)
        absolute_components[score_col] = np.clip(np.abs(z), 0, 5) / 5.0

    if absolute_components:
        abs_matrix = np.vstack(list(absolute_components.values()))
        results_df.loc[valid_indices, "absolute_evidence_strength"] = np.nanmean(abs_matrix, axis=0)

    for pos, idx in enumerate(valid_indices):
        row = results_df.loc[idx]

        evidence_count = 0
        evidence_values_for_ablation = []
        for rank_col in evidence_rank_cols:
            score_col = rank_col[:-5] if rank_col.endswith("_rank") else rank_col
            rank_val = float(row.get(rank_col, 0.0)) if pd.notna(row.get(rank_col, np.nan)) else 0.0

            # Spatial context is not treated as a strong independent proof.
            if rank_col == "spatial_context_score_rank":
                continue

            abs_component = np.nan
            if score_col in absolute_components:
                abs_component = float(absolute_components[score_col][pos])
            is_evidence = bool(rank_val >= 0.90 and (np.isfinite(abs_component) and abs_component >= 0.40))
            if is_evidence:
                evidence_count += 1
            if np.isfinite(rank_val):
                evidence_values_for_ablation.append(rank_val)

        results_df.at[idx, "independent_evidence_count"] = int(evidence_count)
        results_df.at[idx, "ablation_stability_score"] = float(evidence_count / max(1, len(evidence_values_for_ablation)))

        consensus_parts = []
        for col in consensus_rank_cols:
            if col in results_df.columns and pd.notna(row.get(col)):
                threshold = 0.95 if col in ["lof_score_rank", "isolation_forest_score_rank", "robust_z_score_rank"] else 0.90
                value = float(row.get(col))
                if np.isfinite(value):
                    consensus_parts.append(float(value >= threshold))
        results_df.at[idx, "consensus_score"] = float(np.mean(consensus_parts)) if consensus_parts else 0.0

        yolo_conf = row.get("yolo_conf", np.nan)
        seg_rel = float(np.clip(float(yolo_conf) / 0.5, 0.0, 1.0)) if pd.notna(yolo_conf) and np.isfinite(float(yolo_conf)) else 0.8

        area = float(row.get("area", 0.0)) if pd.notna(row.get("area", np.nan)) else 0.0
        texture_rel = float(np.clip(area / 100.0, 0.0, 1.0))

        bg_area = float(row.get("local_background_area", 0.0)) if pd.notna(row.get("local_background_area", np.nan)) else 0.0
        bg_rel = float(np.clip(bg_area / max(50.0, 0.2 * max(area, 1.0)), 0.0, 1.0))
        if bool(row.get("poor_local_background", False)):
            bg_rel *= 0.5
        if bool(row.get("critical_poor_local_background", False)):
            bg_rel = 0.0

        shape_rel = 0.6 if bool(row.get("possible_merged_colony", False)) else 1.0
        if bool(row.get("technical_warning", False)):
            seg_rel = texture_rel = bg_rel = shape_rel = 0.0

        overall = float(np.nanmean([seg_rel, texture_rel, bg_rel, shape_rel]))
        results_df.at[idx, "segmentation_reliability"] = seg_rel
        results_df.at[idx, "texture_reliability"] = texture_rel
        results_df.at[idx, "background_reliability"] = bg_rel
        results_df.at[idx, "shape_reliability"] = shape_rel
        results_df.at[idx, "overall_reliability_score"] = overall

    for i, row in results_df.iterrows():
        raw_score = float(row.get("final_anomaly_score_raw", 0.0))
        abs_strength = float(row.get("absolute_evidence_strength", 0.0))
        within = row.get("within_morphotype_anomaly_score", np.nan)
        within_val = float(within) if pd.notna(within) and np.isfinite(float(within)) else np.nan

        if bool(row.get("technical_warning", False)) or not bool(row.get("valid_for_anomaly", False)):
            status = "technical_exclude"
        elif bool(row.get("possible_merged_colony", False)) and raw_score >= 0.60:
            status = "review_segmentation"
        elif bool(row.get("rare_morphotype_flag", False)) and (not np.isfinite(within_val) or within_val < 0.75):
            status = "rare_morphotype"
        elif (
            raw_score >= float(config.min_final_score_raw)
            and abs_strength >= float(getattr(config, "min_absolute_evidence_strength", 0.35))
            and int(row.get("independent_evidence_count", 0)) >= int(config.min_independent_evidence_count)
            and float(row.get("overall_reliability_score", 0.0)) >= float(config.min_reliability_score)
            and float(row.get("consensus_score", 0.0)) >= float(config.min_consensus_score)
        ):
            status = "select_candidate"
        elif raw_score >= float(config.min_final_score_raw):
            status = "review_only"
        elif float(row.get("final_anomaly_score_percentile", 0.0)) >= 0.90:
            status = "weak_outlier"
        else:
            status = "no_valid_evidence"
        results_df.at[i, "recommendation_status"] = status

    results_df["selected_for_further_analysis"] = results_df["recommendation_status"].eq("select_candidate")
    return results_df


def compute_anomaly_scores(
    features_df: pd.DataFrame,
    config: AnomalyDetectionConfig | None = None,
) -> pd.DataFrame:
    if config is None:
        config = AnomalyDetectionConfig()
    if features_df is None:
        return pd.DataFrame()
    if features_df.empty:
        return features_df.copy()

    results_df = assign_morphotypes(features_df.copy(), config)
    n_total = len(results_df)

    score_columns = list(FEATURE_GROUPS.keys()) + [
        "robust_z_score",
        "lof_score",
        "isolation_forest_score",
        "cluster_outlier_score",
    ]
    for col in score_columns:
        results_df[col] = 0.0
        results_df[f"{col}_rank"] = 0.0
    results_df["final_anomaly_score_raw"] = 0.0
    results_df["final_anomaly_score_percentile"] = 0.0
    results_df["final_anomaly_score"] = 0.0
    results_df["global_anomaly_score"] = 0.0
    results_df["absolute_evidence_strength"] = 0.0
    results_df["anomaly_rank"] = pd.NA
    results_df["selected_for_further_analysis"] = False

    valid_mask = results_df.get("valid_for_anomaly", pd.Series(True, index=results_df.index)).fillna(False).astype(bool).to_numpy()
    if bool(getattr(config, "exclude_technical_from_anomaly", True)) and "technical_warning" in results_df.columns:
        valid_mask = valid_mask & (~results_df["technical_warning"].fillna(False).astype(bool).to_numpy())
    valid_indices = np.where(valid_mask)[0]
    if valid_indices.size == 0:
        results_df["anomaly_type"] = _assign_anomaly_types(results_df)
        results_df = _compute_evidence_reliability_and_status(results_df, valid_indices, config)
        return results_df.sort_values("final_anomaly_score", ascending=False).reset_index(drop=True)

    valid_df = results_df.iloc[valid_indices].copy()
    n_colonies = len(valid_df)

    for group_name, cols in FEATURE_GROUPS.items():
        X_group, used_cols = _prepare_numeric_matrix(valid_df, [c for c in cols if c in valid_df.columns])
        group_score = _robust_outlier_score(X_group) if used_cols else np.zeros(n_colonies, dtype=float)
        results_df.loc[valid_indices, group_name] = group_score

    numeric_columns = [
        c for c in valid_df.select_dtypes(include=[np.number]).columns
        if c not in EXCLUDE_FROM_ANOMALY
        and not c.endswith("_score")
        and not c.endswith("_rank")
        and not c.startswith("final_anomaly_score")
    ]
    X_scaled, numeric_columns = _prepare_numeric_matrix(valid_df, numeric_columns)
    results_df.loc[valid_indices, "robust_z_score"] = _robust_outlier_score(X_scaled)
    results_df.loc[valid_indices, "global_anomaly_score"] = results_df.loc[valid_indices, "robust_z_score"].to_numpy(dtype=float)

    if n_colonies < 5 or X_scaled.shape[1] == 0:
        lof_score = np.zeros(n_colonies, dtype=float)
    else:
        try:
            n_neighbors = max(2, min(int(config.lof_neighbors), n_colonies - 1))
            lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=min(max(config.contamination, 1e-3), 0.5))
            lof.fit_predict(X_scaled)
            lof_score = -lof.negative_outlier_factor_
        except Exception:
            lof_score = np.zeros(n_colonies, dtype=float)
    results_df.loc[valid_indices, "lof_score"] = lof_score

    if n_colonies < 8 or X_scaled.shape[1] == 0:
        iso_score = np.zeros(n_colonies, dtype=float)
    else:
        try:
            iso = IsolationForest(contamination=min(max(config.contamination, 1e-3), 0.5), random_state=config.random_state)
            iso.fit(X_scaled)
            iso_score = -iso.score_samples(X_scaled)
        except Exception:
            iso_score = np.zeros(n_colonies, dtype=float)
    results_df.loc[valid_indices, "isolation_forest_score"] = iso_score

    if n_colonies < 8 or X_scaled.shape[1] == 0:
        cluster_score = np.zeros(n_colonies, dtype=float)
    else:
        X_cluster = X_scaled
        try:
            n_components = min(5, X_cluster.shape[1], n_colonies - 1)
            if 1 <= n_components < X_cluster.shape[1]:
                X_cluster = PCA(n_components=n_components, random_state=config.random_state).fit_transform(X_cluster)
        except Exception:
            X_cluster = X_scaled
        try:
            min_samples = int(config.dbscan_min_samples) if config.dbscan_min_samples is not None else max(3, min(10, n_colonies // 20))
            min_samples = min(max(2, min_samples), n_colonies)
            labels = DBSCAN(eps=float(config.dbscan_eps), min_samples=min_samples).fit_predict(X_cluster)
            unique_non_noise = [lab for lab in np.unique(labels) if lab != -1]
            use_kmeans = np.all(labels == -1) or len(unique_non_noise) <= 1
        except Exception:
            labels = np.full(n_colonies, -1)
            use_kmeans = True
        if not use_kmeans:
            cluster_score = np.zeros(n_colonies, dtype=float)
            max_cluster_dist = 0.0
            for lab_id in [lab for lab in np.unique(labels) if lab != -1]:
                idx = np.where(labels == lab_id)[0]
                center = X_cluster[idx].mean(axis=0)
                d = np.linalg.norm(X_cluster[idx] - center, axis=1)
                cluster_score[idx] = d
                if d.size:
                    max_cluster_dist = max(max_cluster_dist, float(np.max(d)))
            noise_idx = np.where(labels == -1)[0]
            if noise_idx.size:
                cluster_score[noise_idx] = max_cluster_dist + 1.0
        else:
            try:
                k = min(3, max(2, int(np.sqrt(n_colonies))), n_colonies - 1)
                if k < 2:
                    cluster_score = np.zeros(n_colonies, dtype=float)
                else:
                    km = KMeans(n_clusters=k, random_state=config.random_state, n_init=10)
                    km_labels = km.fit_predict(X_cluster)
                    cluster_score = np.linalg.norm(X_cluster - km.cluster_centers_[km_labels], axis=1)
            except Exception:
                cluster_score = np.zeros(n_colonies, dtype=float)
    results_df.loc[valid_indices, "cluster_outlier_score"] = cluster_score

    for score_col in score_columns:
        values = results_df.loc[valid_indices, score_col].to_numpy(dtype=float)
        if _score_has_signal(values):
            results_df.loc[valid_indices, f"{score_col}_rank"] = _rank01(values)

    results_df = compute_weighted_final_anomaly_score(results_df, valid_indices, config)
    final_values = results_df.loc[valid_indices, "final_anomaly_score_raw"].to_numpy(dtype=float)
    if final_values.size:
        results_df.loc[valid_indices, "anomaly_rank"] = pd.Series(final_values).rank(method="dense", ascending=False).astype(int).to_numpy()
    results_df["anomaly_type"] = _assign_anomaly_types(results_df)
    results_df = _compute_evidence_reliability_and_status(results_df, valid_indices, config)

    STATUS_PRIORITY = {
        "select_candidate": 0,
        "review_segmentation": 1,
        "review_only": 2,
        "rare_morphotype": 3,
        "weak_outlier": 4,
        "no_valid_evidence": 5,
        "technical_exclude": 6,
    }
    results_df["status_priority"] = results_df["recommendation_status"].map(STATUS_PRIORITY).fillna(99).astype(int)
    results_df = results_df.sort_values(
        ["valid_for_anomaly", "status_priority", "final_anomaly_score_raw"],
        ascending=[False, True, False],
        na_position="last",
    ).reset_index(drop=True)
    return results_df

def explain_anomaly(
    row: pd.Series,
    features_df: pd.DataFrame,
    top_k: int = 5,
) -> list[str]:
    """Сформировать русскоязычные причины аномальности с числовым контекстом."""
    technical_reason_map = {
        "too_small": "маска слишком мала и похожа на технический фрагмент сегментации",
        "too_large": "маска слишком велика относительно других объектов на чашке",
        "touches_border": "маска касается границы изображения, объект может быть обрезан",
        "suspicious_aspect_ratio": "маска имеет подозрительно вытянутую форму",
        "low_yolo_conf": "низкая уверенность YOLO для этой маски",
        "poor_local_background": "локальный фон вокруг колонии ограничен, интерпретация контрастных признаков менее надёжна",
        "critical_poor_local_background": "локальный фон вокруг колонии практически отсутствует, объект исключён из автоматического отбора",
        "mask_fragment_after_overlap": "маска стала фрагментом после удаления пересечений с другими масками",
        "invalid_geometry": "геометрия маски некорректна для интерпретации как отдельной колонии",
        "possible_segmentation_artifact": "объект похож на технический артефакт сегментации",
    }

    fallback = "интегральный показатель аномальности повышен за счёт совокупности слабых отклонений"

    if bool(row.get("technical_warning", False)) or not bool(row.get("valid_for_anomaly", True)):
        raw_reasons = str(row.get("technical_reason", "") or "")
        reasons = [technical_reason_map.get(r.strip(), r.strip()) for r in raw_reasons.split(";") if r.strip()]
        default_reason = "маска имеет техническое предупреждение, результат требует проверки"
        if default_reason not in reasons:
            reasons.insert(0, default_reason)
        return reasons[: max(1, int(top_k))]

    if features_df is None or features_df.empty:
        return [fallback]

    valid_df = features_df.copy()
    if "valid_for_anomaly" in valid_df.columns:
        valid_df = valid_df[valid_df["valid_for_anomaly"].fillna(False).astype(bool)]
    if "technical_warning" in valid_df.columns:
        valid_df = valid_df[~valid_df["technical_warning"].fillna(False).astype(bool)]
    if valid_df.empty:
        valid_df = features_df.copy()

    def _num(value: Any) -> float:
        try:
            value = float(value)
        except Exception:
            return np.nan
        return value if np.isfinite(value) else np.nan

    def _fmt(value: float) -> str:
        value = _num(value)
        if not np.isfinite(value):
            return "nan"
        if abs(value) >= 1000 or (0 < abs(value) < 0.01):
            return f"{value:.3g}"
        return f"{value:.3f}"

    def _context(col: str) -> tuple[float, float, float, float]:
        if col not in valid_df.columns or col not in row.index:
            return np.nan, np.nan, np.nan, np.nan
        value = _num(row.get(col, np.nan))
        values = valid_df[col].replace([np.inf, -np.inf], np.nan).dropna().astype(float).to_numpy()
        if values.size < 3 or not np.isfinite(value):
            return value, np.nan, np.nan, np.nan
        median = float(np.nanmedian(values))
        mad = _finite_mad(values)
        z = float((value - median) / (1.4826 * mad + 1e-8)) if np.isfinite(mad) and mad > 0 else np.nan
        percentile = float(pd.Series(values).rank(pct=True).iloc[np.searchsorted(np.sort(values), value, side="left")] if False else np.mean(values <= value))
        return value, median, z, percentile

    def _reason(col: str, text: str, score: float | None = None) -> tuple[float, str] | None:
        value, median, z, percentile = _context(col)
        if score is None:
            score = abs(z) if np.isfinite(z) else 0.0
        if not np.isfinite(score) or score <= 0:
            return None
        detail = f"{text}: {col}={_fmt(value)}, медиана={_fmt(median)}, robust_z={_fmt(z)}, percentile={_fmt(percentile)}"
        return float(score), detail

    reasons_scored: list[tuple[float, str]] = []

    for col in ["log_area", "area_to_median_ratio"]:
        value, median, z, _ = _context(col)
        if np.isfinite(z) and z > 2.0:
            item = _reason(col, "площадь значительно выше медианы по чашке", abs(z))
            if item:
                reasons_scored.append(item)
        elif np.isfinite(z) and z < -2.0:
            item = _reason(col, "площадь значительно ниже медианы по чашке", abs(z))
            if item:
                reasons_scored.append(item)

    checks = [
        ("circularity", "форма менее округлая по сравнению с большинством колоний", "low", 2.0),
        ("eccentricity", "колония более вытянутая по сравнению с большинством", "high", 2.0),
        ("solidity", "контур колонии более неровный", "low", 2.0),
        ("convexity_defect_ratio", "контур колонии более неровный", "high", 2.0),
        ("radial_contour_cv", "контур колонии более неровный", "high", 2.0),
        ("intensity_iqr", "яркостная неоднородность выше типичного уровня", "high", 2.0),
        ("intensity_entropy", "яркостная неоднородность выше типичного уровня", "high", 2.0),
        ("center_rim_intensity_delta", "колония имеет выраженное отличие яркости между центром и периферией", "abs", 2.0),
        ("radial_intensity_slope", "колония имеет нетипичный радиальный профиль яркости", "abs", 2.0),
        ("glcm_contrast", "текстурная неоднородность выше типичного уровня", "high", 2.0),
        ("lbp_entropy", "текстурная неоднородность выше типичного уровня", "high", 2.0),
        ("hist_intensity_entropy", "гистограмма яркости имеет нетипичную форму", "high", 2.0),
        ("intensity_hist_js_to_plate_median", "гистограмма яркости отличается от типичной гистограммы чашки", "high", 1.5),
        ("intensity_hist_js_to_neighbors", "гистограмма яркости отличается от ближайших соседей", "high", 1.5),
        ("local_color_delta_lab", "цвет сильнее отличается от локального фона", "high", 2.0),
        ("local_delta_L", "яркость отличается от локального фона", "abs", 2.0),
        ("feature_knn_distance", "колония далека от других объектов в пространстве морфологических признаков", "high", 2.0),
    ]
    for col, text, direction, threshold in checks:
        value, median, z, _ = _context(col)
        take = (
            (direction == "high" and np.isfinite(z) and z > threshold)
            or (direction == "low" and np.isfinite(z) and z < -threshold)
            or (direction == "abs" and np.isfinite(z) and abs(z) > threshold)
        )
        if take:
            item = _reason(col, text, abs(z))
            if item:
                reasons_scored.append(item)

    neighbor_reason_map = {
        "relative_area_vs_neighbors": "колония отличается от ближайших соседей по площади",
        "relative_circularity_vs_neighbors": "колония отличается от ближайших соседей по форме",
        "relative_solidity_vs_neighbors": "колония отличается от ближайших соседей по форме",
        "relative_color_delta_lab_vs_neighbors": "колония отличается от ближайших соседей по цвету",
        "relative_entropy_vs_neighbors": "колония отличается от ближайших соседей по текстуре",
        "relative_texture_vs_neighbors": "колония отличается от ближайших соседей по текстуре",
        "relative_histogram_vs_neighbors": "колония отличается от ближайших соседей по гистограмме яркости",
    }
    for col, text in neighbor_reason_map.items():
        value = _num(row.get(col, np.nan))
        if np.isfinite(value) and abs(value) >= 2.0:
            reasons_scored.append((abs(value), f"{text}: {col}={_fmt(value)}"))

    nn_value, nn_median, nn_z, _ = _context("nearest_neighbor_distance")
    if np.isfinite(nn_z) and nn_z > 2.0:
        reasons_scored.append((
            abs(nn_z),
            f"колония пространственно изолирована, но этот признак используется только как дополнительный: nearest_neighbor_distance={_fmt(nn_value)}, медиана={_fmt(nn_median)}, robust_z={_fmt(nn_z)}",
        ))

    if bool(row.get("possible_merged_colony", False)):
        reasons_scored.append((2.0, "форма объекта может соответствовать слипшимся колониям или сомнительной маске"))

    if bool(row.get("rare_morphotype_flag", False)):
        morphotype_id = row.get("morphotype_id", np.nan)
        morphotype_fraction = row.get("morphotype_fraction", np.nan)
        reasons_scored.append((
            1.5,
            f"объект относится к редкому морфотипу, но не обязательно является аномалией: morphotype_id={morphotype_id}, доля={_fmt(morphotype_fraction)}",
        ))

    if bool(row.get("poor_local_background", False)):
        reasons_scored.append((1.0, "локальный фон вокруг колонии ограничен, интерпретация контрастных признаков менее надёжна"))

    if not reasons_scored:
        return [fallback]

    reasons_scored = sorted(reasons_scored, key=lambda x: x[0], reverse=True)
    reasons: list[str] = []
    for _, reason in reasons_scored:
        if reason not in reasons:
            reasons.append(reason)
        if len(reasons) >= max(1, int(top_k)):
            break

    return reasons if reasons else [fallback]

def select_top_anomalies(
    results_df: pd.DataFrame,
    top_k: int | None = None,
    top_percent: float | None = None,
    min_score: float | None = None,
) -> pd.DataFrame:
    """Выбрать только подтверждённые кандидаты, не технические и не слабые top-N."""
    if results_df is None:
        return pd.DataFrame()
    if results_df.empty:
        selected = results_df.copy()
        if "selected_for_further_analysis" not in selected.columns:
            selected["selected_for_further_analysis"] = pd.Series(dtype=bool)
        return selected

    score_col = "final_anomaly_score_raw" if "final_anomaly_score_raw" in results_df.columns else "final_anomaly_score"
    if score_col not in results_df.columns:
        raise ValueError("В results_df отсутствует колонка с итоговым anomaly score")

    df = results_df.copy()
    if "valid_for_anomaly" in df.columns:
        df = df[df["valid_for_anomaly"].fillna(False).astype(bool)].copy()
    if "technical_warning" in df.columns:
        df = df[~df["technical_warning"].fillna(False).astype(bool)].copy()
    if "recommendation_status" in df.columns:
        df = df[df["recommendation_status"].eq("select_candidate")].copy()

    if df.empty:
        selected = results_df.head(0).copy()
        selected["selected_for_further_analysis"] = pd.Series(dtype=bool)
        return selected

    df = df.sort_values(score_col, ascending=False, na_position="last").reset_index(drop=True)
    n_valid = int(results_df["valid_for_anomaly"].fillna(False).astype(bool).sum()) if "valid_for_anomaly" in results_df.columns else len(df)

    if top_k is not None:
        n_select = max(1, int(top_k))
        selected = df.head(n_select).copy()
    elif top_percent is not None:
        p = float(top_percent)
        if p > 1.0:
            p = p / 100.0
        p = min(max(p, 0.0), 1.0)
        n_select = max(1, int(math.ceil(n_valid * p)))
        selected = df.head(n_select).copy()
    elif min_score is not None:
        selected = df[df[score_col] >= float(min_score)].copy()
    else:
        if n_valid < 20:
            n_select = 1 if n_valid < 10 else 2
        elif n_valid <= 100:
            n_select = 3
        elif n_valid <= 300:
            n_select = 5
        else:
            n_select = max(5, int(math.ceil(n_valid * 0.02)))
        selected = df.head(n_select).copy()

    selected["selected_for_further_analysis"] = True
    return selected.reset_index(drop=True)


def collect_review_candidates(results_df: pd.DataFrame) -> pd.DataFrame:
    """Собрать объекты, которые стоит просмотреть, но не выбирать как подтверждённые аномалии."""
    if results_df is None or results_df.empty:
        return pd.DataFrame()
    if "recommendation_status" not in results_df.columns:
        return results_df.head(0).copy()
    review_statuses = {"review_only", "review_segmentation", "rare_morphotype", "weak_outlier"}
    df = results_df[results_df["recommendation_status"].isin(review_statuses)].copy()
    score_col = "final_anomaly_score_raw" if "final_anomaly_score_raw" in df.columns else "final_anomaly_score"
    if score_col in df.columns:
        df = df.sort_values(score_col, ascending=False, na_position="last")
    return df.reset_index(drop=True)

def visualize_anomalies(
    image_rgb: np.ndarray,
    masks,
    results_df: pd.DataFrame,
    selected_ids: list[int] | None = None,
    save_path: str | Path | None = None,
    title: str | None = None,
    show_scores: bool = True,
    show_review: bool = True,
    show_technical_warnings: bool = False,
):
    masks_list = normalize_masks_input(masks)
    if image_rgb is None:
        raise ValueError("image_rgb must not be None")

    selected_info: dict[int, dict[str, Any]] = {}
    technical_ids: set[int] = set()
    review_ids: set[int] = set()

    if results_df is not None and not results_df.empty and "colony_id" in results_df.columns:
        if "technical_warning" in results_df.columns:
            technical_ids = set(
                results_df.loc[
                    results_df["technical_warning"].fillna(False).astype(bool), "colony_id"
                ].dropna().astype(int).tolist()
            )
        if "recommendation_status" in results_df.columns:
            review_ids = set(
                results_df.loc[
                    results_df["recommendation_status"].isin(["review_only", "review_segmentation", "rare_morphotype", "weak_outlier"]),
                    "colony_id",
                ].dropna().astype(int).tolist()
            )
        score_col = "final_anomaly_score_raw" if "final_anomaly_score_raw" in results_df.columns else "final_anomaly_score"
        if selected_ids is None:
            if "selected_for_further_analysis" in results_df.columns:
                selected_df = results_df[results_df["selected_for_further_analysis"].fillna(False).astype(bool)]
            else:
                selected_df = results_df.head(0)
            selected_ids = selected_df["colony_id"].dropna().astype(int).tolist()
        for _, row in results_df.iterrows():
            if pd.notna(row.get("colony_id")):
                cid = int(row["colony_id"])
                selected_info[cid] = {
                    "score": float(row.get(score_col)) if score_col in results_df.columns and pd.notna(row.get(score_col)) else np.nan,
                    "status": str(row.get("recommendation_status", "")),
                }
    elif selected_ids is None:
        selected_ids = []

    selected_set = {int(i) for i in selected_ids}

    smooth_sigma = 1.6
    contour_level = 0.35
    normal_fill_alpha = 0.08
    selected_fill_alpha = 0.06
    review_fill_alpha = 0.05

    def color_for_idx(i: int, n_total: int = 1) -> np.ndarray:
        if n_total < 1:
            n_total = 1
        t = ((i * 37) % n_total) / max(1, n_total - 1)
        hue = (0.02 + 0.96 * t) % 1.0
        sat = 0.85
        val = 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
        return np.array([r, g, b], dtype=float)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image_rgb)
    ax.axis("off")
    if title:
        ax.set_title(title)

    n_total = len(masks_list)
    h, w = image_rgb.shape[:2]

    # Полная заливка инстансов остаётся полупрозрачной: морфология колоний не скрывается.
    fill_rgba = np.zeros((h, w, 4), dtype=float)
    draw_order = [i for i in range(n_total) if (i + 1) not in selected_set]
    draw_order += [i for i in range(n_total) if (i + 1) in selected_set]
    for k in draw_order:
        colony_id = k + 1
        is_technical = colony_id in technical_ids
        is_review = colony_id in review_ids
        is_selected = colony_id in selected_set
        if is_technical and not show_technical_warnings:
            continue
        if is_review and not show_review and not is_selected:
            continue
        mask_bool = masks_list[k].astype(bool)
        if mask_bool.shape != (h, w):
            mask_bool = cv2.resize(mask_bool.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
        if is_selected:
            fill_color = np.array([1.0, 0.95, 0.05])
            fill_alpha = selected_fill_alpha
        elif is_review:
            fill_color = np.array([1.0, 0.45, 0.0])
            fill_alpha = review_fill_alpha
        elif is_technical:
            fill_color = np.array([0.55, 0.55, 0.55])
            fill_alpha = 0.05
        else:
            fill_color = color_for_idx(k, n_total=n_total)
            fill_alpha = normal_fill_alpha
        fill_rgba[mask_bool, :3] = fill_color
        fill_rgba[mask_bool, 3] = fill_alpha
    ax.imshow(fill_rgba)

    for k, mask in enumerate(masks_list):
        colony_id = k + 1
        is_selected = colony_id in selected_set
        is_technical = colony_id in technical_ids
        is_review = colony_id in review_ids
        if is_technical and not show_technical_warnings:
            continue
        if is_review and not show_review and not is_selected:
            continue

        if is_selected:
            color = "yellow"
            linewidth = 3.4
            linestyle = "-"
        elif is_review:
            color = "orange"
            linewidth = 2.2
            linestyle = "--"
        elif is_technical:
            color = "gray"
            linewidth = 1.0
            linestyle = ":"
        else:
            color = color_for_idx(k, n_total=n_total)
            linewidth = 0.7
            linestyle = "-"

        mask_prob = mask.astype(np.float32)
        mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=smooth_sigma, sigmaY=smooth_sigma)
        for contour in find_contours(mask_prob, level=contour_level):
            if is_selected:
                ax.plot(
                    contour[:, 1],
                    contour[:, 0],
                    color="black",
                    linewidth=linewidth + 1.6,
                    antialiased=True,
                    solid_joinstyle="round",
                    solid_capstyle="round",
                )
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
                linestyle=linestyle,
                antialiased=True,
                solid_joinstyle="round",
                solid_capstyle="round",
            )

        if is_selected or (show_review and is_review):
            ys, xs = np.where(mask)
            if len(xs) > 0:
                info = selected_info.get(colony_id, {})
                score = info.get("score", np.nan)
                status = info.get("status", "")
                if not show_scores or not np.isfinite(score):
                    continue
                label_text = f"{score:.2f}"
                ax.text(
                    float(np.mean(xs)),
                    float(np.mean(ys)),
                    label_text,
                    color="black",
                    fontsize=7,
                    ha="center",
                    va="center",
                    bbox=dict(facecolor="yellow" if is_selected else "orange", edgecolor="black", alpha=0.82, pad=1.4),
                )

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")

    return fig

def compute_plate_quality(scores_df: pd.DataFrame, detections_df: pd.DataFrame | None = None) -> pd.DataFrame:
    """Сводная оценка качества чашки/изображения для корректной интерпретации аномалий."""
    if scores_df is None:
        scores_df = pd.DataFrame()
    n_detected = int(len(scores_df))
    n_valid = int(scores_df["valid_for_anomaly"].fillna(False).astype(bool).sum()) if n_detected and "valid_for_anomaly" in scores_df.columns else 0
    n_technical = int(scores_df["technical_warning"].fillna(False).astype(bool).sum()) if n_detected and "technical_warning" in scores_df.columns else 0
    valid_fraction = float(n_valid / n_detected) if n_detected else 0.0
    technical_fraction = float(n_technical / n_detected) if n_detected else 0.0

    if "yolo_conf" in scores_df.columns and n_detected:
        median_yolo_conf = float(pd.to_numeric(scores_df["yolo_conf"], errors="coerce").median())
    elif detections_df is not None and not detections_df.empty and "yolo_conf" in detections_df.columns:
        median_yolo_conf = float(pd.to_numeric(detections_df["yolo_conf"], errors="coerce").median())
    else:
        median_yolo_conf = np.nan

    if "area" in scores_df.columns and n_detected:
        area_values = pd.to_numeric(scores_df["area"], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)
        median_area = float(np.nanmedian(area_values)) if area_values.size else np.nan
        area_cv = float(np.nanstd(area_values) / (np.nanmean(area_values) + 1e-8)) if area_values.size else np.nan
    else:
        median_area = np.nan
        area_cv = np.nan

    if n_detected == 0:
        plate_quality_status = "no_masks"
    elif n_valid < 10:
        plate_quality_status = "low_colony_count"
    elif technical_fraction > 0.40:
        plate_quality_status = "too_many_technical_masks"
    elif np.isfinite(median_yolo_conf) and median_yolo_conf < 0.15:
        plate_quality_status = "low_detection_confidence"
    else:
        plate_quality_status = "ok"

    return pd.DataFrame([{
        "n_detected_colonies": n_detected,
        "n_valid_colonies": n_valid,
        "valid_fraction": valid_fraction,
        "n_technical_warnings": n_technical,
        "technical_fraction": technical_fraction,
        "median_yolo_conf": median_yolo_conf,
        "median_area": median_area,
        "area_cv": area_cv,
        "plate_quality_status": plate_quality_status,
    }])


def build_feature_correlation_report(features_df: pd.DataFrame, threshold: float = 0.90) -> pd.DataFrame:
    """Найти пары сильно коррелирующих признаков для контроля дублирования."""
    if features_df is None or features_df.empty:
        return pd.DataFrame(columns=["feature_1", "feature_2", "correlation"])
    df = features_df.copy()
    if "valid_for_anomaly" in df.columns:
        df = df[df["valid_for_anomaly"].fillna(False).astype(bool)]
    if "technical_warning" in df.columns:
        df = df[~df["technical_warning"].fillna(False).astype(bool)]
    numeric_df = df.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
    if numeric_df.shape[0] < 3 or numeric_df.shape[1] < 2:
        return pd.DataFrame(columns=["feature_1", "feature_2", "correlation"])
    corr = numeric_df.corr(method="spearman", min_periods=3)
    rows = []
    cols = list(corr.columns)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            value = corr.iloc[i, j]
            if pd.notna(value) and abs(float(value)) >= threshold:
                rows.append({"feature_1": cols[i], "feature_2": cols[j], "correlation": float(value)})
    return pd.DataFrame(rows).sort_values("correlation", key=lambda s: s.abs(), ascending=False).reset_index(drop=True) if rows else pd.DataFrame(columns=["feature_1", "feature_2", "correlation"])


def save_feature_space_pca(
    features_df: pd.DataFrame,
    scores_df: pd.DataFrame,
    output_path: str | Path,
) -> None:
    """Сохранить PCA-визуализацию признакового пространства валидных колоний."""
    if features_df is None or features_df.empty or scores_df is None or scores_df.empty:
        return
    df = scores_df.copy()
    if "valid_for_anomaly" in df.columns:
        df = df[df["valid_for_anomaly"].fillna(False).astype(bool)]
    if "technical_warning" in df.columns:
        df = df[~df["technical_warning"].fillna(False).astype(bool)]
    if len(df) < 3:
        return

    core_cols = [
        "log_area", "circularity", "eccentricity", "solidity",
        "intensity_mean", "intensity_iqr", "intensity_entropy",
        "mean_L_lab", "mean_a_lab", "mean_b_lab",
        "glcm_contrast", "lbp_entropy", "local_color_delta_lab",
        "intensity_hist_js_to_plate_median", "feature_knn_distance",
    ]
    cols = [c for c in core_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if len(cols) < 2:
        return
    try:
        X = df[cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
        X = SimpleImputer(strategy="median").fit_transform(X)
        X = RobustScaler().fit_transform(X)
        X2 = PCA(n_components=2, random_state=42).fit_transform(X)

        fig, ax = plt.subplots(figsize=(7, 6))
        status = df["recommendation_status"].astype(str).to_numpy() if "recommendation_status" in df.columns else np.array(["object"] * len(df))
        unique_status = list(dict.fromkeys(status))
        for st in unique_status:
            mask = status == st
            ax.scatter(X2[mask, 0], X2[mask, 1], label=st, alpha=0.75, s=35)

        if "colony_id" in df.columns:
            label_mask = df["recommendation_status"].isin(["select_candidate", "review_segmentation", "review_only"]) if "recommendation_status" in df.columns else pd.Series(False, index=df.index)
            for _, row in df[label_mask].head(20).iterrows():
                pos = df.index.get_loc(row.name)
                ax.text(X2[pos, 0], X2[pos, 1], str(int(row["colony_id"])), fontsize=8)

        ax.set_title("Пространство признаков колоний (PCA)")
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
        ax.legend(loc="best", fontsize=8)
        fig.tight_layout()
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=200)
        plt.close(fig)
    except Exception as exc:
        print(f"Не удалось сохранить PCA-визуализацию: {exc}")


def save_colony_crops(
    image_rgb: np.ndarray,
    masks,
    scores_df: pd.DataFrame,
    output_dir: str | Path,
    statuses: tuple[str, ...] = ("select_candidate", "review_segmentation", "review_only"),
    context_pad: int = 24,
) -> None:
    """Сохранить crop, context и mask для selected/review объектов."""
    if image_rgb is None or scores_df is None or scores_df.empty:
        return
    masks_list = normalize_masks_input(masks)
    if not masks_list:
        return
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    id_to_mask = {i + 1: m for i, m in enumerate(masks_list)}
    df = scores_df[scores_df.get("recommendation_status", "").isin(statuses)].copy() if "recommendation_status" in scores_df.columns else pd.DataFrame()
    if df.empty or "colony_id" not in df.columns:
        return

    h, w = image_rgb.shape[:2]
    for _, row in df.head(30).iterrows():
        try:
            colony_id = int(row["colony_id"])
            mask = id_to_mask.get(colony_id)
            if mask is None or not mask.any():
                continue
            ys, xs = np.where(mask)
            y1 = max(0, int(ys.min()) - context_pad)
            y2 = min(h, int(ys.max()) + context_pad + 1)
            x1 = max(0, int(xs.min()) - context_pad)
            x2 = min(w, int(xs.max()) + context_pad + 1)

            context = image_rgb[y1:y2, x1:x2].copy()
            local_mask = mask[y1:y2, x1:x2]
            crop = image_rgb[ys.min():ys.max()+1, xs.min():xs.max()+1].copy()
            mask_img = (local_mask.astype(np.uint8) * 255)

            prefix = output_dir / f"colony_id_{colony_id:03d}"
            cv2.imwrite(str(prefix) + "_context.png", cv2.cvtColor(context, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(prefix) + "_crop.png", cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(prefix) + "_mask.png", mask_img)
        except Exception:
            continue

def run_single_image_pipeline(
    image_path: str | Path,
    model,
    config: AnomalyDetectionConfig,
    petri_detector_model=None,
) -> dict:
    image_path = Path(image_path)
    image_rgb_original = read_image_rgb(image_path)

    # 1) Предобработка изображения перед сегментацией.
    image_rgb, preprocess_info = prepare_image_for_segmentation(
        image_rgb=image_rgb_original,
        detector_model=petri_detector_model if config.use_petri_detector else None,
        config=config,
    )

    output_dir = Path(config.output_dir) / image_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)

    with open(output_dir / "preprocess_info.json", "w", encoding="utf-8") as f:
        json.dump(preprocess_info, f, ensure_ascii=False, indent=2)

    if config.save_visualizations:
        pre_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
        cv2.imwrite(str(output_dir / "preprocessed_736.png"), pre_bgr)

    # 2) YOLO instance segmentation: сначала получаем маски, только затем считаем признаки.
    masks, detections_df = predict_colony_masks(model, image_rgb, config)

    # 3) Признаки, морфотипы, weighted anomaly score и статус рекомендации.
    features_df = extract_colony_features(
        image_rgb=image_rgb,
        masks=masks,
        detections_df=detections_df,
        plate_mask=None,
        config=config,
    )
    scores_df = compute_anomaly_scores(features_df, config=config)

    if not scores_df.empty:
        scores_df["explanations"] = scores_df.apply(
            lambda row: explain_anomaly(row, scores_df, top_k=5), axis=1
        )
        scores_df["explanations_text"] = scores_df["explanations"].apply(lambda x: "; ".join(x))
    else:
        scores_df = scores_df.copy()
        scores_df["explanations"] = pd.Series(dtype=object)
        scores_df["explanations_text"] = pd.Series(dtype=object)

    # 4) Оценка качества чашки. Если качество низкое, автоматический select_candidate
    # переводится в review_only, чтобы не выдавать слабое статистическое решение как подтверждённый отбор.
    plate_quality_df = compute_plate_quality(scores_df, detections_df)
    plate_quality_status = (
        str(plate_quality_df.loc[0, "plate_quality_status"])
        if not plate_quality_df.empty and "plate_quality_status" in plate_quality_df.columns
        else "ok"
    )
    if plate_quality_status != "ok" and not scores_df.empty and "recommendation_status" in scores_df.columns:
        bad_auto = scores_df["recommendation_status"].eq("select_candidate")
        scores_df.loc[bad_auto, "recommendation_status"] = "review_only"
        scores_df.loc[bad_auto, "selected_for_further_analysis"] = False
        if "explanations_text" in scores_df.columns:
            scores_df.loc[bad_auto, "explanations_text"] = (
                scores_df.loc[bad_auto, "explanations_text"].astype(str)
                + f"; качество чашки: {plate_quality_status}, автоматический отбор заменён на экспертный просмотр"
            )

    selected_df = select_top_anomalies(
        scores_df,
        top_k=config.top_k,
        top_percent=config.top_percent,
        min_score=config.min_score,
    )

    if "selected_for_further_analysis" not in scores_df.columns:
        scores_df["selected_for_further_analysis"] = False
    scores_df["selected_for_further_analysis"] = False

    selected_ids = (
        selected_df["colony_id"].dropna().astype(int).tolist()
        if "colony_id" in selected_df.columns and not selected_df.empty
        else []
    )
    if selected_ids and "colony_id" in scores_df.columns:
        scores_df.loc[
            scores_df["colony_id"].astype(int).isin(selected_ids),
            "selected_for_further_analysis",
        ] = True
        selected_df = scores_df[scores_df["selected_for_further_analysis"].fillna(False).astype(bool)].copy()

    review_candidates_df = collect_review_candidates(scores_df)
    technical_warnings_df = (
        scores_df[scores_df["technical_warning"].fillna(False).astype(bool)].copy()
        if not scores_df.empty and "technical_warning" in scores_df.columns
        else pd.DataFrame()
    )
    valid_features_df = (
        features_df[features_df["valid_for_anomaly"].fillna(False).astype(bool)].copy()
        if not features_df.empty and "valid_for_anomaly" in features_df.columns
        else pd.DataFrame()
    )

    morphotype_cols = [
        "colony_id",
        "morphotype_id",
        "morphotype_size",
        "morphotype_fraction",
        "rare_morphotype_flag",
        "within_morphotype_anomaly_score",
        "size_within_morphotype_z",
        "texture_within_morphotype_z",
        "color_within_morphotype_z",
        "recommendation_status",
    ]
    morphotypes_df = scores_df[[c for c in morphotype_cols if c in scores_df.columns]].copy() if not scores_df.empty else pd.DataFrame()

    feature_correlation_df = (
        build_feature_correlation_report(scores_df)
        if bool(getattr(config, "save_feature_correlation_report", True))
        else pd.DataFrame()
    )

    fig_save_path = output_dir / "anomalies_visualization.png" if config.save_visualizations else None
    fig = visualize_anomalies(
        image_rgb=image_rgb,
        masks=masks,
        results_df=scores_df,
        selected_ids=selected_ids,
        save_path=fig_save_path,
        title=f"Anomaly candidates: {image_path.name}",
        show_scores=True,
        show_review=True,
        show_technical_warnings=False,
    )

    if bool(getattr(config, "save_feature_space_plot", True)):
        save_feature_space_pca(scores_df, scores_df, output_dir / "feature_space_pca.png")

    if bool(getattr(config, "save_colony_crops", True)):
        save_colony_crops(
            image_rgb=image_rgb,
            masks=masks,
            scores_df=scores_df,
            output_dir=output_dir / "colony_crops",
        )

    masks_labeled = np.zeros(image_rgb.shape[:2], dtype=np.int32)
    for colony_id, mask in enumerate(normalize_masks_input(masks), start=1):
        if mask.shape != masks_labeled.shape:
            mask = cv2.resize(
                mask.astype(np.uint8),
                (masks_labeled.shape[1], masks_labeled.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            ).astype(bool)
        masks_labeled[mask] = int(colony_id)
    np.save(output_dir / "masks_labeled.npy", masks_labeled)

    if config.save_csv:
        detections_df.to_csv(output_dir / "detections_yolo.csv", index=False, encoding="utf-8-sig")
        features_df.to_csv(output_dir / "colony_features.csv", index=False, encoding="utf-8-sig")
        valid_features_df.to_csv(output_dir / "colony_features_valid.csv", index=False, encoding="utf-8-sig")
        scores_df.to_csv(output_dir / "colony_anomaly_scores.csv", index=False, encoding="utf-8-sig")
        selected_df.to_csv(output_dir / "selected_anomalies.csv", index=False, encoding="utf-8-sig")
        review_candidates_df.to_csv(output_dir / "review_candidates.csv", index=False, encoding="utf-8-sig")
        technical_warnings_df.to_csv(output_dir / "technical_warnings.csv", index=False, encoding="utf-8-sig")
        morphotypes_df.to_csv(output_dir / "morphotypes.csv", index=False, encoding="utf-8-sig")

        plate_quality_df.to_csv(output_dir / "plate_quality.csv", index=False, encoding="utf-8-sig")
        feature_correlation_df.to_csv(output_dir / "feature_correlation_report.csv", index=False, encoding="utf-8-sig")

        if config.save_xlsx:
            try:
                with pd.ExcelWriter(output_dir / "colony_analysis_report.xlsx") as writer:
                    detections_df.to_excel(writer, sheet_name="detections_yolo", index=False)
                    features_df.to_excel(writer, sheet_name="colony_features_all", index=False)
                    valid_features_df.to_excel(writer, sheet_name="colony_features_valid", index=False)
                    scores_df.to_excel(writer, sheet_name="anomaly_scores", index=False)
                    selected_df.to_excel(writer, sheet_name="selected_anomalies", index=False)
                    review_candidates_df.to_excel(writer, sheet_name="review_candidates", index=False)
                    technical_warnings_df.to_excel(writer, sheet_name="technical_warnings", index=False)
                    morphotypes_df.to_excel(writer, sheet_name="morphotypes", index=False)
                    plate_quality_df.to_excel(writer, sheet_name="plate_quality", index=False)
                    feature_correlation_df.to_excel(writer, sheet_name="feature_correlation", index=False)
            except Exception as exc:
                print(f"Не удалось сохранить Excel-отчёт: {exc}")

    return {
        "image_path": image_path,
        "image_original": image_rgb_original,
        "image": image_rgb,
        "preprocess": preprocess_info,
        "masks": masks,
        "detections": detections_df,
        "features": features_df,
        "scores": scores_df,
        "selected": selected_df,
        "review_candidates": review_candidates_df,
        "technical_warnings": technical_warnings_df,
        "morphotypes": morphotypes_df,
        "plate_quality": plate_quality_df,
        "feature_correlation": feature_correlation_df,
        "figure": fig,
    }

def run_batch_pipeline(config: AnomalyDetectionConfig) -> pd.DataFrame:
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_paths = collect_image_paths(config.input_path)

    if config.model_weights_path:
        seg_weights_path = Path(config.model_weights_path)
        if not seg_weights_path.exists():
            raise FileNotFoundError(f"Segmentation weights file not found: {seg_weights_path}")
    else:
        seg_weights_path = find_best_yolo_weights(config.model_dir)

    print("Используем веса модели сегментации для batch:", seg_weights_path)
    model = YOLO(str(seg_weights_path))

    petri_detector_model = None
    if config.use_petri_detector or str(config.preprocess_mode).lower() == "detect_petri":
        petri_weights_path = Path(config.petri_detector_weights_path)
        if not petri_weights_path.exists():
            raise FileNotFoundError(f"Petri detector weights file not found: {petri_weights_path}")
        print("Используем веса детектора чашки Петри для batch:", petri_weights_path)
        petri_detector_model = YOLO(str(petri_weights_path))

    selected_rows: list[pd.DataFrame] = []
    review_rows: list[pd.DataFrame] = []
    summary_rows: list[dict[str, Any]] = []

    selected_required_cols = [
        "image_name",
        "colony_id",
        "final_anomaly_score_raw",
        "final_anomaly_score_percentile",
        "final_anomaly_score",
        "anomaly_rank",
        "recommendation_status",
        "anomaly_type",
        "independent_evidence_count",
        "consensus_score",
        "overall_reliability_score",
        "explanations_text",
        "yolo_conf",
        "area",
        "log_area",
        "area_to_median_ratio",
        "circularity",
        "solidity",
        "intensity_mean",
        "intensity_iqr",
        "glcm_contrast",
        "lbp_entropy",
        "histogram_score",
        "local_color_delta_lab",
        "nearest_neighbor_distance",
        "feature_knn_distance",
    ]

    for image_path in tqdm(image_paths, desc="Batch processing"):
        try:
            result = run_single_image_pipeline(
                image_path=image_path,
                model=model,
                config=config,
                petri_detector_model=petri_detector_model,
            )
            scores_df = result["scores"]
            selected_df = result["selected"]
            review_df = result.get("review_candidates", pd.DataFrame())
            plate_quality_df = result.get("plate_quality", pd.DataFrame())

            n_detected = int(len(scores_df))
            n_valid = int(scores_df["valid_for_anomaly"].fillna(False).astype(bool).sum()) if n_detected and "valid_for_anomaly" in scores_df.columns else 0
            n_technical = int(scores_df["technical_warning"].fillna(False).astype(bool).sum()) if n_detected and "technical_warning" in scores_df.columns else 0
            valid_fraction = float(n_valid / n_detected) if n_detected else 0.0
            technical_fraction = float(n_technical / n_detected) if n_detected else 0.0
            plate_quality_status = (
                str(plate_quality_df.loc[0, "plate_quality_status"])
                if plate_quality_df is not None and not plate_quality_df.empty and "plate_quality_status" in plate_quality_df.columns
                else ("no_masks" if n_detected == 0 else ("no_valid_colonies" if n_valid == 0 else "ok"))
            )
            n_selected = int(len(selected_df))
            n_review = int(len(review_df))
            n_rare = int(scores_df["rare_morphotype_flag"].fillna(False).astype(bool).sum()) if n_detected and "rare_morphotype_flag" in scores_df.columns else 0

            valid_scores = (
                scores_df[scores_df["valid_for_anomaly"].fillna(False).astype(bool)].copy()
                if n_detected and "valid_for_anomaly" in scores_df.columns
                else pd.DataFrame()
            )
            max_raw = float(valid_scores["final_anomaly_score_raw"].max()) if not valid_scores.empty and "final_anomaly_score_raw" in valid_scores.columns else np.nan
            mean_raw = float(valid_scores["final_anomaly_score_raw"].mean()) if not valid_scores.empty and "final_anomaly_score_raw" in valid_scores.columns else np.nan
            max_percentile = float(valid_scores["final_anomaly_score_percentile"].max()) if not valid_scores.empty and "final_anomaly_score_percentile" in valid_scores.columns else np.nan

            if n_detected == 0:
                status = "no_masks"
            elif n_valid == 0:
                status = "no_valid_colonies"
            elif plate_quality_status != "ok":
                status = "low_quality"
            else:
                status = "ok"

            type_source = selected_df if not selected_df.empty else review_df
            if not type_source.empty and "anomaly_type" in type_source.columns:
                main_types = ", ".join(type_source["anomaly_type"].dropna().astype(str).value_counts().head(3).index.tolist())
            else:
                main_types = ""

            summary_rows.append(
                {
                    "image_name": Path(image_path).name,
                    "n_detected_colonies": n_detected,
                    "n_valid_colonies": n_valid,
                    "valid_fraction": valid_fraction,
                    "n_technical_warnings": n_technical,
                    "technical_fraction": technical_fraction,
                    "n_selected_anomalies": n_selected,
                    "n_review_candidates": n_review,
                    "n_rare_morphotypes": n_rare,
                    "max_anomaly_score_raw": max_raw,
                    "mean_anomaly_score_raw": mean_raw,
                    "max_anomaly_score_percentile": max_percentile,
                    "main_anomaly_types": main_types,
                    "plate_quality_status": plate_quality_status,
                    "status": status,
                }
            )

            if not selected_df.empty:
                temp = selected_df.copy()
                temp.insert(0, "image_name", Path(image_path).name)
                for col in selected_required_cols:
                    if col not in temp.columns:
                        temp[col] = np.nan
                selected_rows.append(temp[selected_required_cols])

            if not review_df.empty:
                temp = review_df.copy()
                temp.insert(0, "image_name", Path(image_path).name)
                for col in selected_required_cols:
                    if col not in temp.columns:
                        temp[col] = np.nan
                review_rows.append(temp[selected_required_cols])

        except Exception as exc:
            print(f"Ошибка при обработке {image_path}: {exc}")
            summary_rows.append(
                {
                    "image_name": Path(image_path).name,
                    "n_detected_colonies": 0,
                    "n_valid_colonies": 0,
                    "valid_fraction": 0.0,
                    "n_technical_warnings": 0,
                    "technical_fraction": 0.0,
                    "n_selected_anomalies": 0,
                    "n_review_candidates": 0,
                    "n_rare_morphotypes": 0,
                    "max_anomaly_score_raw": np.nan,
                    "mean_anomaly_score_raw": np.nan,
                    "max_anomaly_score_percentile": np.nan,
                    "main_anomaly_types": "",
                    "plate_quality_status": "error",
                    "status": "error",
                }
            )

    all_selected = (
        pd.concat(selected_rows, ignore_index=True)
        if selected_rows
        else pd.DataFrame(columns=selected_required_cols)
    )
    all_review = (
        pd.concat(review_rows, ignore_index=True)
        if review_rows
        else pd.DataFrame(columns=selected_required_cols)
    )
    if not all_selected.empty:
        all_selected = all_selected.sort_values("final_anomaly_score_raw", ascending=False, na_position="last").reset_index(drop=True)
    if not all_review.empty:
        all_review = all_review.sort_values("final_anomaly_score_raw", ascending=False, na_position="last").reset_index(drop=True)

    batch_summary = pd.DataFrame(
        summary_rows,
        columns=[
            "image_name",
            "n_detected_colonies",
            "n_valid_colonies",
            "valid_fraction",
            "n_technical_warnings",
            "technical_fraction",
            "n_selected_anomalies",
            "n_review_candidates",
            "n_rare_morphotypes",
            "max_anomaly_score_raw",
            "mean_anomaly_score_raw",
            "max_anomaly_score_percentile",
            "main_anomaly_types",
            "plate_quality_status",
            "status",
        ],
    )

    all_selected.to_csv(output_dir / "all_selected_anomalies.csv", index=False, encoding="utf-8-sig")
    all_review.to_csv(output_dir / "all_review_candidates.csv", index=False, encoding="utf-8-sig")
    batch_summary.to_csv(output_dir / "batch_summary.csv", index=False, encoding="utf-8-sig")

    if config.save_xlsx:
        try:
            with pd.ExcelWriter(output_dir / "batch_colony_analysis_report.xlsx") as writer:
                all_selected.to_excel(writer, sheet_name="selected_anomalies", index=False)
                all_review.to_excel(writer, sheet_name="review_candidates", index=False)
                batch_summary.to_excel(writer, sheet_name="batch_summary", index=False)
        except Exception as exc:
            print(f"Не удалось сохранить batch Excel-отчёт: {exc}")

    return all_selected

def _apply_stability_perturbation(image_rgb: np.ndarray, variant: str, rng: np.random.Generator) -> np.ndarray:
    img = image_rgb.astype(np.float32)
    if variant == "brightness_plus_5pct":
        img = img * 1.05
    elif variant == "brightness_minus_5pct":
        img = img * 0.95
    elif variant == "contrast_plus_5pct":
        img = (img - 127.5) * 1.05 + 127.5
    elif variant == "contrast_minus_5pct":
        img = (img - 127.5) * 0.95 + 127.5
    elif variant == "slight_blur":
        img = cv2.GaussianBlur(np.clip(img, 0, 255).astype(np.uint8), (3, 3), sigmaX=0.6).astype(np.float32)
    elif variant == "slight_gaussian_noise":
        img = img + rng.normal(0.0, 3.0, size=img.shape)
    else:
        raise ValueError(f"неизвестный вариант perturbation: {variant}")
    return np.clip(img, 0, 255).astype(np.uint8)


def _selected_centroid_table(scores_df: pd.DataFrame, top_k: int) -> pd.DataFrame:
    if scores_df is None or scores_df.empty:
        return pd.DataFrame()
    df = scores_df.copy()
    if "valid_for_anomaly" in df.columns:
        df = df[df["valid_for_anomaly"].fillna(False).astype(bool)]
    if "technical_warning" in df.columns:
        df = df[~df["technical_warning"].fillna(False).astype(bool)]
    required = {"colony_id", "centroid_x", "centroid_y", "final_anomaly_score"}
    if not required.issubset(df.columns):
        return pd.DataFrame()
    return df.sort_values("final_anomaly_score", ascending=False, na_position="last").head(top_k).copy()


def _match_selected_to_baseline(
    baseline_scores: pd.DataFrame,
    perturbed_scores: pd.DataFrame,
    top_k: int,
    max_match_distance: float = 20.0,
) -> tuple[set[int], int, float]:
    base_top = _selected_centroid_table(baseline_scores, top_k=top_k)
    pert_top = _selected_centroid_table(perturbed_scores, top_k=top_k)
    if base_top.empty or pert_top.empty:
        return set(), 0, np.nan

    base_coords = base_top[["centroid_x", "centroid_y"]].to_numpy(dtype=float)
    base_ids = base_top["colony_id"].astype(int).to_numpy()
    tree = cKDTree(base_coords)

    mapped_ids: set[int] = set()
    for _, row in pert_top.iterrows():
        point = np.array([float(row["centroid_x"]), float(row["centroid_y"])])
        if not np.all(np.isfinite(point)):
            continue
        dist, idx = tree.query(point, k=1)
        if np.isfinite(dist) and dist <= max_match_distance:
            mapped_ids.add(int(base_ids[int(idx)]))

    base_set = set(base_ids.tolist())
    n_common = len(base_set & mapped_ids)
    union = base_set | mapped_ids
    jaccard = float(n_common / len(union)) if union else np.nan
    return mapped_ids, n_common, jaccard


def _spearman_by_centroid_matching(
    baseline_scores: pd.DataFrame,
    perturbed_scores: pd.DataFrame,
    max_match_distance: float = 20.0,
) -> tuple[float, int]:
    if baseline_scores is None or baseline_scores.empty or perturbed_scores is None or perturbed_scores.empty:
        return np.nan, 0
    needed = {"centroid_x", "centroid_y", "final_anomaly_score"}
    if not needed.issubset(baseline_scores.columns) or not needed.issubset(perturbed_scores.columns):
        return np.nan, 0

    base_df = baseline_scores.copy()
    pert_df = perturbed_scores.copy()
    if "valid_for_anomaly" in base_df.columns:
        base_df = base_df[base_df["valid_for_anomaly"].fillna(False).astype(bool)]
    if "valid_for_anomaly" in pert_df.columns:
        pert_df = pert_df[pert_df["valid_for_anomaly"].fillna(False).astype(bool)]
    if base_df.empty or pert_df.empty:
        return np.nan, 0

    base_coords = base_df[["centroid_x", "centroid_y"]].to_numpy(dtype=float)
    tree = cKDTree(base_coords)
    base_scores = base_df["final_anomaly_score"].to_numpy(dtype=float)

    matched_base_scores: list[float] = []
    matched_pert_scores: list[float] = []
    for _, row in pert_df.iterrows():
        point = np.array([float(row["centroid_x"]), float(row["centroid_y"])])
        if not np.all(np.isfinite(point)):
            continue
        dist, idx = tree.query(point, k=1)
        if np.isfinite(dist) and dist <= max_match_distance:
            base_score = float(base_scores[int(idx)])
            pert_score = float(row["final_anomaly_score"])
            if np.isfinite(base_score) and np.isfinite(pert_score):
                matched_base_scores.append(base_score)
                matched_pert_scores.append(pert_score)

    if len(matched_base_scores) < 3:
        return np.nan, len(matched_base_scores)
    try:
        corr = stats.spearmanr(matched_base_scores, matched_pert_scores, nan_policy="omit").correlation
        return float(corr), len(matched_base_scores)
    except Exception:
        return np.nan, len(matched_base_scores)


def run_stability_test(
    image_path: str | Path,
    model,
    config: AnomalyDetectionConfig,
    top_k: int = 5,
    petri_detector_model=None,
) -> pd.DataFrame:
    image_path = Path(image_path)
    rng = np.random.default_rng(config.random_state)

    stability_config = AnomalyDetectionConfig(**config.__dict__)
    stability_config.top_k = int(top_k)
    stability_config.output_dir = str(Path(config.output_dir) / "stability_test" / image_path.stem)
    stability_config.save_visualizations = False
    stability_config.save_csv = False
    stability_config.save_xlsx = False

    baseline_result = run_single_image_pipeline(
        image_path=image_path,
        model=model,
        config=stability_config,
        petri_detector_model=petri_detector_model,
    )
    baseline_scores = baseline_result["scores"]
    baseline_top = _selected_centroid_table(baseline_scores, top_k=top_k)
    baseline_top_ids = set(baseline_top["colony_id"].astype(int).tolist()) if not baseline_top.empty else set()

    original_rgb = read_image_rgb(image_path)
    temp_dir = Path(config.output_dir) / "_stability_inputs" / image_path.stem
    temp_dir.mkdir(parents=True, exist_ok=True)

    variants = [
        "brightness_plus_5pct",
        "brightness_minus_5pct",
        "contrast_plus_5pct",
        "contrast_minus_5pct",
        "slight_blur",
        "slight_gaussian_noise",
    ]

    rows: list[dict[str, Any]] = []
    for variant in variants:
        perturbed_rgb = _apply_stability_perturbation(original_rgb, variant, rng)
        perturbed_path = temp_dir / f"{image_path.stem}_{variant}.png"
        cv2.imwrite(str(perturbed_path), cv2.cvtColor(perturbed_rgb, cv2.COLOR_RGB2BGR))

        try:
            result = run_single_image_pipeline(
                image_path=perturbed_path,
                model=model,
                config=stability_config,
                petri_detector_model=petri_detector_model,
            )
            perturbed_scores = result["scores"]
            mapped_ids, n_common, jaccard = _match_selected_to_baseline(
                baseline_scores=baseline_scores,
                perturbed_scores=perturbed_scores,
                top_k=top_k,
            )
            spearman_corr, n_matched = _spearman_by_centroid_matching(
                baseline_scores=baseline_scores,
                perturbed_scores=perturbed_scores,
            )
            rows.append(
                {
                    "variant": variant,
                    "baseline_top_k_ids": ";".join(map(str, sorted(baseline_top_ids))),
                    "matched_perturbed_top_k_as_baseline_ids": ";".join(map(str, sorted(mapped_ids))),
                    "top_k_jaccard": jaccard,
                    "n_common_top_k": n_common,
                    "spearman_corr_scores": spearman_corr,
                    "n_matched_for_spearman": n_matched,
                    "status": "ok",
                }
            )
        except Exception as exc:
            rows.append(
                {
                    "variant": variant,
                    "baseline_top_k_ids": ";".join(map(str, sorted(baseline_top_ids))),
                    "matched_perturbed_top_k_as_baseline_ids": "",
                    "top_k_jaccard": np.nan,
                    "n_common_top_k": 0,
                    "spearman_corr_scores": np.nan,
                    "n_matched_for_spearman": 0,
                    "status": f"error: {exc}",
                }
            )

    stability_df = pd.DataFrame(rows)
    if not stability_df.empty:
        spearman_component = (stability_df["spearman_corr_scores"].clip(-1, 1) + 1.0) / 2.0
        stability_df["stability_score"] = np.nanmean(
            np.vstack([
                stability_df["top_k_jaccard"].to_numpy(dtype=float),
                spearman_component.to_numpy(dtype=float),
            ]),
            axis=0,
        )
    output_dir = Path(config.output_dir) / image_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)
    stability_df.to_csv(output_dir / "stability_test.csv", index=False, encoding="utf-8-sig")
    return stability_df

DEFAULT_PETRI_IMAGES = [
    Path("C:/ColonyNet/Петри/IMG_4615.jpg"),
    Path("C:/ColonyNet/Петри/IMG_4655.jpg"),
    Path("C:/ColonyNet/Петри/IMG_4377.jpg"),
    Path("C:/ColonyNet/Петри/IMG_7438.jpg"),
    Path("C:/ColonyNet/Петри/IMG_6137.jpg"),
]


def make_full_pipeline_config(
    output_dir: str | Path = "outputs/colony_anomaly_detection/petri_full_pipeline",
) -> AnomalyDetectionConfig:
    """Create config for raw Petri photos: detector first, then 736x736 segmentation."""
    return replace(
        AnomalyDetectionConfig(),
        preprocess_mode="detect_petri",
        use_petri_detector=True,
        output_dir=str(output_dir),
    )


def load_pipeline_models(config: AnomalyDetectionConfig):
    """Load segmentation model X and Petri dish detector."""
    if config.model_weights_path:
        seg_weights_path = Path(config.model_weights_path)
        if not seg_weights_path.exists():
            raise FileNotFoundError(f"Segmentation weights file not found: {seg_weights_path}")
    else:
        seg_weights_path = find_best_yolo_weights(config.model_dir)

    petri_weights_path = Path(config.petri_detector_weights_path)
    if not petri_weights_path.exists():
        raise FileNotFoundError(f"Petri detector weights file not found: {petri_weights_path}")

    print("Segmentation weights:", seg_weights_path)
    print("Petri detector weights:", petri_weights_path)
    segmentation_model = YOLO(str(seg_weights_path))
    petri_detector_model = YOLO(str(petri_weights_path))
    return segmentation_model, petri_detector_model


def run_full_pipeline_for_images(
    image_paths: list[str | Path] | None = None,
    config: AnomalyDetectionConfig | None = None,
) -> dict[str, Any]:
    """Run full two-model pipeline for a fixed list of Petri images."""
    if config is None:
        config = make_full_pipeline_config()
    else:
        config = replace(config, preprocess_mode="detect_petri", use_petri_detector=True)

    paths = [Path(p) for p in (image_paths or DEFAULT_PETRI_IMAGES)]
    missing = [p for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing input images: " + ", ".join(str(p) for p in missing))

    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    segmentation_model, petri_detector_model = load_pipeline_models(config)

    results: dict[str, dict[str, Any]] = {}
    summary_rows: list[dict[str, Any]] = []
    selected_tables: list[pd.DataFrame] = []
    review_tables: list[pd.DataFrame] = []
    technical_tables: list[pd.DataFrame] = []

    for image_path in paths:
        print(f"\n=== Full pipeline: {image_path.name} ===")
        result = run_single_image_pipeline(
            image_path=image_path,
            model=segmentation_model,
            config=config,
            petri_detector_model=petri_detector_model,
        )
        results[image_path.name] = result

        scores_df = result.get("scores", pd.DataFrame())
        selected_df = result.get("selected", pd.DataFrame())
        review_df = result.get("review_candidates", pd.DataFrame())
        technical_df = result.get("technical_warnings", pd.DataFrame())
        plate_quality_df = result.get("plate_quality", pd.DataFrame())

        n_detected = int(len(scores_df))
        n_valid = (
            int(scores_df["valid_for_anomaly"].fillna(False).astype(bool).sum())
            if n_detected and "valid_for_anomaly" in scores_df.columns
            else 0
        )
        n_selected = int(len(selected_df))
        n_review = int(len(review_df))
        n_technical = int(len(technical_df))
        max_score = (
            float(scores_df["final_anomaly_score_raw"].max())
            if n_detected and "final_anomaly_score_raw" in scores_df.columns
            else np.nan
        )
        plate_quality_status = (
            str(plate_quality_df.loc[0, "plate_quality_status"])
            if plate_quality_df is not None and not plate_quality_df.empty and "plate_quality_status" in plate_quality_df.columns
            else ("no_masks" if n_detected == 0 else ("no_valid_colonies" if n_valid == 0 else "ok"))
        )

        image_output_dir = output_dir / image_path.stem
        summary_rows.append(
            {
                "image_name": image_path.name,
                "n_detected_colonies": n_detected,
                "n_valid_colonies": n_valid,
                "n_selected_anomalies": n_selected,
                "n_review_candidates": n_review,
                "n_technical_warnings": n_technical,
                "max_anomaly_score_raw": max_score,
                "plate_quality_status": plate_quality_status,
                "output_dir": str(image_output_dir),
            }
        )

        if not selected_df.empty:
            tmp = selected_df.copy()
            tmp.insert(0, "image_name", image_path.name)
            selected_tables.append(tmp)
        if not review_df.empty:
            tmp = review_df.copy()
            tmp.insert(0, "image_name", image_path.name)
            review_tables.append(tmp)
        if not technical_df.empty:
            tmp = technical_df.copy()
            tmp.insert(0, "image_name", image_path.name)
            technical_tables.append(tmp)

        print(
            f"Detected: {n_detected}, valid: {n_valid}, selected: {n_selected}, "
            f"review: {n_review}, technical: {n_technical}, quality: {plate_quality_status}"
        )
        print("Output folder:", image_output_dir)
        fig = result.get("figure")
        if fig is not None:
            plt.close(fig)

    summary_df = pd.DataFrame(summary_rows)
    all_selected = pd.concat(selected_tables, ignore_index=True) if selected_tables else pd.DataFrame()
    all_review = pd.concat(review_tables, ignore_index=True) if review_tables else pd.DataFrame()
    all_technical = pd.concat(technical_tables, ignore_index=True) if technical_tables else pd.DataFrame()

    summary_df.to_csv(output_dir / "petri_5_images_summary.csv", index=False, encoding="utf-8-sig")
    all_selected.to_csv(output_dir / "petri_5_images_selected_anomalies.csv", index=False, encoding="utf-8-sig")
    all_review.to_csv(output_dir / "petri_5_images_review_candidates.csv", index=False, encoding="utf-8-sig")
    all_technical.to_csv(output_dir / "petri_5_images_technical_warnings.csv", index=False, encoding="utf-8-sig")

    if config.save_xlsx:
        try:
            with pd.ExcelWriter(output_dir / "petri_5_images_report.xlsx") as writer:
                summary_df.to_excel(writer, sheet_name="summary", index=False)
                all_selected.to_excel(writer, sheet_name="selected_anomalies", index=False)
                all_review.to_excel(writer, sheet_name="review_candidates", index=False)
                all_technical.to_excel(writer, sheet_name="technical_warnings", index=False)
        except Exception as exc:
            print(f"Could not save Excel report for 5 images: {exc}")

    return {
        "config": config,
        "results": results,
        "summary": summary_df,
        "selected": all_selected,
        "review_candidates": all_review,
        "technical_warnings": all_technical,
        "output_dir": output_dir,
    }


def main() -> None:
    result = run_full_pipeline_for_images()
    print("\n=== Summary ===")
    print(result["summary"].to_string(index=False))
    print("\nSaved to:", result["output_dir"])


# In the notebook main() is not called automatically.


## 7. Run Full Pipeline on Petri Images

Set `RUN_FINAL_FULL_PIPELINE = True` in the first cell, then run this cell. It processes the configured Petri images sequentially with both models and saves anomaly reports.


In [ ]:
config = make_full_pipeline_config()
print("Petri detector:", config.petri_detector_weights_path)
print("Colony segmenter:", config.model_weights_path)
print("Output dir:", config.output_dir)
print("Input images:")
for image_path in DEFAULT_PETRI_IMAGES:
    print(" -", image_path, "exists=", Path(image_path).exists())

if RUN_FINAL_FULL_PIPELINE:
    full_result = run_full_pipeline_for_images(config=config)
    display(full_result["summary"])
    display(full_result["selected"].head())
    display(full_result["review_candidates"].head())
    display(full_result["technical_warnings"].head())
else:
    print("Set RUN_FINAL_FULL_PIPELINE = True to run final predictions and anomaly detection.")


## 8. Output Files

The final pipeline saves per-image folders under `outputs/colony_anomaly_detection/petri_full_pipeline` and aggregate files: summary CSV, selected anomalies CSV, review candidates CSV, technical warnings CSV, Excel report if available, visualizations, masks, and cropped 736x736 images.
